In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
import csv
import time
import math
import itertools
import statistics
import random
import pickle
import shutil
import numpy as np
import pandas as pd
# import modin.pandas as pd
import seaborn as sns
import joblib
import gc
import sys

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.colors import LinearSegmentedColormap
%matplotlib inline
plt.ioff()

import scipy
from scipy import signal, stats
from scipy.signal import filtfilt, butter, lfilter, sosfilt, coherence
from scipy.stats import entropy, skew, kurtosis, zscore, ttest_ind, f_oneway
from scipy.integrate import simpson
from scipy.fft import fft

# !pip install mne
import mne
from mne.preprocessing import ICA
from mne.time_frequency import psd_array_welch
mne.set_log_level('WARNING')
# !pip install mne_connectivity
# !pip install pyvistaqt
# !pip install ipywidgets
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_sensors_connectivity, plot_connectivity_circle

# !pip install asrpy
from asrpy import ASR

# !pip install antropy
from antropy import hjorth_params, petrosian_fd, higuchi_fd, sample_entropy, spectral_entropy

# !pip install PyWavelets
import pywt

from sklearn import svm
from sklearn.decomposition import FastICA, PCA
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, power_transform
from sklearn.cluster import KMeans
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder

# !pip install xgboost
# !pip install lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# !pip install pyeeg
# import pyeeg
# from pyegg import hjorth

import decimal
from decimal import *
getcontext().prec = 2

# !pip install brainflow
# import brainflow
# from brainflow.board_shim import BoardShim, BrainFlowInputParams, LogLevels, BoardIds
# from brainflow.data_filter import DataFilter, FilterTypes, AggOperations

import warnings
warnings.filterwarnings("ignore")

In [2]:
debug = False
### Start Form Input ###
directory_raw = "imotion_raw" #folder lokasi data raw, semua data raw kumpulkan di 1 folder ini tanpa di dalem sub folder lain
directory_raw_plot = directory_raw + "/plot"
directory_ica = "imotion_ica"
directory_ica_plot = directory_ica + "/plot"
directory_ica_fif = directory_ica + "/fif"
directory_psd = "imotion_psd"
directory_graph = "imotion_graph"
directory_encoded = "imotion_encoded"
directory_mav = "imotion_mav"
directory_mav_plot = directory_mav + "/plot"
directory_mav_plot_time_frequency = directory_mav_plot + "/time_frequency"
directory_compare = "imotion_compare"
directory_analysis = "imotion_analysis"
directory_analysis_confusion_matrix = directory_analysis + "/confusion_matrix"
directory_unique = "imotion_unique"
directory_empty = "imotion_raw_empty"
directory_corrupt = "imotion_raw_corrupt"
directory_incomplete = "imotion_raw_incomplete"

channels = ["F8","F7","T8","T7","P8","P7","O2","O1"] #channel yang digunakan untuk rekaman, isi harus urut dan gunakan 'NaN' jika channel tidak digunakan
channeling = ["T8","T7","P8","P7","O2","O1"] #channel yang dipilih untuk analisa encode
channeling_colors = {
    "F8": "#1f77b4",  # Blue
    "F7": "#ff7f0e",  # Orange
    "T8": "#2ca02c",  # Green
    "T7": "#d62728",  # Red
    "P8": "#9467bd",  # Purple
    "P7": "#8c564b",  # Brown
    "O2": "#17becf",  # Cyan
    "O1": "#e377c2"   # Pink
}
eog_channels = ['F7', 'F8']
brainwaves = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
subbanding = ["Alpha", "Beta", "Gamma"] #subband yang dipilih untuk analisa encode
label_prefix = '_' #tanda baca sebelum pembeda label misal "arbi_senang", "arbi_sedih" atau "arbi-senang" berarti '-'
label_prefix_start = 1 # 0 = awal "_label", 1 = akhir "label_"
labels = {
    # "baseline1": {"start": 0, "seconds": 100, "baseline": 2},
    # "hs": {"start": 10, "seconds": 100, "baseline": 2},
    "bengbeng": {"start": 10, "seconds": 29, "baseline": 2},
    # "marjan": {"start": 10, "seconds": 45, "baseline": 2},
    # "ns_gandaria": {"start": 10, "seconds": 30, "baseline": 2},
    # "ns_jeruk": {"start": 10, "seconds": 30, "baseline": 2},
    "pucuk": {"start": 10, "seconds": 30, "baseline": 2},
    "shopee": {"start": 10, "seconds": 15, "baseline": 2},
    # "lokalate": {"start": 10, "seconds": 30, "baseline": 2},
    # "ts_nana": {"start": 10, "seconds": 30, "baseline": 2},
    "ts_pat": {"start": 10, "seconds": 30, "baseline": 2},
    # "honda": {"start": 10, "seconds": 108, "baseline": 2},
    #pengecoh
    # "abc": {"start": 10, "seconds": 30, "baseline": 2},
    # "bearbrand": {"start": 10, "seconds": 15, "baseline": 2},
    # "hilo_platinum": {"start": 10, "seconds": 30, "baseline": 2},
    # "hilo_many": {"start": 10, "seconds": 30, "baseline": 2},
    # "bukalapak": {"start": 10, "seconds": 90, "baseline": 2},
    # "dancow": {"start": 10, "seconds": 42, "baseline": 2},
    # "ensure": {"start": 10, "seconds": 30, "baseline": 2},
    # "gerry": {"start": 10, "seconds": 30, "baseline": 2},
    # "indomie": {"start": 10, "seconds": 29, "baseline": 2},
    # "kopikenangan": {"start": 10, "seconds": 60, "baseline": 2},
    # "marimas": {"start": 10, "seconds": 33, "baseline": 2},
    # "mmpo": {"start": 10, "seconds": 30, "baseline": 2},
    # "nescafe": {"start": 10, "seconds": 90, "baseline": 2},
    # "oreo": {"start": 10, "seconds": 60, "baseline": 2},
    # "samsung": {"start": 10, "seconds": 30, "baseline": 2},
    # "smartfren": {"start": 10, "seconds": 41, "baseline": 2},
    # "tokped": {"start": 10, "seconds": 15, "baseline": 2},
    # "torabika": {"start": 10, "seconds": 30, "baseline": 2},
    # "tehbotol": {"start": 10, "seconds": 30, "baseline": 2}
}
# labels = {
#     "bengbeng": {"start": 10, "seconds": 5}
# }

ext_raw = ".xlsx"

do_psd = False #jadikan True jika mau jalanin proses PSD
do_analyze_encode = False #jadikan True jika mau jalanin analisa encode juga
train_encode = 80 #berapa persentase data encode yang digunakan untuk train (train test split)

live_filename = "Live_EEG" #default filename

timer = 5 #berapa detik perekaman live EEG direkam

second_start = 0 #berapa detik pertama yang dipotong dari data mentah untuk diproses
seconds = 5 #berapa detik data yang akan diproses

measurement_time = 5 #detik untuk live graph recording (coming soon)
interval_sec = 0.1 #interval untuk live graph recording (coming soon)

sample_frequency = 250 #sample frequency default OpenBCI = 250
sample_rate = 256 #sampling data per detik OpenBCI = 256
notch_freq = 50.0 #notch frequency
notch_freqs = [50, 60, 100, 120, 150] #notch frequency
quality_factor = 30.0
lowcut = 1
highcut = 49
order = 4
chunks = 0.2 #detik chunk, harus kelipatan 2 (0.125 = 1/8)

treshold_ica = True #potong ICA jika data ICA melewati atau kurang dari batas treshold
high_ica = 100
low_ica = -100

emg_threshold_factor = 2
flat_variance_threshold = 1e-6

iter_seconds = 4 #hiraukan

print_debug = False
print_raw = False
print_filter = False
print_butter = False
print_ica = False
print_psd = True
### End Form Input ####

final_graph = True

if not os.path.exists(directory_ica):
    os.makedirs(directory_ica)
if not os.path.exists(directory_ica_plot):
    os.makedirs(directory_ica_plot)
if not os.path.exists(directory_ica_fif):
    os.makedirs(directory_ica_fif)
if not os.path.exists(directory_psd):
    os.makedirs(directory_psd)
if not os.path.exists(directory_graph):
    os.makedirs(directory_graph)
if not os.path.exists(directory_raw):
    os.makedirs(directory_raw)
if not os.path.exists(directory_raw_plot):
    os.makedirs(directory_raw_plot)
if not os.path.exists(directory_mav):
    os.makedirs(directory_mav)
if not os.path.exists(directory_mav_plot):
    os.makedirs(directory_mav_plot)
if not os.path.exists(directory_mav_plot_time_frequency):
    os.makedirs(directory_mav_plot_time_frequency)
if not os.path.exists(directory_compare):
    os.makedirs(directory_compare)
if not os.path.exists(directory_encoded):
    os.makedirs(directory_encoded)
if not os.path.exists(directory_analysis):
    os.makedirs(directory_analysis)
if not os.path.exists(directory_analysis_confusion_matrix):
    os.makedirs(directory_analysis_confusion_matrix)
if not os.path.exists(directory_unique):
    os.makedirs(directory_unique)
if not os.path.exists(directory_empty):
    os.makedirs(directory_empty)
if not os.path.exists(directory_corrupt):
    os.makedirs(directory_corrupt)
if not os.path.exists(directory_incomplete):
    os.makedirs(directory_incomplete)
    
start_time = time.time()
end_time = time.time()

cut = list(range(1, len(channels)+1))
# print(cut)
dict_zip = dict(zip(cut, channels))
# print(dict_zip)

frame_start = second_start * sample_rate
frame_end = ((second_start + seconds) * sample_rate)

nyq = 0.5 * sample_frequency
low = lowcut / nyq
high = highcut / nyq

sub_freqs = {}
sub_freqs["Delta"] = {"Low":1, "High":4}
sub_freqs["Theta"] = {"Low":4, "High":8}
sub_freqs["Alpha"] = {"Low":8, "High":13}
sub_freqs["Beta"] = {"Low":13, "High":35}
sub_freqs["Gamma"] = {"Low":35, "High":49}

low_d, high_d = sub_freqs["Delta"]["Low"], sub_freqs["Delta"]["High"] # Delta
low_t, high_t = sub_freqs["Theta"]["Low"], sub_freqs["Theta"]["High"] # Theta
low_a, high_a = sub_freqs["Alpha"]["Low"], sub_freqs["Alpha"]["High"] # Alpha
low_b, high_b = sub_freqs["Beta"]["Low"], sub_freqs["Beta"]["High"] # Beta
low_g, high_g = sub_freqs["Gamma"]["Low"], sub_freqs["Gamma"]["High"] # Gamma

# b_notch, a_notch = signal.iirnotch(notch_freq, quality_factor, sample_frequency)
# b_butter,a_butter = scipy.signal.butter(order, [low, high], 'bandpass', analog=False)

# b_d, a_d = scipy.signal.butter(order, [low_d / nyq, high_d / nyq], 'bandpass', analog=False)
# b_t, a_t = scipy.signal.butter(order, [low_t / nyq, high_t / nyq], 'bandpass', analog=False)
# b_a, a_a = scipy.signal.butter(order, [low_a / nyq, high_a / nyq], 'bandpass', analog=False)
# b_b, a_b = scipy.signal.butter(order, [low_b / nyq, high_b / nyq], 'bandpass', analog=False)
# b_g, a_g = scipy.signal.butter(order, [low_g / nyq, high_g / nyq], 'bandpass', analog=False)

# n_c = len(channeling) - 2
# if n_c < 1:
#     n_c = 1
# ICA = FastICA(n_components=n_c)
# ICA = ICA(n_components=len(channeling), random_state=42, max_iter=800)

os_raw = os.fsencode(directory_raw)
os_raw_plot = os.fsencode(directory_raw_plot)
os_ica = os.fsencode(directory_ica)
os_processed = os.fsencode(directory_psd)
os_analysis = os.fsencode(directory_analysis)
os_mav = os.fsencode(directory_mav)
os_compare = os.fsencode(directory_compare)
os_encoded = os.fsencode(directory_encoded)
os_empty = os.fsencode(directory_empty)
os_incomplete = os.fsencode(directory_incomplete)

In [3]:
# Ads data and other parameters
# labels_data = {
#     'honda': {'emotion': 'interested', 'chunks': range(450, 476), 'start': 0, 'seconds': 108, 'color': 'red'},
#     'hs': {'emotion': 'interested', 'chunks': range(250, 276), 'start': 0, 'seconds': 100, 'color': 'blue'},
#     'marjan': {'emotion': 'interested', 'chunks': range(195, 221), 'start': 0, 'seconds': 45, 'color': 'green'},
#     'baseline1': {'emotion': 'neutral', 'chunks': range(50, 101), 'start': 0, 'seconds': 100, 'color': 'brown'},
#     'ts_pat': {'emotion': 'meh', 'chunks': range(110, 136), 'start': 0, 'seconds': 30, 'color': 'purple'},
#     'ts_nana': {'emotion': 'meh', 'chunks': range(90, 116), 'start': 0, 'seconds': 30, 'color': 'cyan'},
#     'ns_jeruk': {'emotion': 'meh', 'chunks': range(20, 46), 'start': 0, 'seconds': 30, 'color': 'orange'},
# }
# classifiers_data = {
#     'honda': {'emotion': 'interested', 'chunks': range(450, 476), 'start': 0, 'seconds': 108, 'color': 'red'},
#     'hs': {'emotion': 'interested', 'chunks': range(250, 276), 'start': 0, 'seconds': 100, 'color': 'blue'},
#     'marjan': {'emotion': 'interested', 'chunks': range(195, 221), 'start': 0, 'seconds': 45, 'color': 'green'},
#     # 'baseline1': {'emotion': 'neutral', 'chunks': range(50, 101), 'start': 0, 'seconds': 100, 'color': 'brown'},
#     'ts_pat': {'emotion': 'meh', 'chunks': range(110, 136), 'start': 0, 'seconds': 30, 'color': 'purple'},
#     'ts_nana': {'emotion': 'meh', 'chunks': range(90, 116), 'start': 0, 'seconds': 30, 'color': 'cyan'},
#     'ns_jeruk': {'emotion': 'meh', 'chunks': range(20, 46), 'start': 0, 'seconds': 30, 'color': 'orange'},
# }
labels_data = {
    'bengbeng': {'emotion': 'memorable', 'chunks': range(450, 476), 'start': 0, 'seconds': 29, 'color': 'red'},
    'shopee': {'emotion': 'memorable', 'chunks': range(250, 276), 'start': 0, 'seconds': 15, 'color': 'orange'},
    'baseline1': {'emotion': 'neutral', 'chunks': range(50, 101), 'start': 0, 'seconds': 100, 'color': 'green'},
    'ts_pat': {'emotion': 'ordinary', 'chunks': range(110, 136), 'start': 0, 'seconds': 30, 'color': 'purple'},
    'pucuk': {'emotion': 'ordinary', 'chunks': range(90, 116), 'start': 0, 'seconds': 30, 'color': 'brown'},
}
classifiers_data = {
    'bengbeng': {'emotion': 'memorable', 'chunks': range(450, 476), 'start': 0, 'seconds': 29, 'color': 'red'},
    'shopee': {'emotion': 'memorable', 'chunks': range(250, 276), 'start': 0, 'seconds': 15, 'color': 'orange'},
    # 'baseline1': {'emotion': 'neutral', 'chunks': range(50, 101), 'start': 0, 'seconds': 100, 'color': 'green'},
    'ts_pat': {'emotion': 'ordinary', 'chunks': range(110, 136), 'start': 0, 'seconds': 30, 'color': 'purple'},
    'pucuk': {'emotion': 'ordinary', 'chunks': range(90, 116), 'start': 0, 'seconds': 30, 'color': 'brown'},
}
# corrupts_data = [
#     {'label': 'ns_mangga', 'participant': 'dataresponden31'},
# ]
corrupts_data = [
    # {'label': 'marjan', 'participant': 'dataresponden3', 'emotion': 'meh'},
]

channels_interest = ['T7', 'T8', 'P7', 'P8', 'O1', 'O2']
subbands_interest = ['Alpha', 'Beta', 'Gamma']
features_full = [
    'PSD', 'Mean', 'Mean Absolute Value',
    'Standard Deviation', 'Hjorth Activity',
    'Hjorth Mobility', 'Hjorth Complexity',
    'Skewness', 'Kurtosis',
    'Petrosian', 'Higuchi', 'Classic Entropy',
    'Sample Entropy', 'Spectral Entropy',
    'Fourier Transform', 'Wavelet Transform',
    'Peak to Peak', 'Zero Crossing Rate'
]
# features_interest = ['PSD', 'Mean', 'Mean Absolute Value', 'Standard Deviation',
#                      'Hjorth Activity', 'Hjorth Mobility', 'Hjorth Complexity',
#                      'Spectral Entropy', 'Peak to Peak', 'Zero Crossing Rate',
#                      'Kurtosis', 'Fourier Transform', 'Wavelet Transform',
# ]
features_interest = [
    'PSD'
]
classifier_interest = [
    ['P7_Alpha', 'P8_Alpha', 'T7_Alpha', 'T8_Alpha', 'O1_Alpha', 'O2_Alpha'],
    ['P7_Beta', 'P8_Beta', 'T7_Beta', 'T8_Beta', 'O1_Beta', 'O2_Beta'],
    ['P7_Gamma', 'P8_Gamma', 'T7_Gamma', 'T8_Gamma', 'O1_Gamma', 'O2_Gamma'],
    # ['O1_Gamma', 'O2_Gamma'],
    # ['F7_Beta', 'F8_Beta'],
    # ['F7_Beta', 'F8_Beta', 'O1_Gamma', 'O2_Gamma'],
    # ['F7_Alpha', 'F8_Alpha', 'O1_Alpha', 'O2_Alpha'],
    # ['F7_Theta', 'F8_Theta'],
    # ['O1_Alpha', 'O2_Alpha'],
    # ['F7_Alpha', 'F8_Alpha'],
    # ['F7_Gamma', 'F8_Gamma'],
    # ['T7_Theta', 'T8_Theta'],
    # ['T7_Alpha', 'T8_Alpha'],
    # ['P7_Theta', 'P8_Theta'],
    # ['P7_Alpha', 'P8_Alpha'],
    # ['F7_Beta', 'F8_Beta', 'O1_Beta', 'O2_Beta'],
    # ['O1_Gamma'], ['O2_Gamma'], ['F7_Beta'], ['F8_Beta'], ['F7_Alpha'], ['F8_Alpha'],
    # ['O1_Alpha'], ['O2_Alpha'], ['F7_Theta'], ['F8_Theta'], ['F7_Gamma'], ['F8_Gamma'],
    # ['T7_Theta'], ['T8_Theta'], ['P7_Theta'], ['P8_Theta'], ['P7_Alpha'], ['P8_Alpha'], ['O1_Beta'], ['O2_Beta'],
    # ['F7_Alpha', 'F8_Alpha', 'T7_Alpha', 'T8_Alpha', 'P7_Alpha', 'P8_Alpha', 'O1_Alpha', 'O2_Alpha'],
    # ['F7_Beta', 'F8_Beta', 'T7_Beta', 'T8_Beta', 'P7_Beta', 'P8_Beta', 'O1_Beta', 'O2_Beta'],
    # ['F7_Gamma', 'F8_Gamma', 'T7_Gamma', 'T8_Gamma', 'P7_Gamma', 'P8_Gamma', 'O1_Gamma', 'O2_Gamma'],
    # ['F7_Beta', 'T7_Theta', 'P7_Alpha'],
    # ['F8_Beta', 'T8_Theta', 'P8_Alpha'],
    # ['F7_Gamma', 'T8_Theta', 'O2_Alpha'],
    # ['F7_Alpha', 'T7_Alpha', 'P7_Alpha', 'O1_Alpha'],
    # ['F8_Alpha', 'T8_Alpha', 'P8_Alpha', 'O2_Alpha'],
    # ['F7_Alpha', 'O1_Alpha'],
    # ['F7_Beta', 'O1_Beta'],
    # ['F7_Theta', 'O1_Theta'],
    # ['F8_Alpha', 'O2_Alpha'],
    # ['F8_Beta', 'O2_Beta'],
    # ['F8_Theta', 'O2_Theta'],
    # ['F7_Alpha', 'P7_Alpha'],
    # ['F8_Beta', 'P8_Beta'],
    # ['T7_Theta', 'O1_Theta'],
    # ['T8_Gamma', 'O2_Gamma'],
]

# vectors = ['mean', 'var', 'ptp']
vectors = ['mean']

classifiers_list = {
    # Non-Tree:
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'SVC': SVC(),
    'MLP Classifier': MLPClassifier(),
    
    # Tree-based:
    'Random Forest': RandomForestClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'AdaBoost': AdaBoostClassifier(algorithm='SAMME'),
    'ExtraTrees': ExtraTreesClassifier(),
    
    # Boosting algorithms:
    'XGBoost': XGBClassifier(),
    'LightGBM': LGBMClassifier(),
    
    # Linear and Naive Bayes:
    'LDA': LinearDiscriminantAnalysis(),
    'Naive Bayes': GaussianNB(),
}
classifiers = {
    'Random Forest': RandomForestClassifier(),
    'SVC': SVC(),
    'MLP Classifier': MLPClassifier(),
    # # 'XGBoost': XGBClassifier(eval_metric='mlogloss', use_label_encoder=False),
    # # 'LightGBM': LGBMClassifier(),
    'Naive Bayes': GaussianNB(),
    # # 'AdaBoost': AdaBoostClassifier(algorithm='SAMME'),
    # # 'ExtraTrees': ExtraTreesClassifier(),
    # # 'Logistic Regression': LogisticRegression(max_iter=1000),
    # # 'LDA': LinearDiscriminantAnalysis(),
    # 'K-Nearest Neighbors': KNeighborsClassifier(),
    # # 'Decision Tree': DecisionTreeClassifier(),
}

# Define the directories and other configurations
input_folder_ica = directory_ica
input_folder_mav = directory_mav
output_folder = directory_analysis
output_folder_scatter_base = output_folder + '/scatter'

# Ensure base output directory exists
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_folder_scatter_base, exist_ok=True)

In [4]:
def getLiveData():
    global timer
    global channels
    global dict_zip
    global directory_raw
    global live_filename
    
    BoardShim.enable_dev_board_logger()

    serial_port = "COM3"
    board_id = BoardIds.CYTON_BOARD.value
    
    params = BrainFlowInputParams()
    params.serial_port = serial_port
    
    board = BoardShim(board_id, params)
    
    board.prepare_session()
    board.start_stream()
    time.sleep(timer)
    data = board.get_board_data()
    board.stop_stream()
    board.release_session()
    
    df_live = pd.DataFrame(np.transpose(data))
    
    N = len(channels) + 1
    
    dfRaw = df_live.iloc[1: , 1:N].rename(columns = dict_zip, inplace = False)
    
    fn = directory_raw + "/" + live_filename + ".csv"

    DataFilter.write_file(data, fn, 'w')
    
    return dfRaw

def getChannel(idx):
    global dict_zip
    
    channel = dict_zip.get(idx+1)
    if channel is None:
        channel = str(idx+1)
    return channel

def pureFilename(filepath, idx = ""):
    global ext_raw
    
    name = filepath.replace(" ", "_")
    name = name.replace(ext_raw, "")
    if idx != "":
        name = name + "_" + getChannel(idx)
    return name

def chunky(data, n, timestamps):
    # Initialize list for storing the chunks
    all_chunks = []

    # Convert chunk duration from seconds to milliseconds (e.g., 0.2 seconds = 200 milliseconds)
    chunk_milliseconds = n * 1000
    
    # Initialize variables for tracking current chunk boundaries
    current_chunk_start = timestamps.iloc[0]  # Start from the first timestamp
    current_chunk_end = current_chunk_start + chunk_milliseconds  # Set the end of the first chunk
    
    chunk_data = []  # List to accumulate data for the current chunk

    # for idx, timestamp in enumerate(timestamps):
    for i in range(len(timestamps)):
        current_time = timestamps.iloc[i]

        # Check if the current time is within the current chunk
        if current_time < current_chunk_end:
            # Accumulate the data for the current chunk
            chunk_data.append(data.iloc[i])
        else:
            # If the current time exceeds the chunk boundary, finalize the current chunk
            all_chunks.append(chunk_data)  # Convert chunk to numpy array and store it
            
            # Reset chunk data and move to the next chunk
            chunk_data = [data.iloc[i]]  # Start the new chunk with the current data point

            # Move the chunk boundaries forward
            while current_time >= current_chunk_end:
                current_chunk_start = current_chunk_end
                current_chunk_end = current_chunk_start + chunk_milliseconds

    # Handle any remaining data in the final chunk
    if chunk_data:
        all_chunks.append(chunk_data)

    return all_chunks

def chunky2(data, n):
    global sample_rate
    seconds = len(data) / sample_rate
    split = int(math.ceil(seconds / n))
    return np.array_split(data, split)

def meanMulti(directory):
    means = {}
    for file in directory:
        for idx, data in enumerate(file):
            channel = getChannel(idx)
            mean = sum(data) / len(data)
            if channel in means:
                means[channel].append(mean)
            else:
                means[channel] = [mean]
        
    for key in means:
        mean = sum(means[key]) / len(means[key])
        if mean > 0.01:
            print(key + " = " + str(round(mean, 2)))
        else:
            print(key + " = " + str(round(mean, 10)))
            
def getSampleRate(timestamps):
    global sample_frequency
    global highcut
    global nyq
    global low
    global high
    global low_d, high_d
    global low_t, high_t
    global low_a, high_a
    global low_b, high_b
    global low_g, high_g
    global b_notch, a_notch
    global b_butter,a_butter
    global b_d, a_d
    global b_t, a_t
    global b_a, a_a
    global b_b, a_b
    global b_g, a_g
    
    start = 0
    end = 0
    count = 0
    for timestamp in timestamps:
        if start == 0:
            start = float(timestamp)
        
        dif = float(timestamp) - start
        
        if dif >= 1000.0:
            break
        else:
            count = count + 1
    # if sample_frequency != count and count > 0:
    #     sample_frequency = count

    #     nyq = 0.5 * sample_frequency
    #     if nyq < highcut:
    #         nyq = highcut
        
    #     low = lowcut / nyq
    #     high = highcut / nyq

    #     b_notch, a_notch = signal.iirnotch(notch_freq, quality_factor, sample_frequency)
    #     b_butter,a_butter = scipy.signal.butter(order, [low, high], 'bandpass', analog=False)

    #     b_d, a_d = scipy.signal.butter(order, [low_d / nyq, high_d / nyq], 'bandpass', analog=False)
    #     b_t, a_t = scipy.signal.butter(order, [low_t / nyq, high_t / nyq], 'bandpass', analog=False)
    #     b_a, a_a = scipy.signal.butter(order, [low_a / nyq, high_a / nyq], 'bandpass', analog=False)
    #     b_b, a_b = scipy.signal.butter(order, [low_b / nyq, high_b / nyq], 'bandpass', analog=False)
    #     b_g, a_g = scipy.signal.butter(order, [low_g / nyq, high_g / nyq], 'bandpass', analog=False)
    
    return count

def cleanRawData(df):
    global channels
    
    #clean raw data for excel imotion only
    raw_channels = ['Timestamp']
    for x in range(len(channels)):
        raw_channels.append('CH.' + str(x))

#     # Interpolate the missing values
#     df_interpolated = df[raw_channels].interpolate()
    
#     print("df_interpolated 1")
#     print(df_interpolated)

#     # Optionally, fill any remaining NaNs with forward/backward filling
#     df_interpolated = df_interpolated.fillna(method='ffill').fillna(method='bfill')
    
#     print("df_interpolated 2")
#     print(df_interpolated)
#     return df_interpolated
    
    # df = df[df['CH.0'].notna()]
    
    # return df[raw_channels].reset_index(drop=True)
    
    # Create a renaming dictionary to map 'CH.x' to the corresponding channel name
    renaming_dict = {f'CH.{x}': channels[x] for x in range(len(channels))}
    
    # Filter out rows where 'CH.0' is NaN
    df = df[df['CH.0'].notna()]
    
    # Select only the relevant columns and rename them
    df = df[raw_channels].rename(columns=renaming_dict)
    return df.reset_index(drop=True)
        
def readRawData(filename):
    global cut
    global dict_zip
    
    global ext_raw
    global directory_raw
    global directory_raw_plot
    
    global print_debug
    global print_raw

    filepath = directory_raw + "/" + filename
    
    if print_debug:
        print()
        print("Read Raw Data")
        print()
    
    skiprows = 7
    sep = ", "
    if ext_raw == '.xlsx':
        df =  pd.read_excel(filepath)
        dfRaw = cleanRawData(df)
    else:
        if ext_raw == '.csv':
            skiprows = 1
            sep = "	"
        elif ext_raw == '.txt':
            skiprows = 7
            sep = ", "
        df =  pd.read_csv(filepath, skiprows = skiprows, header=None, sep=sep, engine="python")
#         print("\n\ndf")
#         print(df)
#         print("\n\n\n")
        dfRaw = df[cut]
        dfRaw = dfRaw.rename(columns = dict_zip, inplace = False)
        
#     dfRaw = df[cut]
#     dfRaw = dfRaw.rename(columns = dict_zip, inplace = False)
    
    if print_raw:
        print(dfRaw)
    
    return dfRaw

def plotRawData(dfRaw, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Raw Data")
        print()
    
    dfRaw.plot(kind='line',figsize=(15,6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Raw " + filename)
    plt.xlabel("Frame")
    plt.ylabel("Value")
    plt.legend()
    plt.show()
    
def filterRawData(dfRaw):
    global b_notch
    global a_notch
    
    global print_debug
    global print_filter
    
    if print_debug:
        print()
        print("Filter Raw")
        print()

    outputSignals = []
    
    for idx, raw in dfRaw.items():
        filt = signal.filtfilt(b_notch, a_notch, raw)
        dfFilt = pd.DataFrame(filt)
        outputSignals.append(dfFilt)
        
        if print_filter:
            print(str(idx+1) + " = ")
            print(dfFilt)
            print()

          
#     print("Output Signals:\n\n")
#     print(outputSignals)
#     print("\nEND OS\n\n")
    
    #dictSemua = {'T3': outputSignals[0], 'T4': outputSignals[1], 'T5': outputSignals[2], 'T6': outputSignals[3], 'O1': outputSignals[4],'O2': outputSignals[5]} 
    
    #print("OutputSignals[0]")
    #print(outputSignals[0])
    #print("\nEND df OS\n")
    #print(dictSemua)
    
    #isiSemua = pd.DataFrame(dictSemua)
    
    #print(isiSemua)
                
    #plt.figure(2)
    #plt.style.use('seaborn-colorblind')
    #plt.style.use('seaborn-whitegrid')
    #plt.plot(outputSignals[0])
    #plt.plot(isiSemua)
    #plt.ylabel("Output Signals")
    #plt.show()
    
    return outputSignals

def plotFilterData(outputSignals, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Filter Data")
        print()
    
    data = []
    plt.figure(figsize=(15,6))
    for idx, os in enumerate(outputSignals):
        plt.plot(os, label = getChannel(idx))
#         data.append(os)
#     data.plot(figsize=(15, 6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Notch Filter " + filename)
    plt.xlabel("Frame")
    plt.ylabel("Value")
    plt.legend()
    plt.show()
            
def butterData(outputSignals):
    global b_butter
    global a_butter
    
    global print_debug
    global print_butter
    
    if print_debug:
        print()
        print("Butter Data")
        print()
    
    butterData = []
    
    for idx, outputSignal in enumerate(outputSignals):
        y = scipy.signal.filtfilt(b_butter, a_butter, outputSignal, axis=0)
        dfButter = pd.DataFrame(y)
        butterData.append(dfButter)
        
        if print_butter:
            print(str(idx+1) + " = ")
            print(dfButter)
            print()
          
#     print("Butter Data:\n\n")
#     print(butterData)
#     print("\nEND BD\n\n")
                    
    #plt.figure(3)
    #plt.style.use('seaborn-colorblind')
    #plt.style.use('seaborn-whitegrid')
    #plt.plot(outputSignals[0])
    #plt.plot(butterData)
    #plt.ylabel("Butter Data")
    #plt.show()
    
    return butterData

def plotButterData(butterData, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Butter Data")
        print()
    
    data = []
    plt.figure(figsize=(15,6))
    for idx, bd in enumerate(butterData):
        plt.plot(bd, label = getChannel(idx))
#         data.append(bd)
#     data.plot(figsize=(15, 6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Bandpass Filter " + filename)
    plt.xlabel("Frame")
    plt.ylabel("Value")
    plt.legend()
    plt.show()

def smoothClipping(data):
    global high_ica  # Upper amplitude threshold (e.g., 100)
    global low_ica   # Lower amplitude threshold (e.g., -100)
    
    smoothed_data = data.copy()

    # Iterate over each channel (in case of multiple channels)
    for channel in range(smoothed_data.shape[1]):
        high_interpolate = None
        low_interpolate = None
        prev_original_value = 0.0
        
        # Iterate over each time point in the data
        for i in range(smoothed_data.shape[0]):
            original_value = smoothed_data[i, channel]
            
            # Handle values exceeding the high threshold
            if original_value > high_ica:
                if high_interpolate is None:
                    high_interpolate = high_ica  # Start at the threshold
                if prev_original_value > high_ica and prev_original_value > original_value:
                    high_interpolate -= 0.05  # Decrease from the previous interpolation
                else:
                    high_interpolate = high_ica  # Stay at the threshold
                smoothed_data[i, channel] = high_interpolate

            # Handle values exceeding the low threshold
            elif original_value < low_ica:
                if low_interpolate is None:
                    low_interpolate = low_ica  # Start at the low threshold
                if prev_original_value < low_ica and prev_original_value < original_value:
                    low_interpolate += 0.05  # Increase from the previous interpolation
                else:
                    low_interpolate = low_ica  # Stay at the low threshold
                smoothed_data[i, channel] = low_interpolate

            # Reset interpolation when within limits
            else:
                high_interpolate = None
                low_interpolate = None

            # Update the previous original value for comparison in the next iteration
            prev_original_value = original_value

    return smoothed_data
    
def computeSNR(clean_signal, noisy_signal):
    # Compute power of the signal (clean)
    clean_power = np.mean(clean_signal ** 2)
    noisy_power = np.mean(noisy_signal ** 2)
    
    # Compute power of the noise (difference between noisy and clean)
    noise = noisy_signal - clean_signal
    noise_power = np.mean(noise ** 2)
    
    # Compute SNR in dB
    snr = 10 * np.log10(noisy_power / noise_power) if noise_power > 0 else np.inf
    
    # Compute linear SNR, bounded between 0 and 1
    if clean_power + noise_power > 0:
        linear_snr = clean_power / (clean_power + noise_power)
    else:
        linear_snr = 0  # In case there's no signal or noise, SNR is 0
        
    return snr, linear_snr * 100

def computeFFT(data, sfreq, zoom_out = False):
    global lowcut
    global highcut

    min_freq = lowcut
    max_freq = highcut
    if zoom_out:
        min_freq = 0.1
        max_freq = 55
    
    # Compute FFT
    fft_result = np.fft.fft(data)
    
    # Get corresponding frequencies
    freqs = np.fft.fftfreq(len(data), d=1/sfreq)
    
    # Get magnitude of the FFT
    magnitude = np.abs(fft_result)
    
    # Use only the positive frequencies
    positive_freqs = freqs[freqs >= 0]
    positive_magnitude = magnitude[:len(positive_freqs)]
    
    # Filter out frequencies less than min_freq and greater than max_freq
    valid_indices = (positive_freqs >= min_freq) & (positive_freqs <= max_freq)
    filtered_freqs = positive_freqs[valid_indices]
    filtered_magnitude = positive_magnitude[valid_indices]
    
    return filtered_freqs, filtered_magnitude

def plotFFTsubband(plot_data, directory, filename):
    global sample_rate
    global channeling
    global channeling_colors
    global subbanding
    global sub_freqs

    sfreq = sample_rate
    
    data_freq = {}

    for cn in channeling:
        data_freq[cn] = {}
        
        for subband in subbanding:
            # Extract the data from the MNE Raw object
            data = plot_data[cn][subband].flatten()
        
            # Apply FFT
            fft_data = np.fft.fft(data)
            freqs = np.fft.fftfreq(len(data), 1/sfreq)
        
            # Get the positive frequencies and their corresponding magnitudes
            positive_freqs = freqs[:len(freqs)//2]
            magnitude = np.abs(fft_data[:len(fft_data)//2])
            
            bw_index = np.where((freqs >= sub_freqs[subband]['Low']) & (freqs < sub_freqs[subband]['High']))  # Find frequencies around 1-5 Hz
            bw_freqs = positive_freqs[bw_index]
            bw_magnitude = magnitude[bw_index]
    
            data_freq[cn][subband] = {'freq': bw_freqs, 'magnitude': bw_magnitude}
            
            # Plot FFT
            plt.figure(figsize=(10, 6))
            # plt.plot(positive_freqs, magnitude, color=channeling_colors.get(cn, 'black'))
            plt.plot(bw_freqs, bw_magnitude, color=channeling_colors.get(cn, 'black'))
            plt.xlabel('Frequency (Hz)')
            plt.ylabel('Magnitude')
            plt.title(f'FFT of ICA {cn} {subband}')
            plt.grid(True)
            plt.tight_layout()
            
            # Save the plot
            if not os.path.exists(directory):
                os.makedirs(directory)
            output_path = os.path.join(directory, f'{filename}_{cn}_{subband}.png')
            plt.savefig(output_path)
            plt.close()
        
        plt.figure(figsize=(10, 6))
        for subband in subbanding:
            plt.plot(data_freq[cn][subband]['freq'], data_freq[cn][subband]['magnitude'], color=channeling_colors.get(cn, 'black'))
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Magnitude')
        plt.title(f'FFT of ICA {cn}')
        plt.grid(True)
        plt.tight_layout()
        
        # Save the plot
        if not os.path.exists(directory):
            os.makedirs(directory)
        output_path = os.path.join(directory, f'{filename}_{cn}.png')
        plt.savefig(output_path)
        plt.close()
    
    for subband in subbanding:
        plt.figure(figsize=(10, 6))
        for cn in channeling:
            plt.plot(data_freq[cn][subband]['freq'], data_freq[cn][subband]['magnitude'], color=channeling_colors.get(cn, 'black'))
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Magnitude')
        plt.title(f'FFT of ICA {subband}')
        plt.grid(True)
        plt.tight_layout()
    
        # Save the plot
        if not os.path.exists(directory):
            os.makedirs(directory)
        output_path = os.path.join(directory, f'{filename}_{subband}.png')
        plt.savefig(output_path)
        plt.close()
        
    plt.figure(figsize=(10, 6))
    for cn in channeling:
        for subband in subbanding:
            plt.plot(data_freq[cn][subband]['freq'], data_freq[cn][subband]['magnitude'], color=channeling_colors.get(cn, 'black'))
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.title(f'FFT of ICA')
    plt.grid(True)
    plt.tight_layout()
    
    # Save the plot
    if not os.path.exists(directory):
        os.makedirs(directory)
    output_path = os.path.join(directory, f'{filename}.png')
    plt.savefig(output_path)
    plt.close()
    
def plotFFTsubband2(raw_band, directory, filename, cn, bw, low, high):
    global sample_rate
    global channeling_colors

    sfreq = sample_rate
    
    # Extract the data from the MNE Raw object
    data = raw_band.get_data().flatten()

    # Apply FFT
    fft_data = np.fft.fft(data)
    freqs = np.fft.fftfreq(len(data), 1/sfreq)

    # Get the positive frequencies and their corresponding magnitudes
    positive_freqs = freqs[:len(freqs)//2]
    magnitude = np.abs(fft_data[:len(fft_data)//2])
    
    bw_index = np.where((freqs >= low) & (freqs < high))  # Find frequencies around 1-5 Hz
    bw_freqs = positive_freqs[bw_index]
    bw_magnitude = magnitude[bw_index]
    
    # Plot FFT
    plt.figure(figsize=(10, 6))
    # plt.plot(positive_freqs, magnitude, color=channeling_colors.get(cn, 'black'))
    plt.plot(bw_freqs, bw_magnitude, color=channeling_colors.get(cn, 'black'))
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.title(f'FFT of ICA {cn} {bw}')
    plt.grid(True)
    plt.tight_layout()
    
    # Save the plot
    if not os.path.exists(directory):
        os.makedirs(directory)
    output_path = os.path.join(directory, filename)
    plt.savefig(output_path)
    plt.close()
    
def plotmne(raw_data, directory, title, filename, zoom_out = False):
    global channeling
    
    sfreq = raw_data.info['sfreq']
    
    f7_data = raw_data.get_data(picks='F7')[0]
    f8_data = raw_data.get_data(picks='F8')[0]
    p7_data = raw_data.get_data(picks='P7')[0]
    p8_data = raw_data.get_data(picks='P8')[0]
    t7_data = raw_data.get_data(picks='T7')[0]
    t8_data = raw_data.get_data(picks='T8')[0]
    o1_data = raw_data.get_data(picks='O1')[0]
    o2_data = raw_data.get_data(picks='O2')[0]
    
    df = pd.DataFrame({'F7': f7_data, 'F8': f8_data, 'P7': p7_data, 'P8': p8_data, 'T7': t7_data, 'T8': t8_data, 'O1': o1_data, 'O2': o2_data})
    df_freq = pd.DataFrame({'F7': [], 'F8': [], 'P7': [], 'P8': [], 'T7': [], 'T8': [], 'O1': [], 'O2': []})

    directory_time = directory + '/domain_time'
    if not os.path.exists(directory_time):
        os.makedirs(directory_time)
        
    directory_freq = directory + '/domain_freq'
    if not os.path.exists(directory_freq):
        os.makedirs(directory_freq)

    for cn in channeling:
        plotSave(df[[cn]], 100, 7, directory_time, f'{title}_{cn}_{filename}', "Sample", "Amplitude")
        
        # freqs, magnitude = computeFFT(df[cn], sfreq)
        # fft_df = pd.DataFrame({f'{cn}_Frequency': freqs, f'{cn}_Magnitude': magnitude})
        
        # # Use plotSave to plot frequency vs magnitude
        # plotSave(fft_df[[f'{cn}_Frequency', f'{cn}_Magnitude']], 100, 7, directory_freq, f'{title}_{cn}_freq_{filename}', "Frequency (Hz)", "Magnitude")
                
        # Compute FFT and get frequency and magnitude
        freqs, magnitude = computeFFT(df[cn], sfreq, zoom_out)
        df_freq[cn] = {'freqs': freqs, 'magnitude': magnitude}
        
        # Plot FFT directly
        plt.figure(figsize=(10, 6))
        plt.plot(freqs, magnitude, color=channeling_colors.get(cn, 'black'))
        plt.title(f'FFT of {cn} - {title}')
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Magnitude')
        plt.grid(True)
        plt.tight_layout()
        
        # Save the plot
        output_path = os.path.join(directory_freq, f'{title}_{cn}_freq_{filename}.png')
        plt.savefig(output_path)
        plt.close()

        # Now, filter to isolate only the 1-5 Hz frequency component
        one_hz_index = np.where((freqs >= 1) & (freqs < 5))  # Find frequencies around 1-5 Hz
        one_hz_freqs = freqs[one_hz_index]
        one_hz_magnitude = magnitude[one_hz_index]
        
        # Plot only the 1-5 Hz frequency component
        plt.figure(figsize=(10, 4))
        plt.bar(one_hz_freqs, one_hz_magnitude, color=channeling_colors.get(cn, 'black'), width=0.05)
        plt.title(f'1-5 Hz Component of {cn} - {title}')
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Magnitude')
        plt.tight_layout()
        
        # Save the 1 Hz component plot
        output_path_1hz = os.path.join(directory_freq, f'{title}_{cn}_freq_5Hz_{filename}.png')
        plt.savefig(output_path_1hz)
        plt.close()

    # Plot FFT directly
    plt.figure(figsize=(10, 6))
    for cn in channeling:
        plt.plot(df_freq[cn]['freqs'], df_freq[cn]['magnitude'], color=channeling_colors.get(cn, 'black'))
    plt.title(f'FFT of {title}')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.grid(True)
    plt.tight_layout()
    
    # Save the plot
    output_path = os.path.join(directory_freq, f'{title}_freq_{filename}.png')
    plt.savefig(output_path)

    plotSave(df, 100, 7, directory_time, f'{title}_Complete_{filename}', "Sample", "Amplitude")

def ICAmne(dfRaw, filename):
    global emg_threshold_factor
    global lowcut
    global highcut
    global notch_freqs
    global treshold_ica
    global eog_channels
    global channels
    global channeling
    global high_ica  # Upper amplitude threshold (e.g., 100)
    global low_ica   # Lower amplitude threshold (e.g., -100)
    global sample_rate
    global flat_variance_threshold  # Threshold for detecting flat data
    global chunks
    global directory_ica_fif
    global directory_ica_plot

    snr_result = {}
    
    window_size = int(sample_rate * chunks)  # Window size for variance calculation (e.g., 0.5 seconds * sample_rate)    
    
    # # ASR parameters
    # rms_threshold = 3  # Threshold for RMS deviation
    # reference_window = int(sample_rate * 1)  # 1-second reference window for calculating RMS

    # Initialize ICA with the number of components equal to the number of channels
    ica_obj = ICA(n_components=len(channeling), random_state=42, max_iter=800)
    ICAData = []

    butterData = dfRaw.to_numpy().T

    # Convert the data into a NumPy array and create MNE info and raw data
    data = butterData
    # info = mne.create_info(ch_names=channeling, sfreq=sample_rate, ch_types='eeg')
    info = mne.create_info(ch_names=channels, sfreq=sample_rate, ch_types='eeg')
    raw = mne.io.RawArray(data, info)

    # Set a standard montage to add sensor locations
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage)

    plotmne(raw, f'{directory_ica}/plot/{filename}', '01_Raw EEG', filename)

    # Apply Bandpass Filter from 1 to 49 Hz
    raw_bandpass = raw.copy().filter(l_freq=lowcut, h_freq=highcut, fir_design='firwin')

    # Compute and log SNR after bandpass filtering
    snr_bandpass, snr_bandpass_linear = computeSNR(raw_bandpass.get_data(), raw.get_data())

    plotmne(raw_bandpass, f'{directory_ica}/plot/{filename}', '02_Bandpass-Filtered EEG', filename)

    # Apply Notch Filter at 50, 60, 100, and 150 Hz
    raw_notch = raw_bandpass.copy().notch_filter(freqs=notch_freqs, method='spectrum_fit')

    # Compute and log SNR after notch filtering
    snr_notch, snr_notch_linear = computeSNR(raw_notch.get_data(), raw_bandpass.get_data())

    plotmne(raw_notch, f'{directory_ica}/plot/{filename}', '03_Notch-Filtered EEG', filename)

    # raw_asr = applyASR(raw_notch.copy())

    # Initialize ASR with the sampling frequency
    asr = ASR(sfreq=sample_rate, cutoff=15)

    # Fit the ASR model using the entire raw data (calibration step)
    asr.fit(raw_notch)

    # Apply the ASR transformation to clean the data
    raw_asr = asr.transform(raw_notch)
    
    # Create a new RawArray object with the cleaned data
    # raw_asr = mne.io.RawArray(asr_data, raw_notch.info)

    snr_asr, snr_asr_linear = computeSNR(raw_asr.get_data(), raw_notch.get_data())

    plotmne(raw_asr, f'{directory_ica}/plot/{filename}', '04_ASR-Filtered EEG', filename)

    raw_ica = raw_asr.copy()
    # Fit the ICA model to the modified raw data
    ica_obj.fit(raw_ica)
    
    save_dir = os.path.join(f'{directory_ica_plot}/{filename}/ica_components')
    os.makedirs(save_dir, exist_ok=True)
    
    # Save the components plot
    components_fig = ica_obj.plot_components(show=False)  # Do not show, but save
    components_fig.savefig(os.path.join(save_dir, f'{filename}_ICA_Components.png'))
    
    # Save the sources plot. Loop through the entire duration in 10-second chunks
    total_duration = raw_ica.times[-1]
    for start_time in range(0, int(total_duration), 10):
        stop_time = min(start_time + 10, total_duration) # Ensure we don't go past the total duration
        sources_fig = ica_obj.plot_sources(raw_ica, start=start_time, stop=stop_time, show=False)  # Do not show, but save
        sources_fig.savefig(os.path.join(save_dir, f'{filename}_ICA_Sources_{start_time}_{stop_time}.png'))
    sources_fig = ica_obj.plot_sources(raw_ica, start=0, stop=total_duration, show=False)  # Do not show, but save
    sources_fig.savefig(os.path.join(save_dir, f'{filename}_ICA_Sources_Full.png'))

    # Automatically find EOG-like artifact components using F7 and F8 as reference channels
    if eog_channels:
        eog_indices, eog_scores = ica_obj.find_bads_eog(raw_ica, ch_name=eog_channels)
    else:
        eog_indices = []
        eog_scores = None
        
    # Dynamic thresholding for variance, kurtosis, and peak-to-peak amplitude
    ica_sources = ica_obj.get_sources(raw_ica).get_data()

    # 1. Variance-based dynamic threshold (+ 2 = 95th percentile, + 3 = 99,7th percentile)
    variances = np.var(ica_sources, axis=1)
    var_mean = np.mean(variances)
    var_std = np.std(variances)
    dynamic_var_threshold = var_mean + 2 * var_std
    excluded_components_variance = np.where(variances > dynamic_var_threshold)[0]

    # 2. Kurtosis-based dynamic threshold (+ 2 = 95th percentile, + 3 = 99,7th percentile)
    kurtoses = np.apply_along_axis(stats.kurtosis, 1, ica_sources)
    kurt_mean = np.mean(kurtoses)
    kurt_std = np.std(kurtoses)
    dynamic_kurtosis_threshold = kurt_mean + 2 * kurt_std
    excluded_components_kurtosis = np.where(kurtoses > dynamic_kurtosis_threshold)[0]

    # 3. Peak-to-Peak Amplitude threshold
    peak_to_peak_amplitudes = np.ptp(ica_sources, axis=1)
    excluded_components_ptp = np.where(peak_to_peak_amplitudes > 150)[0]

    # 4. EMG Artifact Detection based on high-frequency power (30-100 Hz)
    psds, freqs = psd_array_welch(ica_sources, sfreq=sample_rate, fmin=30, fmax=100, n_fft=calculateNFFT(sample_rate))
    emg_power_per_component = np.mean(psds, axis=1)
    median_emg_power = np.median(emg_power_per_component)
    emg_threshold = 5 * median_emg_power
    emg_indices = np.where(emg_power_per_component > emg_threshold)[0]

    # Combine all artifact indices
    artifact_indices = np.unique(np.concatenate([
        eog_indices, 
        excluded_components_variance, 
        excluded_components_kurtosis, 
        excluded_components_ptp, 
        emg_indices
    ]))

    # If there are artifacts detected, mark them for exclusion
    if artifact_indices.any():
        ica_obj.exclude = artifact_indices

    # Apply ICA correction to remove artifacts from the raw data
    ica_obj.apply(raw_ica)

    snr_ica, snr_ica_linear = computeSNR(raw_ica.get_data(), raw_asr.get_data())

    snr_total, snr_total_linear = computeSNR(raw_ica.get_data(), raw.get_data())

    plotmne(raw_ica, f'{directory_ica}/plot/{filename}', '05_ICA-Filtered EEG', filename)

    snr_result = {
        'Source Data': filename,
        'Bandpass-Filtered SNR (dB)': snr_bandpass,
        'Bandpass-Filtered SNR (%)': round(snr_bandpass_linear, 2),
        'Notch-Filtered SNR (dB)': snr_notch,
        'Notch-Filtered SNR (%)': round(snr_notch_linear, 2),
        'ASR-Filtered SNR (dB)': snr_asr,
        'ASR-Filtered SNR (%)': round(snr_asr_linear, 2),
        'ICA-Filtered SNR (dB)': snr_ica,
        'ICA-Filtered SNR (%)': round(snr_ica_linear, 2),
        'Raw-ICA SNR (dB)': snr_total,
        'Raw-ICA SNR (%)': round(snr_total_linear, 2),
        'EOG Excluded': ', '.join(map(str, eog_indices)),
        'Variance Component': ', '.join(map(str, variances)),
        'Variance Threshold': dynamic_var_threshold,
        'Variance Excluded': ', '.join(map(str, excluded_components_variance)),
        'Kurtosis Component': ', '.join(map(str, kurtoses)),
        'Kurtosis Threshold': dynamic_kurtosis_threshold,
        'Kurtosis Excluded': ', '.join(map(str, excluded_components_kurtosis)),
        'Peak-to-Peak Component': ', '.join(map(str, peak_to_peak_amplitudes)),
        'Peak-to-Peak Excluded': ', '.join(map(str, excluded_components_ptp)),
        'EMG Component': ', '.join(map(str, emg_power_per_component)),
        'EMG Threshold': emg_threshold,
        'EMG Excluded': ', '.join(map(str, emg_indices)),
        'ICA Excluded': ', '.join(map(str, artifact_indices)),
    }

    # Get the cleaned data as a NumPy array from the fully corrected version
    cleaned_data = raw_ica.get_data().T

    # Apply amplitude thresholding if needed
    # if treshold_ica:
    #     cleaned_data = smoothClipping(cleaned_data)
        # cleaned_data = np.clip(cleaned_data, a_min=low_ica, a_max=high_ica)

    # Now, create a new MNE Raw object from the cleaned and smoothed data
    info = raw_ica.info  # Reuse the existing info structure (channel names, sampling frequency, etc.)
    smoothed_raw = mne.io.RawArray(cleaned_data.T, info)
    
    # Save the cleaned and smoothed data
    save_path = os.path.join(directory_ica_fif, f'{filename}.fif')
    smoothed_raw.save(save_path, overwrite=True)

    # savePlots(smoothed_raw, filename)

    # Convert the cleaned data into a DataFrame for each channel
    for idx in range(cleaned_data.shape[1]):
        channel_data = pd.DataFrame(cleaned_data[:, idx])
        # Append the DataFrame to the ICAData list
        ICAData.append(channel_data)

    del raw, raw_bandpass, raw_notch, raw_asr, raw_ica, cleaned_data, smoothed_raw
    gc.collect()

    return ICAData, snr_result

def calculateNFFT(sample_rate):
    scaling_factor = 8.192  # This ensures that for 250 Hz, n_fft will be around 2048
    n_fft = int(sample_rate * scaling_factor)  # Calculate based on sample rate
    return int(2 ** np.round(np.log2(n_fft)))  # Round to the nearest power of 2
        
def applyASR(raw, cutoff_deviation=5):
    global sample_rate
    global chunks
    
    # Define window size in samples
    window_size = int(chunks * sample_rate)
    
    # Get the raw data as a NumPy array
    data = raw.get_data()
    
    # Reference covariance matrix (we'll use the first few seconds of data as reference)
    reference_window = int(sample_rate * 1)  # 1-second window for reference
    reference_data = data[:, :reference_window]
    
    # Compute the reference covariance matrix
    reference_cov = np.cov(reference_data)
    
    # Define a cleaned data container
    cleaned_data = data.copy()
    
    # Loop over the data in windows
    for start_idx in range(0, data.shape[1] - window_size, window_size):
        # Get the current window of data
        window_data = data[:, start_idx:start_idx + window_size]
        
        # Compute covariance of the current window
        window_cov = np.cov(window_data)
        
        # Calculate RMS deviation from the reference
        rms_deviation = np.sqrt(np.mean((window_cov - reference_cov) ** 2))
        
        # If deviation exceeds the threshold, reconstruct the data
        if rms_deviation > cutoff_deviation:
            # Reconstruct the data by projecting it onto the reference subspace
            U, S, V = np.linalg.svd(reference_cov)
            reconstructed_data = np.dot(U, np.dot(U.T, window_data))
            
            # Replace the original window with reconstructed data
            cleaned_data[:, start_idx:start_idx + window_size] = reconstructed_data
            
    # Create a new RawArray object with the cleaned data
    raw_clean = mne.io.RawArray(cleaned_data, raw.info)
    
    return raw_clean

def savePlots(smoothed_raw, filename):
    global directory_ica_fif
    global sample_chunks
    global chunks

    output_dir = directory_ica_fif
    
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # 1. Single butterfly plot for the entire data range (all channels combined) with custom colors
    try:
        # Extract the entire data (all channels, all time points)
        all_data = smoothed_raw.get_data()  # Returns a numpy array of shape (n_channels, n_times)
        
        # Create an Evoked object for plotting the butterfly plot for the entire data
        evoked = mne.EvokedArray(all_data, smoothed_raw.info, tmin=0)

        # Plot the butterfly plot without automatic spatial coloring
        butterfly_fig = evoked.plot(spatial_colors=False, show=False)
        
        # Manually set the colors for each channel from the channeling_colors dictionary
        for ch_idx, ch_name in enumerate(smoothed_raw.info['ch_names']):
            color = channeling_colors.get(ch_name, 'black')  # Default to black if channel is not in the dictionary
            plt.gca().get_lines()[ch_idx].set_color(color)
        
        # Save the butterfly plot for the entire data range
        butterfly_fig.savefig(os.path.join(output_dir, f'{filename}_Butterfly.png'), dpi=300)
    except Exception as e:
        print(f"Single butterfly plot not saved: {e}")

    # 2. Butterfly plot for each channel individually with custom colors
    try:
        # Get the channel names
        channel_names = smoothed_raw.info['ch_names']

        # Loop through each channel and create a butterfly plot
        for ch_idx, ch_name in enumerate(channel_names):
            # Extract data for the current channel (ch_idx)
            channel_data = smoothed_raw.get_data(picks=[ch_idx])  # Get data for the specific channel
            
            # Make sure the data is shaped correctly as (1, n_samples)
            channel_data = channel_data.reshape(1, -1)

            # Create a new info object for this single channel
            info_channel = mne.create_info([ch_name], smoothed_raw.info['sfreq'], ch_types='eeg')

            # Create an Evoked object for plotting the butterfly plot for the current channel
            evoked_channel = mne.EvokedArray(channel_data, info_channel, tmin=0)

            # Plot the butterfly plot for the current channel with the assigned color
            butterfly_fig_channel = evoked_channel.plot(spatial_colors=False, show=False)
            
            # Manually set the color from the channeling_colors dictionary
            plt.gca().get_lines()[0].set_color(channeling_colors.get(ch_name, 'black'))  # Use 'black' as fallback color
            
            # Save the butterfly plot for the current channel
            butterfly_fig_channel.savefig(os.path.join(output_dir, f'{filename}_Butterfly_{ch_name}.png'), dpi=300)
    except Exception as e:
        print(f"Butterfly plot for each channel not saved: {e}")

def ICAData(butterData):
    global channeling
    global treshold_ica
    global high_ica
    global low_ica
    
    global print_debug
    global print_ica
    
    if print_debug:
        print()
        print("ICA Data")
        print()
        
    n_c = len(channeling) - 2
    if n_c < 1:
        n_c = 1
    ICAca = FastICA(n_components=n_c)
    
    ICAData = []
    
    for idx, butter in enumerate(butterData):
        InpData = pd.DataFrame(data=butter)
        X = InpData.values
        IndependentComponentValues = ICAca.fit_transform(X)
        ReducedData = pd.DataFrame(data = IndependentComponentValues)
        # print('ReducedData Before')
        # print(ReducedData)
        ReducedData = ReducedData * 1000
        if treshold_ica:
            for RD in ReducedData:
                if int(RD) > int(high_ica):
                    RD = high_ica
                elif int(RD) < int(low_ica):
                    RD = low_ica
        ReducedData = ReducedData.round(3)
        # print('ReducedData After')
        # print(ReducedData)
        ICAData.append(ReducedData)
        
        if print_ica:
            print(str(idx+1) + " = ")
            print(ReducedData)
            print()
                      
#     print("ICA Data:\n\n")
#     print(ICAData)
#     print("\nEND ID\n\n")

    # print("ICAData()")
    # print(ICAData)
    return ICAData
        
def plotAmp(ICAData, filename, print_singles):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Amplitude")
        print()

    plt.figure(figsize=(15,6))
    for idx, icad in enumerate(ICAData):
        plt.plot(icad, label = getChannel(idx))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Amplitude " + filename)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (uV)")
    plt.legend()
    plt.show()
    
    if print_singles:
        for idx, icad in enumerate(ICAData):
            plt.figure(figsize=(15,6))
            plt.style.use('seaborn-colorblind')
            plt.style.use('seaborn-whitegrid')
            plt.plot(icad, label = getChannel(idx))
            plt.title("Amplitude " + filename)
            plt.xlabel("Time (s)")
            plt.ylabel("Amplitude (uV)")
            plt.legend()
            plt.show()
        
def saveICAcsv(ICAData, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Save ICA csv")
        print()
    
    for idx, icad in enumerate(ICAData):
        name = pureFilename(filename, idx)
            
        df_ica = pd.DataFrame(icad)
        ica_file = directory_ica + "/" + name + ".csv"
        fileOverwrite(ica_file)
        df_ica.to_csv(ica_file, header = False, index = False)
        
# def butter_bandpass(lowcut, highcut, fs, order=5):
#     nyq = 0.5 * fs
#     low = lowcut / nyq
#     high = highcut / nyq
#     b, a = butter(order, [lowcut, highcut], btype='band')
#     return b, a


# def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
#     b, a = butter_bandpass(lowcut, highcut, fs, order=order)
#     y = lfilter(b, a, data[0])
#     return y
        
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], analog=False, btype='band', output='sos')
    return sos

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    sos = butter_bandpass(lowcut, highcut, fs, order=order)
    y = sosfilt(sos, data[0])
    return y

def bandpassFilter2(ICAmne, filename):
    global low_d, high_d
    global low_t, high_t
    global low_a, high_a
    global low_b, high_b
    global low_g, high_g
    global threshold_ica, low_ica, high_ica
    global sample_frequency
    global directory_ica
    global directory_ica_plot

    delta = []
    theta = []
    alpha = []
    beta = []
    gamma = []
    all_list = []
    ica_list = []
    d_list = []
    t_list = []
    a_list = []
    b_list = []
    g_list = []

    purefilename = pureFilename(filename)

    pd.set_option('display.float_format', '{:.5f}'.format)

    plot_fft = {}

    # Iterate through each channel/component from ICAmne
    for idx, icad in enumerate(ICAmne):
        cn = getChannel(idx)

        plot_fft[cn] = {}
        
        # Create MNE Raw object for each ICA component
        info = mne.create_info(ch_names=[getChannel(idx)], sfreq=sample_frequency, ch_types=['eeg'])
        raw = mne.io.RawArray(icad.to_numpy().T, info)

        # Apply bandpass filters for each frequency band using MNE
        delta_band = raw.copy().filter(l_freq=low_d, h_freq=high_d, fir_design='firwin')
        theta_band = raw.copy().filter(l_freq=low_t, h_freq=high_t, fir_design='firwin')
        alpha_band = raw.copy().filter(l_freq=low_a, h_freq=high_a, fir_design='firwin')
        beta_band = raw.copy().filter(l_freq=low_b, h_freq=high_b, fir_design='firwin')
        gamma_band = raw.copy().filter(l_freq=low_g, h_freq=high_g, fir_design='firwin')

        # Get the data and apply clipping before appending
        delta_data_clipped = np.clip(delta_band.get_data().T, low_ica, high_ica)
        theta_data_clipped = np.clip(theta_band.get_data().T, low_ica, high_ica)
        alpha_data_clipped = np.clip(alpha_band.get_data().T, low_ica, high_ica)
        beta_data_clipped = np.clip(beta_band.get_data().T, low_ica, high_ica)
        gamma_data_clipped = np.clip(gamma_band.get_data().T, low_ica, high_ica)

        plot_fft[cn]['Delta'] = delta_data_clipped
        plot_fft[cn]['Theta'] = theta_data_clipped
        plot_fft[cn]['Alpha'] = alpha_data_clipped
        plot_fft[cn]['Beta'] = beta_data_clipped
        plot_fft[cn]['Gamma'] = gamma_data_clipped

        # Append the filtered data to respective lists
        delta.append(pd.DataFrame(delta_data_clipped))
        theta.append(pd.DataFrame(theta_data_clipped))
        alpha.append(pd.DataFrame(alpha_data_clipped))
        beta.append(pd.DataFrame(beta_data_clipped))
        gamma.append(pd.DataFrame(gamma_data_clipped))

        # Create dataframes for saving and plotting
        df_ica = pd.DataFrame({f'{cn}_ICA': icad[0]})
        df_d = pd.DataFrame({f'{cn}_Delta': delta_band.get_data()[0]})
        df_t = pd.DataFrame({f'{cn}_Theta': theta_band.get_data()[0]})
        df_a = pd.DataFrame({f'{cn}_Alpha': alpha_band.get_data()[0]})
        df_b = pd.DataFrame({f'{cn}_Beta': beta_band.get_data()[0]})
        df_g = pd.DataFrame({f'{cn}_Gamma': gamma_band.get_data()[0]})

        # Append data for Excel output
        all_list.append(df_ica)
        all_list.append(df_d)
        all_list.append(df_t)
        all_list.append(df_a)
        all_list.append(df_b)
        all_list.append(df_g)

        # Append data for individual lists
        ica_list.append(df_ica)
        d_list.append(df_d)
        t_list.append(df_t)
        a_list.append(df_a)
        b_list.append(df_b)
        g_list.append(df_g)

        # Plot each band data
        plotSave(df_ica, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_1_1_{cn}_ICA", "Sample", "Amplitude")
        plotSave(df_d, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_2_1_{cn}_Delta", "Sample", "Amplitude")
        plotSave(df_t, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_3_1_{cn}_Theta", "Sample", "Amplitude")
        plotSave(df_a, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_4_1_{cn}_Alpha", "Sample", "Amplitude")
        plotSave(df_b, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_5_1_{cn}_Beta", "Sample", "Amplitude")
        plotSave(df_g, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_6_1_{cn}_Gamma", "Sample", "Amplitude")

    plotFFTsubband(plot_fft, f'{directory_ica_plot}/{purefilename}/ica_subband_freq', f'{purefilename}')
    
    # Create and save concatenated plots for each filtered data
    plotSave(pd.concat(ica_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_1_0_ICA", "Sample", "Amplitude")
    plotSave(pd.concat(d_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_2_0_Delta", "Sample", "Amplitude")
    plotSave(pd.concat(t_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_3_0_Theta", "Sample", "Amplitude")
    plotSave(pd.concat(a_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_4_0_Alpha", "Sample", "Amplitude")
    plotSave(pd.concat(b_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_5_0_Beta", "Sample", "Amplitude")
    plotSave(pd.concat(g_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_6_0_Gamma", "Sample", "Amplitude")

    # Save the bandpassed data to an Excel file
    all_filename = directory_ica + "/" + purefilename + ".xlsx"
    fileOverwrite(all_filename)
    xls_list = pd.concat(all_list, axis=1)
    xls_list.index += 1
    xls_list.to_excel(all_filename, index_label="Index")

    return {"Delta": delta, "Theta": theta, "Alpha": alpha, "Beta": beta, "Gamma": gamma}

def mneConnectivity(data_dict, raw_data, sample_frequency, info, connectivity_dir):
    channels = list(data_dict.keys())
    metrics = ['coh', 'pli', 'plv']
    results = {metric: {} for metric in metrics}

    # Prepare broadband data as a list of arrays, each with shape (n_channels, n_times)
    broadband_data = [np.array([raw_data[ch] for ch in channels])]  # List of single (n_channels, n_times) array

    # Calculate broadband connectivity for each metric
    for metric in metrics:
        conn_result = spectral_connectivity_epochs(
            broadband_data, method=metric, sfreq=sample_frequency, faverage=True, verbose=False
        )
        results[metric]['Broadband'] = conn_result.get_data(output='dense')[:, :, 0]  # Extract 2D matrix (n_channels, n_channels)

    # Calculate connectivity for each subband in data_dict
    for band in data_dict[channels[0]]:
        subband_data = [np.array([data_dict[ch][band] for ch in channels])]  # List of single (n_channels, n_times) array
        
        for metric in metrics:
            conn_result = spectral_connectivity_epochs(
                subband_data, method=metric, sfreq=sample_frequency, faverage=True, verbose=False
            )
            results[metric][band] = conn_result.get_data(output='dense')[:, :, 0]  # Extract 2D matrix

    # Save connectivity data to Excel
    with pd.ExcelWriter(os.path.join(connectivity_dir, 'connectivity_matrices.xlsx')) as writer:
        for metric in metrics:
            for band, matrix in results[metric].items():
                df = pd.DataFrame(matrix, index=channels, columns=channels)
                df.to_excel(writer, sheet_name=f'{metric}_{band}')

    # Generate and save visualizations for each metric and band
    for metric in metrics:
        for band, connectivity_matrix in results[metric].items():
            # Topomap plot with plot_sensors_connectivity
            fig = plot_sensors_connectivity(info, connectivity_matrix, threshold_prop=0.5)
            fig.suptitle(f'{metric.upper()} - {band}')
            fig.savefig(os.path.join(connectivity_dir, f'{metric}_{band}_sensors_topomap.png'))
            plt.close(fig)

            # Circular connectivity plot with plot_connectivity_circle
            fig, ax = plt.subplots(figsize=(10, 8))
            plot_connectivity_circle(
                connectivity_matrix, node_names=channels, title=f'{metric.upper()} - {band}',
                vmin=0.0, vmax=1.0, colorbar=True
            )
            fig.savefig(os.path.join(connectivity_dir, f'{metric}_{band}_connectivity_circle.png'))
            plt.close(fig)

def compute_pli(data1, data2):
    phase_diff = np.angle(np.exp(1j * (np.angle(data1) - np.angle(data2))))
    pli = np.abs(np.mean(np.sign(phase_diff)))
    return pli

def compute_plv(data1, data2):
    phase_diff = np.angle(np.exp(1j * (np.angle(data1) - np.angle(data2))))
    plv = np.abs(np.mean(np.exp(1j * phase_diff)))
    return plv

def connectivity_matrices(data_dict, raw_data, sample_frequency, channels, connectivity_dir, info):
    metrics = ['coh', 'pli', 'plv']
    results = {metric: {} for metric in metrics}

    # Calculate broadband connectivity
    for metric in metrics:
        matrix = np.zeros((len(channels), len(channels)))
        
        for i, ch1 in enumerate(channels):
            for j, ch2 in enumerate(channels):
                if i != j:
                    if metric == 'coh':
                        f, coh_values = coherence(raw_data[ch1].flatten(), raw_data[ch2].flatten(), fs=sample_frequency)
                        matrix[i, j] = np.mean(coh_values)
                    elif metric == 'pli':
                        matrix[i, j] = compute_pli(raw_data[ch1].flatten(), raw_data[ch2].flatten())
                    elif metric == 'plv':
                        matrix[i, j] = compute_plv(raw_data[ch1].flatten(), raw_data[ch2].flatten())
        
        results[metric]['Broadband'] = matrix

    # Calculate subband connectivity
    for band in data_dict[channels[0]]:
        for metric in metrics:
            matrix = np.zeros((len(channels), len(channels)))
            
            for i, ch1 in enumerate(channels):
                for j, ch2 in enumerate(channels):
                    if i != j:
                        if metric == 'coh':
                            f, coh_values = coherence(data_dict[ch1][band].flatten(), data_dict[ch2][band].flatten(), fs=sample_frequency)
                            matrix[i, j] = np.mean(coh_values)
                        elif metric == 'pli':
                            matrix[i, j] = compute_pli(data_dict[ch1][band].flatten(), data_dict[ch2][band].flatten())
                        elif metric == 'plv':
                            matrix[i, j] = compute_plv(data_dict[ch1][band].flatten(), data_dict[ch2][band].flatten())
            
            results[metric][band] = matrix

    # Save matrices to Excel
    with pd.ExcelWriter(os.path.join(connectivity_dir, 'connectivity_matrices.xlsx')) as writer:
        for metric in metrics:
            for band, matrix in results[metric].items():
                df = pd.DataFrame(matrix, index=channels, columns=channels)
                df.to_excel(writer, sheet_name=f'{metric}_{band}')

    # Visualize and save matrices
    for metric in metrics:
        for band, matrix in results[metric].items():
            plt.figure(figsize=(8, 6))
            plt.imshow(matrix, cmap='Blues', interpolation='nearest')
            plt.colorbar(label=metric.upper())
            plt.xticks(ticks=np.arange(len(channels)), labels=channels, rotation=90)
            plt.yticks(ticks=np.arange(len(channels)), labels=channels)
            plt.gca().invert_yaxis()
            plt.title(f'{metric.upper()} Matrix - {band}')
            plt.savefig(os.path.join(connectivity_dir, f'{metric}_{band}_matrix.png'))
            plt.close()

def bandpassFilter(ICAmne, filename):
    global low_d, high_d, low_t, high_t, low_a, high_a, low_b, high_b, low_g, high_g
    global threshold_ica, low_ica, high_ica, sample_rate, directory_ica, directory_ica_plot
    sample_frequency = sample_rate

    delta, theta, alpha, beta, gamma = [], [], [], [], []
    all_list, ica_list, d_list, t_list, a_list, b_list, g_list = [], [], [], [], [], [], []
    purefilename = pureFilename(filename)
    plot_fft = {}

    # Directory for saving connectivity files
    connectivity_dir = f'{directory_ica_plot}/{purefilename}/connectivity'
    os.makedirs(connectivity_dir, exist_ok=True)  # Ensure the directory exists
    
    # Dictionary to store clipped subband data for each channel
    data_dict = {}
    raw_data = {}

    # Create a single `info` object for connectivity visualization across all channels
    channel_names = [getChannel(idx) for idx in range(len(ICAmne))]
    unified_info = mne.create_info(ch_names=channel_names, sfreq=sample_frequency, ch_types=['eeg'] * len(channel_names))

    # Iterate through each channel/component from ICAmne
    for idx, icad in enumerate(ICAmne):
        cn = getChannel(idx)
        plot_fft[cn] = {}

        # Create a separate `info` object for each ICA component with only one channel
        single_channel_info = mne.create_info(ch_names=[cn], sfreq=sample_frequency, ch_types=['eeg'])
        raw = mne.io.RawArray(icad.to_numpy().T, single_channel_info)

        # Store the raw ICA component data (before filtering) in `raw_data`
        raw_data[cn] = icad.to_numpy().T

        # Apply bandpass filters for each frequency band using MNE
        delta_band = raw.copy().filter(l_freq=low_d, h_freq=high_d, fir_design='firwin')
        theta_band = raw.copy().filter(l_freq=low_t, h_freq=high_t, fir_design='firwin')
        alpha_band = raw.copy().filter(l_freq=low_a, h_freq=high_a, fir_design='firwin')
        beta_band = raw.copy().filter(l_freq=low_b, h_freq=high_b, fir_design='firwin')
        gamma_band = raw.copy().filter(l_freq=low_g, h_freq=high_g, fir_design='firwin')

        # Clip data and store it in data_dict for each subband
        data_dict[cn] = {
            'Delta': np.clip(delta_band.get_data().T, low_ica, high_ica),
            'Theta': np.clip(theta_band.get_data().T, low_ica, high_ica),
            'Alpha': np.clip(alpha_band.get_data().T, low_ica, high_ica),
            'Beta': np.clip(beta_band.get_data().T, low_ica, high_ica),
            'Gamma': np.clip(gamma_band.get_data().T, low_ica, high_ica)
        }

        plot_fft[cn]['Delta'] = data_dict[cn]['Delta']
        plot_fft[cn]['Theta'] = data_dict[cn]['Theta']
        plot_fft[cn]['Alpha'] = data_dict[cn]['Alpha']
        plot_fft[cn]['Beta'] = data_dict[cn]['Beta']
        plot_fft[cn]['Gamma'] = data_dict[cn]['Gamma']
        
        # Append the filtered data to respective lists
        delta.append(pd.DataFrame(data_dict[cn]['Delta']))
        theta.append(pd.DataFrame(data_dict[cn]['Theta']))
        alpha.append(pd.DataFrame(data_dict[cn]['Alpha']))
        beta.append(pd.DataFrame(data_dict[cn]['Beta']))
        gamma.append(pd.DataFrame(data_dict[cn]['Gamma']))

        # Create dataframes for saving and plotting
        df_ica = pd.DataFrame({f'{cn}_ICA': icad[0]})
        df_d = pd.DataFrame({f'{cn}_Delta': delta_band.get_data()[0]})
        df_t = pd.DataFrame({f'{cn}_Theta': theta_band.get_data()[0]})
        df_a = pd.DataFrame({f'{cn}_Alpha': alpha_band.get_data()[0]})
        df_b = pd.DataFrame({f'{cn}_Beta': beta_band.get_data()[0]})
        df_g = pd.DataFrame({f'{cn}_Gamma': gamma_band.get_data()[0]})

        # Append data for Excel output
        all_list.append(df_ica)
        all_list.append(df_d)
        all_list.append(df_t)
        all_list.append(df_a)
        all_list.append(df_b)
        all_list.append(df_g)

        # Append data for individual lists
        ica_list.append(df_ica)
        d_list.append(df_d)
        t_list.append(df_t)
        a_list.append(df_a)
        b_list.append(df_b)
        g_list.append(df_g)

        # Plot each band data
        plotSave(df_ica, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_1_1_{cn}_ICA", "Sample", "Amplitude")
        plotSave(df_d, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_2_1_{cn}_Delta", "Sample", "Amplitude")
        plotSave(df_t, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_3_1_{cn}_Theta", "Sample", "Amplitude")
        plotSave(df_a, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_4_1_{cn}_Alpha", "Sample", "Amplitude")
        plotSave(df_b, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_5_1_{cn}_Beta", "Sample", "Amplitude")
        plotSave(df_g, 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_6_1_{cn}_Gamma", "Sample", "Amplitude")

    # Generate COH and PLI matrices using MNE-Connectivity
    # mneConnectivity(data_dict, raw_data, sample_frequency, unified_info, connectivity_dir)
    connectivity_matrices(data_dict, raw_data, sample_frequency, channel_names, connectivity_dir, unified_info)

    # Final plot of FFT for each subband
    plotFFTsubband(plot_fft, f'{directory_ica_plot}/{purefilename}/ica_subband_freq', f'{purefilename}')

    # Save concatenated plots for each filtered data
    plotSave(pd.concat(ica_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_1_0_ICA", "Sample", "Amplitude")
    plotSave(pd.concat(d_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_2_0_Delta", "Sample", "Amplitude")
    plotSave(pd.concat(t_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_3_0_Theta", "Sample", "Amplitude")
    plotSave(pd.concat(a_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_4_0_Alpha", "Sample", "Amplitude")
    plotSave(pd.concat(b_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_5_0_Beta", "Sample", "Amplitude")
    plotSave(pd.concat(g_list, axis=1), 100, 7, f'{directory_ica_plot}/{purefilename}/ica_subband', f"{purefilename}_6_0_Gamma", "Sample", "Amplitude")

    # Save the bandpassed data to an Excel file
    all_filename = directory_ica + "/" + purefilename + ".xlsx"
    fileOverwrite(all_filename)
    xls_list = pd.concat(all_list, axis=1)
    xls_list.index += 1
    xls_list.to_excel(all_filename, index_label="Index")

    # Return the data for each band
    return {"Delta": delta, "Theta": theta, "Alpha": alpha, "Beta": beta, "Gamma": gamma}

def plotSave(df, xsize, ysize, plot_dir, plotname, xlabel, ylabel):
    """Creates and saves a plot for a given DataFrame."""
    global channeling_colors
    
    # Get colors for the columns based on the channel names
    colors = [channeling_colors.get(col.split('_')[0], 'black') for col in df.columns]
    
    plt.close()
    
    plt.figure(figsize=(xsize, ysize))
    df.plot(color=colors, legend=True)
    plt.title(plotname)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    # plt.legend(title='Channels')
    plt.tight_layout()

    # Create plot directory if it does not exist
    os.makedirs(plot_dir, exist_ok=True)
    
    # Define the output path for the plot
    output_path = os.path.join(plot_dir, f"{plotname}.png")
    plt.savefig(output_path)
    plt.close()
            
def mavData(ICAData, filename, print_mav, timestamps):
    global chunks
    global directory_mav
    global directory_psd
    global sub_freqs
    global do_psd
    global brainwaves
    global features_full
    
    global print_debug
    
    global debug
    
    if print_debug:
        print()
        print("MAV")
        print()
        
    bandpassICAData = bandpassFilter(ICAData, filename)
    # plotTimeFreq(bandpassICAData, filename)
    
    chunk_count = True
    chunk_temp = []

    mav_delta_list = []
    mav_theta_list = []
    mav_alpha_list = []
    mav_beta_list = []
    mav_gamma_list = []

    mav_delta = {}
    mav_theta = {}
    mav_alpha = {}
    mav_beta = {}
    mav_gamma = {}
    
    mav_list = []
    
    for bw, data in bandpassICAData.items():
        for idx, icad in enumerate(data):
            name = pureFilename(filename, idx)
            # data_chunks = chunky(icad, chunks, timestamps)
            data_chunks = chunky2(icad, chunks)
            channel = getChannel(idx)
            
            total_data = np.concatenate(data_chunks)

            if debug:
                print("\n------------------------------\n" + channel + "\n------------------------------\n")        
                print("Data Chunks = " + str(len(data_chunks)))
            counter_chunk = 1
            
            mean_temp = []
            mav_temp = []
            var_temp = []
            sd_temp = []
            en_temp = []
            mob_temp = []
            com_temp = []
            skewness_temp = []
            kurt_temp = []
            petrosian_temp = []
            higuchi_temp = []
            sample_en_temp = []
            spec_en_temp = []
            peak_to_peak_temp = []
            psd_temp = []
            fourier_temp = []
            wavelet_temp = []
            zcross_temp = []

            for chunk in data_chunks:
                if debug:
                    print()
                    print(str(counter_chunk) + ".")
                mean, mav, var, sd, en, mob, com, skewness, kurt, petrosian, higuchi, samp_en, spec_en, peak_to_peak, psd, fourier, wavelet, zcross = calculateFeature(chunk, bw)

                if chunk_count:
                    chunk_temp.append(len(chunk))
                mean_temp.append(mean)
                mav_temp.append(mav)
                var_temp.append(var)
                sd_temp.append(sd)
                en_temp.append(en)
                mob_temp.append(mob)
                com_temp.append(com)
                skewness_temp.append(skewness)
                kurt_temp.append(kurt)
                petrosian_temp.append(petrosian)
                higuchi_temp.append(higuchi)
                sample_en_temp.append(samp_en)
                spec_en_temp.append(spec_en)
                peak_to_peak_temp.append(peak_to_peak)
                psd_temp.append(psd)
                fourier_temp.append(fourier)
                wavelet_temp.append(wavelet)
                zcross_temp.append(zcross)

                counter_chunk += 1
            
            if chunk_count:
                mav_list.append(pd.DataFrame(chunk_temp, columns=["Data Length"]))
                chunk_count = False
            cn = channel + "_" + bw + "_"
            temp_list = {}
            temp_list[cn + "PSD"] = psd_temp
            temp_list[cn + "Mean"] = mean_temp
            temp_list[cn + "Mean Absolute Value"] = mav_temp
            temp_list[cn + "Standard Deviation"] = sd_temp
            temp_list[cn + "Hjorth Activity"] = var_temp
            temp_list[cn + "Hjorth Mobility"] = mob_temp
            temp_list[cn + "Hjorth Complexity"] = com_temp
            temp_list[cn + "Skewness"] = skewness_temp
            temp_list[cn + "Kurtosis"] = kurt_temp
            temp_list[cn + "Petrosian"] = petrosian_temp
            temp_list[cn + "Higuchi"] = higuchi_temp
            temp_list[cn + "Classic Entropy"] = en_temp
            temp_list[cn + "Sample Entropy"] = sample_en_temp
            temp_list[cn + "Spectral Entropy"] = spec_en_temp
            temp_list[cn + "Fourier Transform"] = fourier_temp
            temp_list[cn + "Wavelet Transform"] = wavelet_temp
            temp_list[cn + "Peak to Peak"] = peak_to_peak_temp
            temp_list[cn + "Zero Crossing Rate"] = zcross_temp
            # temp_list[cn + "Power"] = welchs[channel][bw]

            mav_list.append(pd.DataFrame(temp_list))
            
            if bw == "Delta":
                mav_delta = {"Filename":name+"_Delta", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_delta_list.append(mav_delta)
            elif bw == "Theta":
                mav_theta = {"Filename":name+"_Theta", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_theta_list.append(mav_theta)
            elif bw == "Alpha":
                mav_alpha = {"Filename":name+"_Alpha", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_alpha_list.append(mav_alpha)
            elif bw == "Beta":
                mav_beta = {"Filename":name+"_Beta", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_beta_list.append(mav_beta)
            elif bw == "Gamma":
                mav_gamma = {"Filename":name+"_Gamma", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_gamma_list.append(mav_gamma)
    
    index_label = "Chunk (" + str(chunks) + ")"
    mav_filename = directory_mav + "/" + pureFilename(filename) + ".xlsx"
    fileOverwrite(mav_filename)
    all_list = pd.concat(mav_list, axis=1)
    all_list.index += 1
    all_list.to_excel(mav_filename, index_label=index_label)
    
    saveFeaturePlots(mav_list, pureFilename(filename))
        
    return mav_delta_list, mav_theta_list, mav_alpha_list, mav_beta_list, mav_gamma_list

def plotTimeFreq(bandpassICAData, filename):
    global sample_rate
    
    # Iterate over each subband and channel to compute time-frequency representations (TFRs)
    for bw, data in bandpassICAData.items():
        for idx, icad in enumerate(data):
            sfreq = sample_rate  # Use the sample rate from your global variable
            name = pureFilename(filename, idx)
            channel = getChannel(idx)

            # Reshape the data to be compatible with mne.EpochsArray
            # icad needs to be shaped as (n_epochs, n_channels, n_times)
            # Since this is continuous data, we assume 1 epoch and 1 channel
            icad = icad[None, None, :]  # Reshaping to (1 epoch, 1 channel, n_times)

            # Create the info object for this single channel
            info = mne.create_info([channel], sfreq, ch_types=['eeg'])

            # Create an Epochs object using the reshaped data
            epochs = mne.EpochsArray(icad, info)

            # Compute the time-frequency representation using Morlet wavelets
            freqs = np.logspace(*np.log10([1, 50]), num=40)  # Frequency range for Morlet
            n_cycles = freqs / 2.0  # Number of cycles per frequency

            # Compute TFR
            tfr = mne.time_frequency.tfr_morlet(epochs, freqs=freqs, n_cycles=n_cycles, return_itc=False)

            # Plot the time-frequency representation
            fig, ax = plt.subplots()
            tfr.plot([0], axes=ax, show=False, colorbar=True, title=f'{channel} {bw} Time-Frequency')

            # Define the directory where the plot will be saved
            plot_dir = os.path.join(directory_mav_plot_time_frequency, bw)
            if not os.path.exists(plot_dir):
                os.makedirs(plot_dir)

            # Save the figure
            plot_filename = os.path.join(plot_dir, f'{name}_{bw}.png')
            fig.savefig(plot_filename)
            plt.close(fig)  # Close the figure to free memory

def saveFeaturePlots(mav_list, purefilename):
    global features_full
    global brainwaves
    
    for feature_name in features_full:
        for subband_name in brainwaves:
            subbands = []
            for df in mav_list:
                # Filter columns based on the subband and feature name
                subband_columns = [col for col in df.columns if subband_name in col and col.endswith(f"_{feature_name}")]
                if not subband_columns:
                    continue
        
                # Create a DataFrame with only the relevant columns
                filtered_df = df[subband_columns]
                
                # Define the plot directory and name
                filename = filtered_df.columns[0].split('_')
                plot_dir = f"imotion_mav/plot/{feature_name}"
                plotname = f"{purefilename}_{filename[0]}_{filename[1]}"
                
                # Call plotSave to generate and save the plot
                plotSave(filtered_df, 100, 7, plot_dir, plotname, xlabel="Chunks", ylabel=feature_name)
                subbands.append(filtered_df)
            if subbands:
                plotSave(pd.concat(subbands, axis=1), 100, 7, plot_dir, f"{purefilename}_{subband_name}", xlabel="Chunks", ylabel=feature_name)
            else:
                print(f'Plot Fail: {feature_name}, {subband_name}')

def plotMAV(counter, data, title, name, print_mav):
    global channels
    
    plt.figure(counter, figsize=(15,6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    for idx, row in enumerate(data):
        plt.plot(row, label = getChannel(idx))
    plt.ylabel("SD " + title)
    plt.legend()
    for channel in channels:
        if channel != 'Nan':
            find = "_" + channel
            name = name.replace(find, "")
    graph_title = directory_graph + "/Graph_SD_" + name + "_" + title + ".png"
    fileOverwrite(graph_title)
    plt.savefig(graph_title)
    if print_mav:
        plt.show()
        for idx, row in enumerate(data):
            average = sum(row)/len(row)
            print(getChannel(idx) + " Average = " + "{:.2f}".format(average) + " uV^2")
    else:
        plt.close(counter)
        
def welchData(data):
    global sample_frequency
    global sample_rate
    global win
    global chunks
    
    global debug
    
    fs = sample_rate
    nperseg = fs * chunks
    
#     freqs, psd = signal.periodogram(data[0], fs=100, scaling='density')
    freqs, psd = signal.welch(data[0], fs=fs, nperseg=nperseg)
#     freqs, psd = signal.welch(data[0], fs=100, nperseg=(len(data)*4))
    
    freq_res = freqs[1] - freqs[0]
                      
#     print("Len Data = " + str(len(data)))
#     print("freqs (" + str(len(freqs)) + "):")
#     print(freqs)
#     print("freq_res:")
#     print(freq_res)
#     print("psd (" + str(len(psd)) + "):")
#     print(psd)
#     print()
    return freqs, psd, freq_res
            
def psdData(ICAData, filename, print_psd):
    global chunks
    global sub_freqs
    global brainwaves
    global low_d
    global low_t
    global low_a
    global low_b
    global low_g
    global high_d
    global high_t
    global high_a
    global high_b
    global high_g
    
    global print_debug
    
    global debug
    
    if print_debug:
        print()
        print("PSD")
        print()
    
    chunk_count = True
    chunk_temp = []
        
    delta = []
    theta = []
    alpha = []
    beta = []
    gamma = []
    
    psd_list = []
    all_list = []
    
    for idx, icad in enumerate(ICAData):
        name = pureFilename(filename, idx)
        data_chunks = chunky(icad, chunks)
        channel = getChannel(idx)
        
        if debug:
            print("\n------------------------------\n" + channel + "\n------------------------------\n")        
            print("Data Chunks = " + str(len(data_chunks)))

        psd_delta = []
        psd_theta = []
        psd_alpha = []
        psd_beta = []
        psd_gamma = []
        temp_list = {}

        counter_chunk = 0
        for chunk in data_chunks:
            counter_chunk += 1
            if debug:
                print()
                print(str(counter_chunk) + ".")
                
            if chunk_count:
                chunk_temp.append(len(chunk))
                
            freqs, psd, freq_res = welchData(chunk)

            idx_band_d = np.logical_and(freqs >= low_d, freqs < high_d)
            band_power_d = None
            if len(psd[idx_band_d]):
                if len(psd[idx_band_d]) > 1:
                    band_power_d = simpson(psd[idx_band_d], dx=freq_res)
                else:
                    band_power_d = psd[idx_band_d][0]
            psd_delta.append(band_power_d)
            
            idx_band_t = np.logical_and(freqs >= low_t, freqs < high_t)
            band_power_t = None
            if len(psd[idx_band_t]):
                if len(psd[idx_band_t]) > 1:
                    band_power_t = simpson(psd[idx_band_t], dx=freq_res)
                else:
                    band_power_t = psd[idx_band_t][0]
            psd_theta.append(band_power_t)

            idx_band_a = np.logical_and(freqs >= low_a, freqs < high_a)
            band_power_a = None
            if len(psd[idx_band_a]):
                if len(psd[idx_band_a]) > 1:
                    band_power_a = simpson(psd[idx_band_a], dx=freq_res)
                else:
                    band_power_a = psd[idx_band_a][0]
            psd_alpha.append(band_power_a)

            idx_band_b = np.logical_and(freqs >= low_b, freqs < high_b)
            band_power_b = None
            if len(psd[idx_band_b]):
                if len(psd[idx_band_b]) > 1:
                    band_power_b = simpson(psd[idx_band_b], dx=freq_res)
                else:
                    band_power_b = psd[idx_band_b][0]
            psd_beta.append(band_power_b)

            idx_band_g = np.logical_and(freqs >= low_g, freqs < high_g)
            band_power_g = None
            if len(psd[idx_band_g]):
                if len(psd[idx_band_g]) > 1:
                    band_power_g = simpson(psd[idx_band_g], dx=freq_res)
                else:
                    band_power_g = psd[idx_band_g][0]
            psd_gamma.append(band_power_g)
            
#             temp_list[str(counter_chunk)] = [band_power_d, band_power_t, band_power_a, band_power_b, band_power_g]
        
        if chunk_count:
            all_list.append(pd.DataFrame(chunk_temp, columns=["Data Length"]))
            chunk_count = False
        
        delta.append(psd_delta)
        theta.append(psd_theta)
        alpha.append(psd_alpha)
        beta.append(psd_beta)
        gamma.append(psd_gamma)
        
#         columns = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
#         psd_filename = directory_psd + "/" + name + ".xlsx"
#         fileOverwrite(psd_filename)
#         psd_list = pd.DataFrame.from_dict(temp_list, orient='index', columns=columns)
#         psd_list.to_excel(psd_filename, index_label='Chunks')
        
        cn = channel + "_Delta"
        all_list.append(pd.DataFrame({cn:psd_delta}))
        cn = channel + "_Theta"
        all_list.append(pd.DataFrame({cn:psd_theta}))
        cn = channel + "_Alpha"
        all_list.append(pd.DataFrame({cn:psd_alpha}))
        cn = channel + "_Beta"
        all_list.append(pd.DataFrame({cn:psd_beta}))
        cn = channel + "_Gamma"
        all_list.append(pd.DataFrame({cn:psd_gamma}))
        
    index_label = "Chunk (" + str(chunks) + ")"
    all_filename = directory_psd + "/" + pureFilename(filename) + ".xlsx"
    fileOverwrite(all_filename)
    xls_list = pd.concat(all_list, axis=1)
    xls_list.index += 1
    xls_list.to_excel(all_filename, index_label=index_label)
                
#     if final_graph:
#         plotPSD(1, delta, "Delta", name, print_psd)
#         plotPSD(2, theta, "Theta", name, print_psd)
#         plotPSD(3, alpha, "Alpha", name, print_psd)
#         plotPSD(4, beta, "Beta", name, print_psd)
#         plotPSD(5, gamma, "Gamma", name, print_psd)
        
    return delta, theta, alpha, beta, gamma

def plotPSD(counter, data, title, name, print_psd):
    global channels
    
    plt.figure(counter, figsize=(15,6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    for idx, row in enumerate(data):
        plt.plot(row, label = getChannel(idx))
    plt.ylabel("PSD " + title)
    plt.legend()
    for channel in channels:
        if channel != 'Nan':
            find = "_" + channel
            name = name.replace(find, "")
    graph_title = directory_graph + "/Graph_PSD_" + name + "_" + title + ".png"
    fileOverwrite(graph_title)
    plt.savefig(graph_title)
    if print_psd:
        plt.show()
        for idx, row in enumerate(data):
            average = sum(row)/len(row)
            print(getChannel(idx) + " Average = " + "{:.2f}".format(average) + " uV^2")
    else:
        plt.close(counter)
        
def analyzeStatistic(analyze_list = [], train = 100):
    global chunks
    global labels
    global channeling
    global subbanding
    global iter_seconds
    global os_mav
    global directory_compare
    global directory_mav
    global directory_analysis
    global directory_encoded
    global directory_unique
    
    test = False

    for bw in subbanding:
        for channel in channeling:
            sub = channel + "_" + bw
            difference_list = {}
            for label in labels:
                filesave = label
                specific = label
                dif = ""
                dif_list = []
                encoded_list = {}
                compare_list = {}
                train_list = {}
                test_list = {}
                encoded_test_list = {}
                df_encoded_test_list = {}

                counter = 1
                filenames = []
                print(bw + " Files (" + label + ") " + channel + ":\n-------------------------")
                if not analyze_list:
                    for file in os.listdir(os_mav):
                        temp = []
    #                     mean_temp = []
    #                     mav_temp = []
    #                     sd_temp = []

                        filename = os.fsdecode(file)
                        sub = channel + "_" + bw
                        if filename.find(sub) != -1:
                            if filename.endswith(ext_raw):
                                if specific == "":
                                    print(str(counter) + ". " + filename)

                                    filepath = directory_mav + "/" + filename
                                    with open(filepath, newline='') as f:
                                        reader = csv.reader(f)
                                        data = list(reader)
                                        for x in data:
                                            temp.append(float(x[3]))
    #                                         mean_temp.append(float(x[1]))
    #                                         mav_temp.append(float(x[2]))
    #                                         sd_temp.append(float(x[3]))
                                        if len(temp) > 0:
                                            train_list[filename] = temp
                                            filenames.append(filename)
                                    counter += 1
                                else:
                                    if filename.find(specific) != -1:
                                        filepath = directory_mav + "/" + filename
                                        with open(filepath, newline='') as f:
                                            reader = csv.reader(f)
                                            data = list(reader)
                                            for x in data:
                                                temp.append(float(x[3]))
    #                                             mean_temp.append(float(x[1]))
    #                                             mav_temp.append(float(x[2]))
    #                                             sd_temp.append(float(x[3]))
                                            if len(temp) > 0:
                                                train_list[filename] = temp
                                                filenames.append(filename)
                    if train > 0 and train < 100:
                        test = True

                        train_keys = random.sample(list(train_list), int(train / 100 * len(train_list)))
                        filenames = []
                        temp_list = {}
                        for fn, al in train_list.items():
                            if fn in train_keys:
                                print(str(counter) + ". " + fn)
                                temp_list[fn] = al
                                filenames.append(fn)
                                counter += 1
                            else:
                                test_list[fn] = al
                        train_list = temp_list
                else:
                    if analyze_list[label]:
                        if train > 0 and train < 100:
                            test = True
                            train_temp = {}
                            test_temp = {}

                            train_keys = random.sample(list(analyze_list[label]), int(train / 100 * len(analyze_list[label])))
                            for fn, al in analyze_list[label].items():
                                if fn in train_keys:
                                    train_temp[fn] = al
                                else:
                                    test_temp[fn] = al
                            for fn, al in train_temp.items():
                                temp = []

                                for x in al[bw]:
                                    temp = x["SD"]

                                if len(temp) > 0:
                                    train_list[fn] = temp
                                    filenames.append(fn)

                                print(str(counter) + ". " + fn)

                                counter += 1
                            for fn, al in test_temp.items():
                                temp = []

                                for x in al[bw]:
                                    temp = x["SD"]

                                if len(temp) > 0:
                                    test_list[fn] = temp
                        else:
                            for fn, al in analyze_list[label].items():
                                temp = []
                                mean_temp = []
                                mav_temp = []
                                sd_temp = []

                                filename = fn
                                sub = channel + "_" + bw

                                for x in al[bw]:
                                    temp = x["SD"]

                                if len(temp) > 0:
                                    train_list[filename] = temp
                                    filenames.append(filename)

                                print(str(counter) + ". " + filename)

                                counter += 1
                    else:
                        print("\nNot found: " + label + "\n")

                print("-------------------------")

                if test:
                    for fn, tl in test_list.items():
                        encoded_test_list[fn] = encodeGrowthData(tl)
                    for key, data in encoded_test_list.items():
                        joined = "".join(map(str, data))
                        df_encoded_test_list[key] = joined

                if len(train_list) > 0:
                    for fn in filenames:
                        encoded_list[fn] = encodeGrowthData(train_list[fn])
                    for key1, data1 in encoded_list.items():
                        temp_list = {}

                        for key2, data2 in encoded_list.items():
                            if key1 != key2:
                                temp_list2 = []
                                result_list = compareGrowthEncoded(data1, data2)
                                for res in result_list:
                                    match = res[0]
                                    start = res[1]
                                    end = match + start
                                    matching = data1[start:end]
                                    match_string = "".join(str(v) for v in matching)
                                    temp_list2.append(match_string)
                                    dif_list.append(match_string)
                                temp_list[key2] = temp_list2
    #                             match_max, start1, start2 = compareGrowthEncoded(data1, data2)
    #                             temp_list.append({key2 : [match_max, start1, start2, data1, data2]})

                        compare_list[key1] = temp_list

                    fn = "compare-std-" + channel + "-" + bw + ".xlsx"
                    if filesave != "":
                        fn = "compare-std-" + channel + "-" + bw + "-" + filesave + ".xlsx"
                    final_list = {}
                    for key1, data1 in encoded_list.items():
                        for key2, data2 in compare_list[key1].items():
                            final_list[key1 + ' | ' + key2] = data2
                    dfFinal = pd.DataFrame.from_dict(final_list, orient='index')
                    dfFinal.to_excel(directory_compare + "/" + fn, index_label='Filenames')

                    df_encoded_list = {}
                    for key, data in encoded_list.items():
                        joined = "".join(map(str, data))
                        df_encoded_list[key] = joined
                    efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + ".xlsx"
                    if filesave != "":
                        efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + "_" + filesave + ".xlsx"
                    dfEncoded = pd.DataFrame.from_dict(df_encoded_list, orient='index', columns=['Encode'])
                    dfEncoded.to_excel(efn, index_label='Filename')

                    if df_encoded_test_list:
                        efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + "_test.xlsx"
                        if filesave != "":
                            efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + "_" + filesave + "_test.xlsx"
                        dfEncoded = pd.DataFrame.from_dict(df_encoded_test_list, orient='index', columns=['Encode'])
                        dfEncoded.to_excel(efn, index_label='Filename')

                    cfn = "compare_distance_std_" + channel + "_" + bw + "_"
                    if filesave != "":
                        cfn = cfn + filesave + "_"
    #                 lev_match = levMatch(dif_list, True, False, 2)
                    best_list = []
                    if test:
                        lev_min, min_lev, final_list, best_list = minLevMatch(dif_list, dif_list, True, 2, cfn + "full.xlsx", df_encoded_list, df_encoded_test_list)
                    else:
                        lev_min, min_lev, final_list, best_list = minLevMatch(dif_list, dif_list, True, 2, cfn + "full.xlsx", df_encoded_list)
    #                 lev_min, min_lev, final_list = minLevMatch(lev_match, dif_list, True, 5, cfn + "partial.xlsx")
    #                 lev_min, min_lev, final_list = minLevMatch(lev_match, lev_match, True, 5, cfn + "minimal.xlsx")
                    difference_list[label] = best_list
            
            print(difference_list)
            
            uniqueness = dict()
            for a, b in difference_list.items():
                temp = []
                banned = []
                for x, y in difference_list.items():
                    if a != x:
                        for element in b:
                            if element not in y:
                                if element not in temp:
                                    if element not in banned:
                                        temp.append(element)
                                else:
                                    temp.remove(element)
                                    if element not in banned:
                                        banned.append(element)
                uniqueness[a] = [', '.join(temp)]
                
            filename = directory_unique + "/unique_match_std_" + sub + ".xlsx"
            unique_list = pd.DataFrame.from_dict(uniqueness, orient='index', columns=['Encode List'])
            unique_list.to_excel(filename, index_label='Label')
                
    print()
    
def analyzePSD():
    global chunks
    global iter_seconds
    global os_processed
    global directory_psd
    global directory_compare
    global directory_analysis
    global directory_encoded
    global brainwaves

    dif = ""
    dif_list = []
    for bw in brainwaves:
        for channel in channels:
            if channel != 'Nan':
                data_list = dict()
                encoded_list = dict()
                compare_list = dict()

                counter = 1
                filenames = []
                print(bw + " Files " + channel + ":\n-------------------------")
                for file in os.listdir(os_processed):
                    temp = []
                    filename = os.fsdecode(file)
                    sub = channel + "_" + bw
                    if filename.find(sub) != -1:
                        if filename.endswith(".csv"):

                            print(str(counter) + ". " + filename)

                            filepath = directory_psd + "/" + filename
                            df = pd.read_csv(filepath)
                            with open(filepath, newline='') as f:
                                reader = csv.reader(f)
                                data = list(reader)
                                for x in data:
                                    for y in x:
                                        temp.append(float(y))
                                if len(temp) > 0:
                                    data_list[filename] = temp
                                    filenames.append(filename)
                            counter += 1
                print("-------------------------")

                if len(data_list) > 0:
                    for fn in filenames:
                        encoded_list[fn] = encodeGrowthData(data_list[fn])

                    for key1, data1 in encoded_list.items():
                        temp_list = []

                        for key2, data2 in encoded_list.items():
                            if key1 != key2:
                                temp_list2 = []
                                result_list = compareGrowthEncoded(data1, data2)
                                for res in result_list:
                                    match = res[0]
                                    start = res[1]
                                    end = match + start
                                    matching = data1[start:end]
                                    match_string = "".join(str(v) for v in matching)
                                    temp_list2.append(match_string)
                                    if bw == "Alpha" and channel == "T3":
                                        dif_list.append(match_string)
                                temp_list.append({key2 : temp_list2})
    #                             match_max, start1, start2 = compareGrowthEncoded(data1, data2)
    #                             temp_list.append({key2 : [match_max, start1, start2, data1, data2]})

                        compare_list[key1] = temp_list

                    fn = "compare-psd-" + channel + "-" + bw + ".xlsx"
                    final_list = []
                    for key1, data1 in encoded_list.items():
    #                     fn = "compare-psd-" + key1.replace(".csv","") + ".xlsx"
    #                     final_list = []

                        iter_check = int(math.floor(iter_seconds / chunks))
                        for urut in range(iter_check):
                            sem = {}
                            for key2, data2 in encoded_list.items():
                                if key1 != key2:
                                    for x in compare_list[key1]:
                                        for a, b in x.items():
                                            if a == key2:
                                                comparing = key1 + " | " + key2
                                                sem[comparing] = b[urut]
                            final_list.append(sem)
    #                     dfFinal = pd.DataFrame(final_list)
    #                     dfFinal = dfFinal.transpose()
    #                     dfFinal.to_excel(directory_compare + "/" + fn)
                    dfFinal = pd.DataFrame(final_list)
                    dfFinal = dfFinal.transpose()
                    dfFinal.to_excel(directory_compare + "/" + fn)

                    df_encoded_list = {}
                    for key, data in encoded_list.items():
                        joined = "".join(map(str, data))
                        df_encoded_list[key] = joined
                    efn = directory_encoded + "/encoded_psd_" + channel + "_" + bw + ".xlsx"
                    dfEncoded = pd.DataFrame.from_dict(df_encoded_list, orient='index')
                    dfEncoded.to_excel(efn)
                    
    lev_match = levMatch(dif_list, True, False, 2)
    lev_min, min_lev, final_list = minLevMatch(dif_list, dif_list, True, 2, "compare_distance_psd_full.xlsx")
    lev_min, min_lev, final_list = minLevMatch(lev_match, dif_list, True, 5, "compare_distance_psd_partial.xlsx")
    lev_min, min_lev, final_list = minLevMatch(lev_match, lev_match, True, 5, "compare_distance_psd_minimal.xlsx")
#     print("C1 Min = " + str(c1_min[0]) + " [" + str(c1_min[1]) + "]")
#     print("C2 Min = " + str(c2_min[0]) + " [" + str(c2_min[1]) + "]")
    print()
            
#             print()
#             for x, y in compare_list.items():
#                 print(x + ":")
#                 for z in y:
#                     for a, b in z.items():
#                         print()
#                         print(a)
#                         for c in b:
#                             print(c)
#                         match_max = b[0]
#                         start1 = b[1]
#                         start2 = b[2]
#                         end = start1 + match_max
#                         print()
#                         print(a + " = " + str(match_max) + " match")
#                         print("Index = [" + str(start1) + ", " + str(start2) + "]")
#                         print(b[3][start1:end])
                        
#                 print()

def levMatch(dif_list, print_iter, print_min, iter_mod = 2):
    global start_time
    start_time = time.time()
    
    lev_max = -1
    c1_max = []
    c2_max = []
    lev_min = -1
    c1_min = []
    c2_min = []
    same_match = []
    min_match = []
    
    dif_list_len = len(dif_list)
    iter_max = dif_list_len * (dif_list_len - 1)
    iter_counter = 1
    iter_temp = 0
    
    checked = []
    
    print()
    print("Iterations = " + str(iter_max))
    print()
    for c1, d1 in enumerate(dif_list):
        checked2 = []
        if d1 not in checked:
            checked.append(d1)
            for c2, d2 in enumerate(dif_list):
                if d2 not in checked2:
                    checked2.append(d2)
                    if print_iter:
                        iter_temp = printIter(iter_temp, iter_counter, iter_max, iter_mod)

                    if d1 != d2:
                            lev = 100
                            if len(d1) == len(d2):
                                lev = hamming(d1, d2)
                            else:
                                lev = levenshtein(d1, d2)
                            if lev > lev_max:
                                lev_max = lev
                                c1_max = [c1, d1]
                                c2_max = [c2, d2]
                            if lev < lev_min or lev_min == -1:
                                lev_min = lev
                                c1_min = [c1, d1]
                                c2_min = [c2, d2]
                            if lev == 1:
                                if d1 not in min_match:
                                    min_match.append(d1)
                                if d2 not in min_match:
                                    min_match.append(d2)
                    elif c1 != c2:
                        if d1 not in same_match:
                            same_match.append(d1)

                    iter_counter += 1
        
    same_max = 0
    max_same = []
    for sm in same_match:
        total = dif_list.count(sm)
        if total > same_max:
            same_max = total
            
    for sm in same_match:
        total = dif_list.count(sm)
        if total == same_max:
            max_same.append(sm)
    
    print("\nSame = " + str(len(same_match)) + "/" + str(iter_max) + " (Unique = " + str(same_max) + ")")
    print(max_same)
    print("\nLevenshtein Distance:")
    print("\nLev Max = " + str(lev_max))
    print("C1 Max = " + str(c1_max[0]) + " [" + str(c1_max[1]) + "]")
    print("C2 Max = " + str(c2_max[0]) + " [" + str(c2_max[1]) + "]")
    print("\nLev Min = " + str(lev_min))
    print("Min Match = " + str(len(min_match)))
    if print_min:
        for mm in min_match:
            print(mm) 
        
    return min_match

def minLevMatch(dif_list, ori_list, print_iter, iter_mod = 5, filename = "compare_distance.xlsx", encoded_list = [], test_list = {}):
    global start_time
    global directory_analysis
    global labels
    
    start_time = time.time()
    
    min_match = []
    
    iter_max = len(dif_list) * (len(ori_list))
    iter_counter = 1
    iter_temp = 0
    
    min_lev = ""
    lev_min = -1
    
    checked = []
    checked_list = {}
    
    analyze_list = dict()
    
    print()
    print("Iterations = " + str(iter_max))
    print()
    for c1, d1 in enumerate(dif_list):
        if len(d1) > 0:
    #         lev_list = []
            checked2 = []
            lev_total = 0
            lev_counter = 0

            if d1 not in checked:
                checked.append(d1)
                for c2, d2 in enumerate(ori_list):
                    if len(d2) > 0:
                        if d2 not in checked2:
                            checked2.append(d2)
                            if print_iter:
                                iter_temp = printIter(iter_temp, iter_counter, iter_max, iter_mod)

                            lev = 100
                            if d1 != d2:
                                if len(d1) == len(d2):
                                    lev = hamming(d1, d2)
                                else:
                                    lev = levenshtein(d1, d2)
        #                         lev_list.append(lev)
                            else:
                                lev = 0
                            checked_list[d2] = lev

                        lev_total += checked_list[d2]
                        lev_counter += 1
                        iter_counter += 1

    #             lev_mean = sum(lev_list) / len(lev_list)
                lev_mean = lev_total / lev_counter
                analyze_list[d1] = lev_mean
                if lev_min == -1 or lev_mean < lev_min:
                    lev_min = lev_mean
                    min_lev = d1
    
    final_list = dict()
    best_list = []
    columns = ['Distance']
    if encoded_list:
        columns = ['Length','Distance 1','Distance 2','Existences','% Occurrences','AVG Occurrences']
        if test_list:
            columns = ['Length','Distance 1','Distance 2','Existences 1','% Occurrences 1','AVG Occurrences 1','Existences 2','% Occurrences 2','AVG Occurrences 2']
        for a, b in analyze_list.items():
            exist = 0
            occurrence = 0
            pieces = 0
            lev = 0
            total = len(encoded_list)
            for x, y in encoded_list.items():
                length_match = len(a)
                length_encode = len(y)
                length_max = length_encode / length_match
                occur = y.count(a)
                pieces += occur / length_max
                occurrence += occur
                if occur > 0:
                    exist += 1
                if len(a) == len(y):
                    lev += hamming(a, y)
                else:
                    lev += levenshtein(a, y)
            existences = ("%.2f%%" % (100 * exist / total))
            average = ("%.2f%%" % (100 * pieces / total))
            occurrences = math.floor((occurrence / total) * 100) / 100
            distance = math.floor((lev / total) * 100) / 100
            if test_list:
                existences_test = '-'
                average_test = '-'
                occurrences_test = 0
                if existences == "100.00%":
                    exist_test = 0
                    occurrence_test = 0
                    pieces_test = 0
                    total_test = len(test_list)
                    for x, y in test_list.items():
                        length_match = len(a)
                        length_encode = len(y)
                        length_max = length_encode / length_match
                        occur_test = y.count(a)
                        pieces_test += occur_test / length_max
                        occurrence_test += occur_test
                        if occur_test > 0:
                            exist_test += 1
                    existences_test = ("%.2f%%" % (100 * exist_test / total_test))
                    average_test = ("%.2f%%" % (100 * pieces_test / total_test))
                    occurrences_test = math.floor((occurrence_test / total_test) * 100) / 100
                    if existences_test == "100.00%":
                        best_list.append(a)
                final_list[a] = [len(a), b, distance, existences, average, occurrences, existences_test, average_test, occurrences_test]
            else:
                final_list[a] = [len(a), b, distance, existences, average, occurrences]
    else:
        final_list = analyze_list
        
    print("\nLev Mean Min = " + str(lev_min) + " (" + min_lev + ")")
    
    filename = directory_analysis + "/" + filename
    distance_list = pd.DataFrame.from_dict(final_list, orient='index', columns=columns)
    distance_list.to_excel(filename, index_label='Encode')
            
    return lev_min, min_lev, final_list, best_list

def printIter(iter_temp, iter_counter, iter_max, per = 5):
    global start_time
    global end_time
    
    mod = 2
    if iter_max > 1000000:
        mod = 1000
    elif iter_max > 10000:
        mod = 100
    elif iter_max > 1000:
        mod = 10
    iter_mod = iter_counter % mod
    
    if iter_mod == 0:
        iter_per = iter_counter / iter_max * 100
        if int(iter_per) > iter_temp:
            if int(iter_per) % per == 0:
                end_time = time.time()
                print(str(int(iter_per)) + "% (" + printTimer(int(end_time - start_time)) + ")")
                iter_temp = int(iter_per)
                
    return iter_temp
                        
def compareGrowthEncoded(data_list1, data_list2):
    global chunks
    global iter_seconds
    
    iter_max = len(data_list1)
    if len(data_list2) < iter_max:
        iter_max = len(data_list2)
    iter_check = int(math.floor(iter_seconds / chunks))
    
    data_list = []
    result_list = []
    result = 0
    start1 = -1
    start2 = -1
    for i in range(iter_check):
        winner = 0
        match = 0
        temp1 = -1
        temp2 = -1
        before = False
        for n in range(iter_max):
            cur = n+i
            
            if cur < iter_max:
                if data_list1[cur] == data_list2[n]:
                    if match == 0 or before:
                        match += 1
                        before = True
                        if match > winner:
                            winner = match
                            temp1 = cur
                            temp2 = n
                    else:
                        before = False
                        if match > winner:
                            winner = match
                            temp1 = cur
                            temp2 = n
                else:
                    before = False
                    if match > winner:
                        winner = match
                        temp1 = cur
                        temp2 = n
                    match = 0
                        
            if winner > result:
                result = winner
                start1 = temp1
                start2 = temp2
                    
        result_list.append([winner, start1+1-winner])
            
    return result_list
#     return result, start1+1-result, start2+1-result
                        
def encodeGrowthData(data_list):
    encoded_list = []
    first = 0
    second = 0
    check = False
    for data in data_list:
        second = data
        if check:
            if first < second:
                encoded_list.append(1)
            else:
                encoded_list.append(0)
        else:
            check = True
        first = data
        
    return encoded_list

def hamming(s1,s2):
    result = -1
    if len(s1) != len(s2):
        return result
    else:
        for x, (i, j) in enumerate(zip(s1, s2)):
            if i != j:
                result += 1
    return result

def levenshtein(seq1, seq2, print_matrix = False):
    size_x = len(seq1) + 1
    size_y = len(seq2) + 1
    matrix = np.zeros ((size_x, size_y))
    for x in range(size_x):
        matrix [x, 0] = x
    for y in range(size_y):
        matrix [0, y] = y

    for x in range(1, size_x):
        for y in range(1, size_y):
            if seq1[x-1] == seq2[y-1]:
                matrix [x,y] = min(
                    matrix[x-1, y] + 1,
                    matrix[x-1, y-1],
                    matrix[x, y-1] + 1
                )
            else:
                matrix [x,y] = min(
                    matrix[x-1,y] + 1,
                    matrix[x-1,y-1] + 1,
                    matrix[x,y-1] + 1
                )
    if print_matrix:
        print (matrix)
    return (matrix[size_x - 1, size_y - 1])

def calculateFeature(subbandData, bs):
    global sample_rate
    global chunks

    data = subbandData[0]
    # data = subbandData
    nperseg = int(sample_rate * chunks)

    # Determine the appropriate wavelet decomposition level based on the subband
    wavelet_level = 4
    if bs == "Delta":
        wavelet_level = 6  # Higher level for low-frequency Delta band
    elif bs == "Theta":
        wavelet_level = 5  # Slightly lower for Theta
    elif bs == "Alpha":
        wavelet_level = 4  # Level 4 for mid-range Alpha
    elif bs == "Beta":
        wavelet_level = 3  # Lower level for higher-frequency Beta
    elif bs == "Gamma":
        wavelet_level = 2

    # Basic statistical features
    mean = np.mean(data)
    var = np.var(data)
    sd = np.std(data)
    mav = np.mean(np.abs(data))
    
    # Histogram-based entropy (Shannon entropy in the time domain)
    counts, _ = np.histogram(data, bins=10, density=True)
    counts = counts[counts > 0]  # Remove zero counts for entropy calculation
    en = entropy(counts)
    
    # Hjorth parameters (mobility and complexity)
    mob, com = hjorth_params(data)

    # Skewness and Kurtosis
    skewness = skew(data)
    kurt = kurtosis(data)

    # Fractal dimensions (Petrosian and Higuchi)
    petrosian = petrosian_fd(data)
    # if len(data) < 3 or np.all(data == data[0]) or not np.isfinite(data).all():
    #     higuchi = 0
    # else:
    higuchi = higuchi_fd(data)

    # Sample entropy
    samp_en = sample_entropy(data)
    if np.isinf(samp_en):
        samp_en = 0  # Handle infinite values
    
    # Spectral Entropy (frequency domain complexity)
    spec_en = spectral_entropy(data, sf=sample_rate, method='welch')

    # Peak-to-peak amplitude
    peak_to_peak = np.ptp(data)
    
    # Power Spectral Density (PSD) using MNE's psd_welch
    psd, freqs = psd_array_welch(np.array(data), sfreq=sample_rate, n_per_seg=nperseg)
    power = simpson(psd, freqs)
    
    # Fourier Transform (using SciPy's FFT)
    fourier_transform = np.abs(fft(np.array(data)))
    fourier = np.mean(fourier_transform)
    
    # Wavelet Transform (using PyWavelets with 'db4' wavelet)
    coeffs = pywt.wavedec(data, 'db4', level=wavelet_level)
    wavelet = np.sum([np.sum(np.square(c)) for c in coeffs])

    # Zero Crossing Rate (sign change in data)
    zcross = np.sum(np.diff(np.sign(data)) != 0)
    
    return (
        float(mean), float(mav), float(var), float(sd),
        float(en), float(mob), float(com), float(skewness),
        float(kurt), float(petrosian), float(higuchi), float(samp_en),
        float(spec_en), float(peak_to_peak), float(power), float(fourier),
        float(wavelet), int(zcross)
    )

def comparePSD(wave):
    global os_analysis
    
    bw = ""
    if wave == 1:
        bw = "Delta"
    elif wave == 2:
        bw = "Theta"
    elif wave == 3:
        bw = "Alpha"
    elif wave == 4:
        bw = "Beta"
    elif wave == 5:
        bw = "Gamma"
    else:
        print("\nInput doesn't work, please see input list!")
    
    if bw != "":
        for channel in channels:
            if channel != 'Nan':
                data_list = []
                counter = 1
                filenames = []
                print("Files " + channel + ":\n-------------------------")
                for file in os.listdir(os_analysis):
                    temp = []
                    filename = os.fsdecode(file)
                    sub = channel + "_" + bw 
                    if filename.find(sub) != -1:
                        if filename.endswith(".csv"):

                            print(str(counter) + ". " + filename)

                            filepath = directory_psd + "/" + filename
                            df = pd.read_csv(filepath)
                            with open(filepath, newline='') as f:
                                reader = csv.reader(f)
                                data = list(reader)
                                for x in data:
                                    for y in x:
                                        temp.append(float(y))
                                if len(temp) > 0:
                                    data_list.append(temp)
                                    filenames.append(filename)
                            counter += 1
                print("-------------------------")

                if len(data_list) > 0:
                    dfPSD = pd.DataFrame(data_list)
                    dfPSD = dfPSD.transpose()
                    dfPSD.columns = filenames
                    dfMean = dfPSD[filenames].mean()
                    print("\nAverage:")
                    print(dfMean)
                    dfPSD.plot(kind='line',figsize=(15,6))
                    plt.style.use('seaborn-colorblind')
                    plt.style.use('seaborn-whitegrid')
                    plt.title("PSD Analysis " + sub)
                    plt.xlabel("Frame")
                    plt.ylabel("Value")
                    plt.legend()
                    graph_title = directory_graph + "/Analysis_PSD_"+ sub + ".png"
                    fileOverwrite(graph_title)
                    plt.savefig(graph_title)
                    plt.show()

def fileOverwrite(filepath):
    if os.path.exists(filepath):
        os.remove(filepath)

def printCurrentSettings():
    global timer
    global channels
    global second_start
    global seconds
    global chunks
    
    print("\nCurrent channels:")
    print(*channels, sep = ", ")
    print("\nLive Graph EEG:")
    print("Current measurement time = " + str(measurement_time))
    print("\nLive EEG:")
    print("Current timer (in seconds) = " + str(timer))
    print("\nFolder EEG:")
    print("Current starting second = " + str(second_start))
    print("Current seconds = " + str(seconds))
    print("\nAll EEG:")
    print("Current chunk seconds = " + str(chunks))
    
def printTimer(seconds, before = ""):
    minutes = 60
    hours = minutes * 60
    days = hours * 24
    
    result = ""
    left = 0
    if seconds >= days:
        left = seconds % days
        day = int(math.floor(seconds / days))
        if before != "":
            before = before + " "
        if day > 0:
            before = before + str(day) + " day"
        if day > 1:
            before = before + "s"
        result = printTimer(left, before)
    elif seconds >= hours:
        left = seconds % hours
        hour = int(math.floor(seconds / hours))
        if before != "":
            before = before + " "
        if hour > 0:
            before = before + str(hour) + " hour"
        if hour > 1:
            before = before + "s"
        result = printTimer(left, before)
    elif seconds >= minutes:
        left = seconds % minutes
        minute = int(math.floor(seconds / minutes))
        if before != "":
            before = before + " "
        if minute > 0:
            before = before + str(minute) + " minute"
        if minute > 1:
            before = before + "s"
        result = printTimer(left, before)
    else:
        if before != "" and seconds > 0:
            before = before + " "
        if seconds > 0:
            before = before + str(seconds) + " second"
        if seconds > 1:
            before = before + "s"
        result = before
    
    return result

def copyFile(source_folder, destination_folder, filename):
    source_path = os.path.join(source_folder, filename)
    destination_path = os.path.join(destination_folder, filename)
    fileOverwrite(destination_path)
    
    shutil.copy(source_path, destination_path)

def formatLongNumber(value):
    return (f"{value:.2e}" if abs(value) >= 1e4 or (0 < abs(value) < 1e-2) else f"{value:.3f}")

In [5]:
start_time_2 = time.time()

def main():
    global timer
    global channels
    global second_start
    global seconds
    global cut
    global dict_zip
    global frame_start
    global frame_end
    global chunks
    global sample_rate
    global sample_frequency
    global measurement_time
    global do_analyze_encode
    global interval_sec
    global final_graph
    global live_filename
    global start_time
    global end_time
    global train_encode
    global labels
    global label_prefix
    global label_prefix_start
    
    global debug
    
    global os_raw
    global os_analysis
    
    timer = int(timer)
    filerename = "temp"
    
    print("Processing Start!")
    
    printCurrentSettings()
    
    choice = 0
    
    ica_list = []
    delta_list = []
    theta_list = []
    alpha_list = []
    beta_list = []
    gamma_list = []
    sample_rate_list = {'filename': [], 'sample_rate': []}
    
    total_time = 0
    if not os.path.exists(directory_ica):
        os.makedirs(directory_ica)
    if not os.path.exists(directory_psd):
        os.makedirs(directory_psd)
    if not os.path.exists(directory_graph):
        os.makedirs(directory_graph)
    if not os.path.exists(directory_analysis):
        os.makedirs(directory_analysis)
    if not os.path.exists(directory_mav):
        os.makedirs(directory_mav)
    if not os.path.exists(directory_compare):
        os.makedirs(directory_compare)
    if not os.path.exists(directory_encoded):
        os.makedirs(directory_encoded)
    if not os.path.exists(directory_unique):
        os.makedirs(directory_unique)
    if not os.path.exists(directory_corrupt):
        os.makedirs(directory_corrupt)
    if not os.path.exists(directory_empty):
        os.makedirs(directory_empty)
    
    print("\nRunning folder EEG\n")
    
    label_list = {}
    for label, data in labels.items():
        label_list[label] = {}
    
    raw_channels = []
    if ext_raw == '.xlsx':
        for x in range(len(channels)):
            raw_channels.append('CH.' + str(x))
    else:
        raw_channels = channels
    
    counter = 0
    
    temp = ""
    empty_list = []
    corrupt_list = []
    error_list = []
    incomplete_list = []
    snr_list = []
    for label, data in labels.items():
        finder = label_prefix + label
        if label_prefix_start == 1:
            finder = label + label_prefix
            
        second_start = data['start']
        seconds = data['seconds']

        print("-------------------------")
        print(finder)
        print("-------------------------")

        end_time = time.time()
        count_time = int(end_time - start_time)
        total_time = int(end_time - start_time_2)
        print("\nTime = (" + printTimer(count_time) + ")")
        print("Total Time = (" + printTimer(total_time) + ")\n")
            
        for file in os.listdir(os_raw):
            filename = os.fsdecode(file)

            if filename.endswith(ext_raw):
                if filename.find(finder) != -1:

                    print("-------------------------")
                    print(filename)
                    print("-------------------------")
                    
                    start_time = time.time()
                    filepath = directory_raw + "/" + filename
                    dfRaw = readRawData(filename)
                    
                    if ext_raw == '.xlsx':
                        sample_rate = getSampleRate(dfRaw['Timestamp'])
                    else:
                        sample_rate = 256
                    print("\nSample Rate = " + str(sample_rate))
                    if sample_rate is None or sample_rate == 0:
                        empty_list.append(filename)
                        continue
                    sample_rate_list['filename'].append(pureFilename(filename))
                    sample_rate_list['sample_rate'].append(sample_rate)

                    timestamp = dfRaw['Timestamp'].copy()
                    cleaned_data_without_timestamp = dfRaw.drop('Timestamp', axis=1)
                    plotSave(cleaned_data_without_timestamp, 100, 7, directory_raw_plot, filename.replace(".xlsx", ""), "Sample", "Amplitude")
                    
                    temp = label
    
                    frame_start = second_start * sample_rate
                    frame_end = ((second_start + seconds) * sample_rate)
                    frame_total = (seconds - 1) * sample_rate
                    
                    # dfClean = dfRaw[raw_channels].loc[frame_start:frame_end]
                    dfClean = dfRaw[channels].loc[frame_start:frame_end]
                    if len(dfClean) < frame_total:
                        incomplete_list.append([filename, str(int(len(dfClean)/sample_rate)), str(seconds)])
                        continue
                    cleanTimestamp = timestamp.loc[frame_start:frame_end].reset_index(drop=True)
                    del dfRaw, 
                    gc.collect()

                    print("Finished Read Raw Data " + filename)
                    end_time = time.time()
                    count_time = int(end_time - start_time)
                    total_time = int(end_time - start_time_2)
                    print("\nTime = (" + printTimer(count_time) + ")")
                    print("Total Time = (" + printTimer(total_time) + ")\n")
                    
                    # icas, snr_result = ICAmne(dfClean, pureFilename(filename))
                    # snr_list.append(snr_result)
                    # del dfClean
                    # gc.collect()

                    # for idx, icad in enumerate(icas):
                    #     ica_list.append(icad)

                    # do_psd = False
                    # if do_psd:
                    #     delta, theta, alpha, beta, gamma = psdData(icas, filename, False)
                    # mav_delta, mav_theta, mav_alpha, mav_beta, mav_gamma = mavData(icas, filename, False, cleanTimestamp)
                    # del icas
                    # gc.collect()

                    try:
                        # outputSignals = filterRawData(dfClean)
                        # del dfClean
                        # gc.collect()
                        # butters = butterData(outputSignals)
                        # del outputSignals
                        # gc.collect()
                        # # icas = ICAData(butters)
                        # del butters
                        
                        icas, snr_result = ICAmne(dfClean, pureFilename(filename))
                        snr_list.append(snr_result)
                        del dfClean
                        gc.collect()

                        for idx, icad in enumerate(icas):
                            ica_list.append(icad)

                        do_psd = False
                        if do_psd:
                            delta, theta, alpha, beta, gamma = psdData(icas, filename, False)
                        mav_delta, mav_theta, mav_alpha, mav_beta, mav_gamma = mavData(icas, filename, False, cleanTimestamp)
                        del icas
                        gc.collect()

                        testing_list = {
                            "Delta" : mav_delta,
                            "Theta" : mav_theta,
                            "Alpha" : mav_alpha,
                            "Beta" : mav_beta,
                            "Gamma" : mav_gamma
                        }

                        label_list[label][filename] = testing_list
                    except ValueError:
                        corrupt_list.append(filename)
                        error_list.append(f'{filename}: {ValueError}')
                        continue

                    print("Finished Preprocessed " + filename)
                    end_time = time.time()
                    count_time = int(end_time - start_time)
                    total_time = int(end_time - start_time_2)
                    print("\nTime = (" + printTimer(count_time) + ")")
                    print("Total Time = (" + printTimer(total_time) + ")\n")

                    counter += 1

                    # Convert the results to a DataFrame and save it to an Excel file
                    snr_df = pd.DataFrame(snr_list)
                    
                    # Save the results to the 'imotion_analysis' folder
                    output_path = os.path.join(directory_ica, '01_ICA_Processing.xlsx')
                    snr_df.to_excel(output_path, index=False)

    # Convert the results to a DataFrame and save it to an Excel file
    snr_df = pd.DataFrame(snr_list)
    
    # Save the results to the 'imotion_analysis' folder
    output_path = os.path.join(directory_ica, '00_ICA_Processing.xlsx')
    snr_df.to_excel(output_path, index=False)

    # Convert the dictionary to a pandas DataFrame
    sample_rate_df = pd.DataFrame(sample_rate_list)
    
    # Save the DataFrame to an Excel file
    output_path = "sample_rates.xlsx"
    sample_rate_df.to_excel(output_path, index=False)

    end_time_2 = time.time()
    print("FINISH = (" + printTimer(int(end_time_2 - start_time_2)) + ")")
    
    source_path = directory_raw
    destination_path = directory_empty
    for empty in empty_list:
        copyFile(source_path, destination_path, empty)
        
    source_path = directory_raw
    destination_path = directory_corrupt
    for corrupt in corrupt_list:
        copyFile(source_path, destination_path, corrupt)
        
    source_path = directory_raw
    destination_path = directory_incomplete
    for incomplete in incomplete_list:
        copyFile(source_path, destination_path, incomplete[0])
    
    empty_string = '\n'.join(empty_list)
    corrupt_string = '\n'.join(corrupt_list)
    error_string = '\n'.join(error_list)
    incomplete_string = '\n'.join([f"{item[0]} ({item[1]}/{item[2]})" for item in incomplete_list])
    with open('EEG Failed Dataset.txt', 'w') as file:
        file.write("Empty:\n")
        file.write(empty_string)
        file.write("\n\nCorrupt:\n")
        file.write(error_string)
        file.write("\n\nIncomplete:\n")
        file.write(incomplete_string)
    choice = 0
        
    print("\nProcessing End!")

if __name__ == "__main__":
    main()

Processing Start!

Current channels:
F8, F7, T8, T7, P8, P7, O2, O1

Live Graph EEG:
Current measurement time = 5

Live EEG:
Current timer (in seconds) = 5

Folder EEG:
Current starting second = 0
Current seconds = 5

All EEG:
Current chunk seconds = 0.2

Running folder EEG

-------------------------
bengbeng_
-------------------------

Time = ()
Total Time = ()

-------------------------
pucuk_
-------------------------

Time = ()
Total Time = ()

-------------------------
shopee_
-------------------------

Time = ()
Total Time = ()

-------------------------
shopee_dataresponden23.xlsx
-------------------------

Sample Rate = 250
Finished Read Raw Data shopee_dataresponden23.xlsx

Time = (3 seconds)
Total Time = (3 seconds)

Finished Preprocessed shopee_dataresponden23.xlsx

Time = (2 minutes 46 seconds)
Total Time = (2 minutes 46 seconds)

-------------------------
shopee_dataresponden24.xlsx
-------------------------

Sample Rate = 250
Finished Read Raw Data shopee_dataresponden24.

In [6]:
# start_time_2 = time.time()

# # Load sample rates from Excel file
# sample_rates_df = pd.read_excel("sample_rates.xlsx")
# sample_rates_dict = dict(zip(sample_rates_df['filename'], sample_rates_df['sample_rate']))

# output_folder = directory_analysis
# output_file_path = os.path.join(output_folder, 'connectivity_results.xlsx')
# output_cm = directory_analysis_confusion_matrix

# # Channels and subbands of interest
# channels_interest = channeling
# subbandings = ['ICA', 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']

# # Initialize result dictionary with separate entries for COH, PLV, and PLI
# result_data = {
#     'label': [], 'emotion': [], 'participant': []
# }
# for measure in ['COH', 'PLV', 'PLI']:
#     for subband in subbandings:
#         result_data.update({f'{ch1} vs {ch2} {measure} {subband}': [] for ch1 in channels_interest for ch2 in channels_interest if ch1 != ch2})
#         result_data.update({f'{ch} vs All {measure} {subband}': [] for ch in channels_interest})
#         result_data.update({f'All {measure} {subband}': []})

# # Helper function for COH, PLV, and PLI
# def calculate_coh_plv_pli(data1, data2, sfreq):
#     # Calculate coherence
#     freqs, coh_values = coherence(data1, data2, fs=sfreq)
#     mean_coh = round(np.mean(coh_values) * 100, 3)
    
#     # Calculate PLV
#     phase_diff = np.angle(data1) - np.angle(data2)
#     plv_values = np.abs(np.mean(np.exp(1j * phase_diff)))
#     mean_plv = np.mean(plv_values)
    
#     # Calculate PLI
#     pli_values = np.sign(phase_diff)
#     mean_pli = np.abs(np.mean(pli_values))
    
#     return mean_coh, mean_plv, mean_pli

# # Storage for aggregation
# label_matrix_data = {label: {'COH': {}, 'PLV': {}, 'PLI': {}} for label in labels_data.keys()}
# emotion_matrix_data = {emotion: {'COH': {}, 'PLV': {}, 'PLI': {}} for emotion in set(info['emotion'] for info in labels_data.values())}

# # Initialize matrices for confusion matrix data
# for matrix_data in [label_matrix_data, emotion_matrix_data]:
#     for key, measure_data in matrix_data.items():
#         for measure in ['COH', 'PLV', 'PLI']:
#             measure_data[measure] = {subband: np.zeros((len(channels_interest), len(channels_interest))) for subband in subbandings}

# # Initialize counts for each label and emotion
# label_counts = {label: 0 for label in labels_data.keys()}
# emotion_counts = {emotion: 0 for emotion in set(info['emotion'] for info in labels_data.values())}

# # Filter based on label name and participant name in labels_data
# for label_name, label_info in labels_data.items():
#     # Process each file
#     for file in os.listdir(input_folder_ica):
#         if label_name in file:
#             if file.endswith('.xlsx'):
#                 participant_name = file.replace(f"{label_name}_", "").replace(".xlsx", "")

#                 if any(entry['participant'] == participant_name and entry['label'] == label_name for entry in corrupts_data):
#                     continue
                    
#                 sfreq = 256

#                 # Build the key for sample rate lookup
#                 filename_key = f"{label_name}_{participant_name}"

#                 # Retrieve sample rate from the dictionary
#                 if filename_key in sample_rates_dict:
#                     sfreq = sample_rates_dict[filename_key]
#                 else:
#                     continue

#                 start_time = time.time()
                    
#                 emotion = label_info['emotion']
                
#                 result_data['label'].append(label_name)
#                 result_data['emotion'].append(emotion)
#                 result_data['participant'].append(participant_name)

#                 file_path = os.path.join(input_folder_ica, file)
#                 df = pd.read_excel(file_path)
                
#                 # Prepare dictionaries for storing intermediate results with subband
#                 pairwise_results = {f'{ch1} vs {ch2}': {subband: {'COH': [], 'PLV': [], 'PLI': []} for subband in subbandings} for ch1 in channels_interest for ch2 in channels_interest if ch1 != ch2}
#                 channel_avgs = {ch: {subband: {'COH': [], 'PLV': [], 'PLI': []} for subband in subbandings} for ch in channels_interest}
#                 all_values = {subband: {'COH': [], 'PLV': [], 'PLI': []} for subband in subbandings}
                
#                 # Calculate COH, PLV, and PLI for each channel pair
#                 for i, ch1 in enumerate(channels_interest):
#                     for ch2 in channels_interest[i+1:]:
#                         # Extract data for each channel and subband (ICA, Delta, Theta, Alpha, Beta, Gamma)
#                         for subband in subbandings:
#                             col1, col2 = f'{ch1}_{subband}', f'{ch2}_{subband}'
#                             if col1 in df.columns and col2 in df.columns:
#                                 data1, data2 = df[col1].values, df[col2].values

#                                 mean_coh, mean_plv, mean_pli = calculate_coh_plv_pli(data1, data2, sfreq)
                                
#                                 # Update result_data with each measure and subband
#                                 pairwise_results[f'{ch1} vs {ch2}'][subband]['COH'].append(mean_coh)
#                                 pairwise_results[f'{ch1} vs {ch2}'][subband]['PLV'].append(mean_plv)
#                                 pairwise_results[f'{ch1} vs {ch2}'][subband]['PLI'].append(mean_pli)
#                                 pairwise_results[f'{ch2} vs {ch1}'][subband]['COH'].append(mean_coh)
#                                 pairwise_results[f'{ch2} vs {ch1}'][subband]['PLV'].append(mean_plv)
#                                 pairwise_results[f'{ch2} vs {ch1}'][subband]['PLI'].append(mean_pli)
#                                 channel_avgs[ch1][subband]['COH'].append(mean_coh)
#                                 channel_avgs[ch2][subband]['COH'].append(mean_coh)
#                                 channel_avgs[ch1][subband]['PLV'].append(mean_plv)
#                                 channel_avgs[ch2][subband]['PLV'].append(mean_plv)
#                                 channel_avgs[ch1][subband]['PLI'].append(mean_pli)
#                                 channel_avgs[ch2][subband]['PLI'].append(mean_pli)
                                
#                                 # Store values for confusion matrices (accumulate per label and emotion)
#                                 ch1_idx, ch2_idx = channels_interest.index(ch1), channels_interest.index(ch2)
#                                 label_matrix_data[label_name]['COH'][subband][ch1_idx, ch2_idx] += mean_coh
#                                 label_matrix_data[label_name]['PLV'][subband][ch1_idx, ch2_idx] += mean_plv
#                                 label_matrix_data[label_name]['PLI'][subband][ch1_idx, ch2_idx] += mean_pli
                                
#                                 emotion_matrix_data[emotion]['COH'][subband][ch1_idx, ch2_idx] += mean_coh
#                                 emotion_matrix_data[emotion]['PLV'][subband][ch1_idx, ch2_idx] += mean_plv
#                                 emotion_matrix_data[emotion]['PLI'][subband][ch1_idx, ch2_idx] += mean_pli

#                                 # Accumulate values for global 'All' average by subband
#                                 all_values[subband]['COH'].append(mean_coh)
#                                 all_values[subband]['PLV'].append(mean_plv)
#                                 all_values[subband]['PLI'].append(mean_pli)

#                                 # Update counts for labels and emotions
#                                 label_counts[label_name] += 1
#                                 emotion_counts[emotion] += 1

#                 # Store the computed averages in result_data by subband
#                 for pair, subband_measures in pairwise_results.items():
#                     for subband, measures in subband_measures.items():
#                         for measure, values in measures.items():
#                             result_data[f'{pair} {measure} {subband}'].append(np.mean(values) if values else np.nan)

#                 for ch, subband_measures in channel_avgs.items():
#                     for subband, measures in subband_measures.items():
#                         for measure, values in measures.items():
#                             result_data[f'{ch} vs All {measure} {subband}'].append(np.mean(values) if values else np.nan)

#                 # Add the averages for all measures by subband
#                 for subband, measures in all_values.items():
#                     for measure, values in measures.items():
#                         result_data[f'All {measure} {subband}'].append(np.mean(values) if values else np.nan)

#                 end_time = time.time()
#                 process_time = int(end_time - start_time)
#                 total_time = int(end_time - start_time_2)
#                 print(f'Connectivity {label_name} {participant_name} = {printTimer(process_time)} (Total: {printTimer(total_time)})')

# # Generate aggregated confusion matrices for each label
# for label, measure_data in label_matrix_data.items():
#     for measure, subband_matrices in measure_data.items():
#         for subband, matrix in subband_matrices.items():
#             # Average by participant count for each label
#             if label_counts[label] > 0:
#                 matrix = matrix / label_counts[label]

#                 # # Ensure all required entries are in result_data
#                 # for i, ch1 in enumerate(channels_interest):
#                 #     for j, ch2 in enumerate(channels_interest):
#                 #         if i != j:
#                 #             result_data[f'{ch1} vs {ch2} {measure} {subband}'].append(matrix[i, j] if matrix.size > 1 else np.nan)
                            
#                 # # Average by channel for All comparisons
#                 # for ch, row_vals in zip(channels_interest, matrix):
#                 #     result_data[f'{ch} vs All {measure} {subband}'].append(np.mean(row_vals) if row_vals.size > 0 else np.nan)
    
#                 # result_data[f'All {measure} {subband}'].append(np.mean(matrix) if matrix.size > 0 else np.nan)

#                 # # Add an aggregated row indicating all participants for this label
#                 # result_data['label'].append(label)
#                 # result_data['emotion'].append('all')
#                 # result_data['participant'].append('all')
            
#                 plt.figure(figsize=(8, 6))
#                 plt.imshow(matrix + matrix.T, cmap='Blues', interpolation='nearest')  # Symmetric matrix
#                 plt.gca().invert_yaxis()
#                 plt.colorbar(label=measure)
#                 plt.xticks(range(len(channels_interest)), channels_interest, rotation=45)
#                 plt.yticks(range(len(channels_interest)), channels_interest)
#                 # plt.gca().invert_yaxis()
#                 plt.title(f'{measure} Matrix - {subband}\nLabel: {label} (Aggregated)')
#                 plt.tight_layout()
    
#                 # Save the aggregated confusion matrix plot
#                 matrix_output_path = os.path.join(output_cm, f'{measure}_Matrix_Label_{label}_{subband}.png')
#                 plt.savefig(matrix_output_path)
#                 plt.close()

# # Generate aggregated confusion matrices for each emotion
# for emotion, measure_data in emotion_matrix_data.items():
#     for measure, subband_matrices in measure_data.items():
#         for subband, matrix in subband_matrices.items():
#             # Average by participant count for each emotion
#             if emotion_counts[emotion] > 0:
#                 matrix = matrix / emotion_counts[emotion]

#                 # # Ensure all required entries are in result_data
#                 # for i, ch1 in enumerate(channels_interest):
#                 #     for j, ch2 in enumerate(channels_interest):
#                 #         if i != j:
#                 #             result_data[f'{ch1} vs {ch2} {measure} {subband}'].append(matrix[i, j] if matrix.size > 1 else np.nan)
                            
#                 # # Average by channel for All comparisons
#                 # for ch, row_vals in zip(channels_interest, matrix):
#                 #     result_data[f'{ch} vs All {measure} {subband}'].append(np.mean(row_vals) if row_vals.size > 0 else np.nan)
    
#                 # result_data[f'All {measure} {subband}'].append(np.mean(matrix) if matrix.size > 0 else np.nan)

#                 # # Add an aggregated row indicating all participants for this emotion
#                 # result_data['label'].append('all')
#                 # result_data['emotion'].append(emotion)
#                 # result_data['participant'].append('all')
            
#                 plt.figure(figsize=(8, 6))
#                 plt.imshow(matrix + matrix.T, cmap='Blues', interpolation='nearest')  # Symmetric matrix
#                 plt.gca().invert_yaxis()
#                 plt.colorbar(label=measure)
#                 plt.xticks(range(len(channels_interest)), channels_interest, rotation=45)
#                 plt.yticks(range(len(channels_interest)), channels_interest)
#                 # plt.gca().invert_yaxis()
#                 plt.title(f'{measure} Matrix - {subband}\nEmotion: {emotion} (Aggregated)')
#                 plt.tight_layout()
    
#                 # Save the aggregated confusion matrix plot
#                 matrix_output_path = os.path.join(output_cm, f'{measure}_Matrix_Emotion_{emotion}_{subband}.png')
#                 plt.savefig(matrix_output_path)
#                 plt.close()

# # Convert result dictionary to DataFrame, remove empty columns, and save to Excel
# df_result = pd.DataFrame(result_data)
# df_result = df_result.dropna(axis=1, how='all')
# df_result.to_excel(output_file_path, index=False)

# print("\nFinished Connectivity")
# end_time = time.time()
# total_time = int(end_time - start_time_2)
# print("Total Time = (" + printTimer(total_time) + ")")

In [7]:
start_time_2 = time.time()

threshold_max = 1e-8
threshold_std = 0.02
scaling_factor = 1e6

small_list = []

classification_df = pd.DataFrame()

# Function to calculate centroid for each ad
def calculate_centroid_for_ad(plot_data, label_name, vector):
    data1key = 'data1' + vector
    data2key = 'data2' + vector
    
    # Calculate mean for data1 and data2 as centroid
    centroid_data1 = plot_data[data1key].mean()
    centroid_data2 = plot_data[data2key].mean()
    
    return centroid_data1, centroid_data2

# Create a function to process each ad
def process_label_data_mav(label_name, label_info):
    global chunks
    
    emotion = label_info['emotion']
    chunks_range = label_info['chunks']
    start = label_info['start']
    seconds = label_info['seconds']
    color = label_info['color']
    scatter_data = []
    
    sample_rates_df = pd.read_excel("sample_rates.xlsx")
    sample_rates_dict = dict(zip(sample_rates_df['filename'], sample_rates_df['sample_rate']))

    # Iterate over all Excel files in the folder
    for file in os.listdir(input_folder_mav):
        if file.endswith('.xlsx') and label_name in file:
            participant_name = file.replace(f"{label_name}_", "").replace(".xlsx", "")

            # Skip corrupts data
            if any(entry['participant'] == participant_name and entry['label'] == label_name for entry in corrupts_data):
                continue

            start_time = time.time()

            # Change category corrupts data
            # entry = next((entry for entry in corrupts_data if entry['participant'] == participant_name and entry['label'] == label_name), None)
            # if entry:
            #     emotion = entry['emotion']
    
            file_path = os.path.join(input_folder_mav, file)

            sfreq = 250
            filename_key = f'{label_name}_{participant_name}'
            if filename_key in sample_rates_dict:
                sfreq = sample_rates_dict[filename_key]
            start_range = start * sfreq
            end_range = seconds * sfreq
            
            chunks_range = range(start_range, end_range)
            
            # Load the MAV Excel file
            df = pd.read_excel(file_path)
            
            # Filter by the chunks of interest
            df_filtered = df[df['Chunk (0.2)'].isin(chunks_range)]

            sample_rate = (1 / chunks) * df['Data Length'][0]
            
            del df
            gc.collect()

            # Extract relevant columns for each channel, subband, and feature
            for feature in features_full:
                for subband in subbands_interest:
                    for i in range(0, len(channels_interest), 2):  # Process in pairs: F7-F8, T7-T8, P7-P8
                        ch1, ch2 = channels_interest[i], channels_interest[i + 1]
                        col1 = f"{ch1}_{subband}_{feature}"
                        col2 = f"{ch2}_{subband}_{feature}"
                        
                        if col1 in df_filtered.columns and col2 in df_filtered.columns:
                            # print(f'{label_name}_{participant_name}_{ch1} {ch2}_{subband}_{feature}')
                            data1 = df_filtered[col1].values
                            data2 = df_filtered[col2].values

                            # if data1.size < 1:
                            #     print(f'{label_name}_{participant_name}_{ch1} {ch2}_{subband}_{feature}')
                            # data1 = np.nan_to_num(data1, nan=1e-6, posinf=1e-6, neginf=1e-6)
                            # data2 = np.nan_to_num(data2, nan=1e-6, posinf=1e-6, neginf=1e-6)

                            # data1 += 1e-6
                            # data2 += 1e-6

                            data1_transformed = None
                            data2_transformed = None
                            # mean_coherence_transformed = None
                            # mean_plv_transformed = None
                            # mean_pli_transformed = None
                            
                            # Apply power transformation to spread out the data
                            if np.max(data1) > threshold_max and np.std(data1) > threshold_std and feature != 'Zero Crossing Rate':
                                # print(f'{label_name}|{participant_name}|{feature}|{ch1}|{subband}')
                                try:
                                    data1_transformed = power_transform(data1.reshape(-1, 1), method='yeo-johnson').flatten()
                                except Exception as e:
                                    data1_transformed = None
                                    # print(f'{label_name}|{participant_name}|{feature}|{ch1}|{subband}')
                            else:
                                data1 = np.nan_to_num(data1, nan=1e-6, posinf=1e-6, neginf=1e-6)
                                # data1 *= scaling_factor
                                small_list.append(f'{label_name}_{participant_name}_{ch1}_{subband}_{feature}')
                            if np.max(data2) > threshold_max and np.std(data2) > threshold_std and feature != 'Zero Crossing Rate':
                                try:
                                    data2_transformed = power_transform(data2.reshape(-1, 1), method='yeo-johnson').flatten()
                                except Exception as e:
                                    data2_transformed = None
                                    # print(f'{label_name}|{participant_name}|{feature}|{ch2}|{subband}')
                            else:
                                data2 = np.nan_to_num(data2, nan=1e-6, posinf=1e-6, neginf=1e-6)
                                # data2 *= scaling_factor
                                small_list.append(f'{label_name}_{participant_name}_{ch2}_{subband}_{feature}')

                            # freqs, coherence_values = signal.coherence(data1, data2, fs=sample_rate, nperseg=len(data1))
                            # mean_coherence = np.mean(coherence_values)

                            # # Calculate PLV (Phase Locking Value)
                            # phase_diff = np.angle(data1) - np.angle(data2)
                            # plv_values = np.abs(np.mean(np.exp(1j * phase_diff)))
                            # mean_plv = np.mean(plv_values)
                            
                            # # Calculate PLI (Phase Lag Index)
                            # pli_values = np.sign(phase_diff)
                            # mean_pli = np.abs(np.mean(pli_values))

                            # if data1_transformed is not None and data2_transformed is not None and isinstance(data1_transformed, np.ndarray) and isinstance(data2_transformed, np.ndarray):
                            if data1_transformed is not None and data2_transformed is not None:
                                # print(f'{label_name}|{participant_name}|{feature}|{ch1}|{ch2}|{subband}')
                                freqs_transformed, coherence_values_transformed = signal.coherence(data1_transformed, data2_transformed, fs=sample_rate, nperseg=len(data1_transformed))
                                mean_coherence_transformed = np.mean(coherence_values_transformed)

                                # # Calculate PLV (Phase Locking Value)
                                # phase_diff_transformed = np.angle(data1_transformed) - np.angle(data2_transformed)
                                # plv_values_transformed = np.abs(np.mean(np.exp(1j * phase_diff_transformed)))
                                # mean_plv_transformed = np.mean(plv_values_transformed)
                                
                                # # Calculate PLI (Phase Lag Index)
                                # pli_values_transformed = np.sign(phase_diff_transformed)
                                # mean_pli_transformed = np.abs(np.mean(pli_values_transformed))
                            # else:
                            #     print(f'{label_name}|{participant_name}|{feature}|{ch1}|{subband}')

                            if feature == 'Zero Crossing Rate':
                                data1mean = np.sum(data1) / seconds
                                data2mean = np.sum(data2) / seconds
                            else:
                                data1mean = np.mean(data1)
                                data2mean = np.mean(data2)
                            
                            scatter_data.append({
                                'label': label_name,
                                'emotion': emotion,
                                'participant': participant_name,
                                'subband': subband,
                                'feature': feature,
                                'ch1': ch1,
                                'ch2': ch2,
                                'data1mean': data1mean,
                                'data2mean': data2mean,
                                'data1var': np.var(data1),
                                'data2var': np.var(data2),
                                'data1ptp': np.ptp(data1),
                                'data2ptp': np.ptp(data2),
                                # 'coh': mean_coherence,
                                # 'pli': mean_pli,
                                # 'plv': mean_plv,
                                'data1mean_transformed': np.mean(data1_transformed) if data1_transformed is not None and isinstance(data1_transformed, np.ndarray) else 0,
                                'data2mean_transformed': np.mean(data2_transformed) if data2_transformed is not None and isinstance(data2_transformed, np.ndarray) else 0,
                                'data1var_transformed': np.var(data1_transformed) if data1_transformed is not None and isinstance(data1_transformed, np.ndarray) else 0,
                                'data2var_transformed': np.var(data2_transformed) if data2_transformed is not None and isinstance(data2_transformed, np.ndarray) else 0,
                                'data1ptp_transformed': np.ptp(data1_transformed) if data1_transformed is not None and isinstance(data1_transformed, np.ndarray) else 0,
                                'data2ptp_transformed': np.ptp(data2_transformed) if data2_transformed is not None and isinstance(data2_transformed, np.ndarray) else 0,
                                # 'coh_transformed': mean_coherence_transformed if mean_coherence_transformed is not None else 0,
                                # 'pli_transformed': mean_pli_transformed if mean_pli_transformed is not None else 0,
                                # 'plv_transformed': mean_plv_transformed if mean_plv_transformed is not None else 0,
                                'color': color
                            })
    
            end_time = time.time()
            total_time = int(end_time - start_time)
            print(f'Process {label_name} {participant_name} = {printTimer(total_time)}')
    print()

    return scatter_data

# Process each ad and collect data for scatter plots
all_scatter_data = []
label_centroids = {}
for label_name, label_info in labels_data.items():
    label_scatter_data = process_label_data_mav(label_name, label_info)
    all_scatter_data.extend(label_scatter_data)
    
    # Convert to DataFrame for easy handling
    scatter_df = pd.DataFrame(label_scatter_data)

    label_centroids[label_name] = {}
    for vector in vectors:
        # Calculate the centroid for this ad
        centroid_data1, centroid_data2 = calculate_centroid_for_ad(scatter_df, label_name, vector)
        
        # Store the centroid
        label_centroids[label_name][vector] = (centroid_data1, centroid_data2)

# Convert to DataFrame for easy handling
scatter_df = pd.DataFrame(all_scatter_data)

# Create scatter plots for each feature, channel pair, and subband
for feature in features_full:
    # Create feature-specific output folder
    output_folder_scatter_feature = os.path.join(output_folder_scatter_base, feature.replace(' ', '_').lower())
    os.makedirs(output_folder_scatter_feature, exist_ok=True)

    start_time = time.time()
    
    for subband in subbands_interest:
        for i in range(0, len(channels_interest), 2):
            ch1, ch2 = channels_interest[i], channels_interest[i + 1]
            plot_data = scatter_df[(scatter_df['ch1'] == ch1) & 
                                   (scatter_df['ch2'] == ch2) & 
                                   (scatter_df['subband'] == subband) & 
                                   (scatter_df['feature'] == feature)]

            for vector in vectors:
                data1key = 'data1' + vector
                data2key = 'data2' + vector
                
                # Create the plot
                plt.figure(figsize=(10, 6))
                for label_name, label_info in labels_data.items():
                    label_data = plot_data[plot_data['label'] == label_name]
                    plt.scatter(label_data[data1key], label_data[data2key], color=label_info['color'], label=f"{label_name} ({label_info['emotion']})", alpha=0.7)
                    
                    # Calculate the centroid for this ad (mean of data1 and data2)
                    centroid_data1 = label_data[data1key].mean()
                    centroid_data2 = label_data[data2key].mean()
    
                    # Plot the centroid as a separate point with a distinctive marker and size
                    plt.scatter(centroid_data1, centroid_data2, color=label_info['color'], marker='X', s=200, edgecolor='black', label=f"{label_name} Centroid", zorder=5)

                    # Add trendline (linear regression)
                    if len(label_data) > 1:
                        # Perform linear regression to get the slope and intercept
                        slope, intercept = np.polyfit(label_data[data1key], label_data[data2key], 1)
                        trendline = np.poly1d([slope, intercept])

                        # Generate x values for the trendline
                        x_values = np.linspace(label_data[data1key].min(), label_data[data1key].max(), 100)
                        # Plot the trendline
                        plt.plot(x_values, trendline(x_values), color=label_info['color'], linestyle='--', label=f"{label_name} Trendline")
                
                plt.title(f"{ch1}_{subband} vs {ch2}_{subband} ({feature} - {vector})")
                plt.xlabel(f"{ch1}_{subband} {feature}")
                plt.ylabel(f"{ch2}_{subband} {feature}")
                plt.legend()
                plt.grid(True)
    
                # Save the plot as an image
                plt_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_{vector}_vs_{ch2}_{vector}_{subband}.png")
                plt.savefig(plt_path)
                plt.close()

            # Save the scatter data to Excel
            # excel_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_vs_{ch2}_{subband}.xlsx")
            # plot_data[['label', 'emotion', 'participant', 'data1', 'data2']].to_excel(excel_path, index=False)

            data_to_add = []
            if label_data['data1mean_transformed'].mean() != 0 and label_data['data2mean_transformed'].mean() != 0:
                data_to_add = plot_data[['label', 'emotion', 'participant',
                                         'data1mean', 'data2mean',
                                         'data1mean_transformed', 'data2mean_transformed',
                                         # 'data1var', 'data2var', 'data1var_transformed',
                                         # 'data2var_transformed', 'data1ptp', 'data2ptp',
                                         # 'data1ptp_transformed', 'data2ptp_transformed',
                                         # 'coh', 'coh_transformed',
                                         # 'pli', 'pli_transformed',
                                         # 'plv', 'plv_transformed'
                                        ]].copy()
                data_to_add.columns = ['label', 'emotion', 'participant',
                                       f'{ch1}_{subband}_{feature}_mean', f'{ch2}_{subband}_{feature}_mean',
                                       f'{ch1}_{subband}_{feature}_mean_Transformed', f'{ch2}_{subband}_{feature}_mean_Transformed',
                                       # f'{ch1}_{subband}_{feature}_var', f'{ch2}_{subband}_{feature}_var',
                                       # f'{ch1}_{subband}_{feature}_var_Transformed', f'{ch2}_{subband}_{feature}_var_Transformed',
                                       # f'{ch1}_{subband}_{feature}_ptp', f'{ch2}_{subband}_{feature}_ptp',
                                       # f'{ch1}_{subband}_{feature}_ptp_Transformed', f'{ch2}_{subband}_{feature}_ptp_Transformed',
                                       # f'{ch1}_{ch2}_{subband}_{feature}_coh', f'{ch1}_{ch2}_{subband}_{feature}_coh_Transformed',
                                       # f'{ch1}_{ch2}_{subband}_{feature}_pli', f'{ch1}_{ch2}_{subband}_{feature}_pli_Transformed',
                                       # f'{ch1}_{ch2}_{subband}_{feature}_plv', f'{ch1}_{ch2}_{subband}_{feature}_plv_Transformed'
                                      ]
                
                for vector in vectors:
                    data1key = 'data1' + vector + '_transformed'
                    data2key = 'data2' + vector + '_transformed'
                    
                    # Create the transformed plot
                    plt.figure(figsize=(10, 6))
                    for label_name, label_info in labels_data.items():
                        label_data = plot_data[plot_data['label'] == label_name]
                        plt.scatter(label_data[data1key], label_data[data2key], color=label_info['color'], label=f"{label_name} ({label_info['emotion']})", alpha=0.7)
                    
                        # Calculate the centroid for this ad (mean of data1 and data2)
                        centroid_data1 = label_data[data1key].mean()
                        centroid_data2 = label_data[data2key].mean()
        
                        # Plot the centroid as a separate point with a distinctive marker and size
                        plt.scatter(centroid_data1, centroid_data2, color=label_info['color'], marker='X', s=200, edgecolor='black', label=f"{label_name} Centroid", zorder=5)

                        # Add trendline (linear regression)
                        if len(label_data) > 1:
                            # Perform linear regression to get the slope and intercept
                            slope, intercept = np.polyfit(label_data[data1key], label_data[data2key], 1)
                            trendline = np.poly1d([slope, intercept])
    
                            # Generate x values for the trendline
                            x_values = np.linspace(label_data[data1key].min(), label_data[data1key].max(), 100)
                            # Plot the trendline
                            plt.plot(x_values, trendline(x_values), color=label_info['color'], linestyle='--', label=f"{label_name} Trendline")
                
                    plt.title(f"{ch1}_{subband} vs {ch2}_{subband} ({feature} - {vector}) Transformed")
                    plt.xlabel(f"{ch1}_{subband} {feature}")
                    plt.ylabel(f"{ch2}_{subband} {feature}")
                    plt.legend()
                    plt.grid(True)
        
                    # Save the plot as an image
                    plt_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_{vector}_vs_{ch2}_{subband}_{vector}_transformed.png")
                    plt.savefig(plt_path)
                    plt.close()
            else:
                data_to_add = plot_data[['label', 'emotion', 'participant',
                                         'data1mean', 'data2mean',
                                         # 'data1var', 'data2var',
                                         # 'data1ptp', 'data2ptp',
                                         # 'coh', 'pli', 'plv'
                                        ]].copy()
                data_to_add.columns = ['label', 'emotion', 'participant',
                                       f'{ch1}_{subband}_{feature}_mean', f'{ch2}_{subband}_{feature}_mean',
                                       # f'{ch1}_{subband}_{feature}_var', f'{ch2}_{subband}_{feature}_var',
                                       # f'{ch1}_{subband}_{feature}_ptp', f'{ch2}_{subband}_{feature}_ptp',
                                       # f'{ch1}_{ch2}_{subband}_{feature}_coh', f'{ch1}_{ch2}_{subband}_{feature}_pli',
                                       # f'{ch1}_{ch2}_{subband}_{feature}_plv'
                                      ]

            # Save the scatter data to Excel
            # excel_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_vs_{ch2}_{subband}_transformed.xlsx")
            # plot_data[['label', 'emotion', 'participant', 'data1_transformed', 'data2_transformed']].to_excel(excel_path, index=False)
            
            # Select the relevant columns from plot_data to classification_df
            # data_to_add = plot_data[['label', 'emotion', 'participant', 'data1', 'data2', 'data1_transformed', 'data2_transformed']].copy()
            # data_to_add.columns = ['label', 'emotion', 'participant', f'{ch1}_{subband}_{feature}', f'{ch2}_{subband}_{feature}', f'{ch1}_{subband}_{feature}_Transformed', f'{ch2}_{subband}_{feature}_Transformed']

            if label_data['data1mean_transformed'].mean() != 0 and label_data['data2mean_transformed'].mean() != 0:
                # If the classification_df is empty, initialize it with the first set of columns
                if classification_df.empty:
                    classification_df = data_to_add
                else:
                    # Merge the new columns to the existing classification_df, aligning by 'label', 'emotion', 'participant'
                    classification_df = pd.merge(
                        classification_df, 
                        data_to_add[['label', 'emotion', 'participant',
                                     f'{ch1}_{subband}_{feature}_mean', f'{ch2}_{subband}_{feature}_mean',
                                     f'{ch1}_{subband}_{feature}_mean_Transformed', f'{ch2}_{subband}_{feature}_mean_Transformed',
                                     # f'{ch1}_{subband}_{feature}_var', f'{ch2}_{subband}_{feature}_var',
                                     # f'{ch1}_{subband}_{feature}_var_Transformed', f'{ch2}_{subband}_{feature}_var_Transformed',
                                     # f'{ch1}_{subband}_{feature}_ptp', f'{ch2}_{subband}_{feature}_ptp',
                                     # f'{ch1}_{subband}_{feature}_ptp_Transformed', f'{ch2}_{subband}_{feature}_ptp_Transformed',
                                     # f'{ch1}_{ch2}_{subband}_{feature}_coh', f'{ch1}_{ch2}_{subband}_{feature}_coh_Transformed',
                                     # f'{ch1}_{ch2}_{subband}_{feature}_pli', f'{ch1}_{ch2}_{subband}_{feature}_pli_Transformed',
                                     # f'{ch1}_{ch2}_{subband}_{feature}_plv', f'{ch1}_{ch2}_{subband}_{feature}_plv_Transformed'
                                    ]],
                        on=['label', 'emotion', 'participant'],
                        how='outer'
                    )
            else:
                # If the classification_df is empty, initialize it with the first set of columns
                if classification_df.empty:
                    classification_df = data_to_add
                else:
                    # Merge the new columns to the existing classification_df, aligning by 'label', 'emotion', 'participant'
                    classification_df = pd.merge(
                        classification_df, 
                        data_to_add[['label', 'emotion', 'participant',
                                     f'{ch1}_{subband}_{feature}_mean', f'{ch2}_{subband}_{feature}_mean',
                                     # f'{ch1}_{subband}_{feature}_var', f'{ch2}_{subband}_{feature}_var',
                                     # f'{ch1}_{subband}_{feature}_ptp', f'{ch2}_{subband}_{feature}_ptp',
                                     # f'{ch1}_{ch2}_{subband}_{feature}_coh',
                                     # f'{ch1}_{ch2}_{subband}_{feature}_pli',
                                     # f'{ch1}_{ch2}_{subband}_{feature}_plv'
                                    ]],
                        on=['label', 'emotion', 'participant'],
                        how='outer'
                    )

            for vector in vectors:
                data1key = 'data1' + vector
                data2key = 'data2' + vector
                data1key_transformed = 'data1' + vector + '_transformed'
                data2key_transformed = 'data2' + vector + '_transformed'
                
                X = plot_data[[data1key, data2key]].values  # Data for clustering  # Data for clustering
                if len(X) > 1:  # Ensure we have enough data points to cluster
                    kmeans = KMeans(n_clusters=len(labels_data), random_state=42)
                    # plot_data['cluster'] = kmeans.fit_predict(X)
                    plot_data = plot_data.copy()
                    plot_data.loc[:, 'cluster'] = kmeans.fit_predict(X)
                    
                    # Save the clustering results to Excel
                    cluster_excel_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_{vector}_vs_{ch2}_{subband}_{vector}.xlsx")
                    plot_data[['label', 'emotion', 'participant', data1key, data2key, 'cluster',
                               # 'coh', 'pli', 'plv'
                              ]].to_excel(cluster_excel_path, index=False)
                    
                    # Plot the clusters
                    plt.figure(figsize=(10, 6))
                    sns.scatterplot(x=data1key, y=data2key, hue='cluster', palette='Set1', data=plot_data, legend='full', s=100)
                    
                    # Add trendline (without hue, just one line for the overall data)
                    sns.regplot(x=data1key, y=data2key, data=plot_data, scatter=False, color='gray', line_kws={'linestyle': '--'}, ci=None)
                    
                    # Add cluster centroids
                    centroids = kmeans.cluster_centers_
                    plt.scatter(centroids[:, 0], centroids[:, 1], c='black', s=200, alpha=0.75, marker='X', label='Centroids')
                
                    # Customize the plot
                    plt.title(f"Clustered {ch1}_{subband} vs {ch2}_{subband} ({feature} - {vector})")
                    plt.xlabel(f"{ch1}_{subband} {feature} {vector}")
                    plt.ylabel(f"{ch2}_{subband} {feature} {vector}")
                    plt.legend()
                    plt.grid(True)
    
                    # Save the plot with clustering
                    cluster_plot_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_{vector}_vs_{ch2}_{subband}_{vector}_cluster.png")
                    plt.savefig(cluster_plot_path)
                    plt.close()
    
                if label_data[data1key_transformed].mean() != 0 and label_data[data2key_transformed].mean() != 0:
                    X = plot_data[[data1key_transformed, data2key_transformed]].values  # Data for clustering  # Data for clustering
                    if len(X) > 1:  # Ensure we have enough data points to cluster
                        kmeans = KMeans(n_clusters=len(labels_data), random_state=42)
                        # plot_data['cluster'] = kmeans.fit_predict(X)
                        plot_data = plot_data.copy()
                        plot_data.loc[:, 'cluster'] = kmeans.fit_predict(X)
                        
                        # Save the clustering results to Excel
                        cluster_excel_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_{vector}_vs_{ch2}_{subband}_{vector}_transformed.xlsx")
                        plot_data[['label', 'emotion', 'participant', data1key_transformed, data2key_transformed, 'cluster',
                                   # 'coh_transformed', 'pli_transformed', 'plv_transformed'
                                  ]].to_excel(cluster_excel_path, index=False)
                        
                        # Plot the clusters
                        plt.figure(figsize=(10, 6))
                        sns.scatterplot(x=data1key_transformed, y=data2key_transformed, hue='cluster', palette='Set1', data=plot_data, legend='full', s=100)
                    
                        # Add trendline (without hue, just one line for the overall data)
                        sns.regplot(x=data1key_transformed, y=data2key_transformed, data=plot_data, scatter=False, color='gray', line_kws={'linestyle': '--'}, ci=None)
                        
                        # Add cluster centroids
                        centroids = kmeans.cluster_centers_
                        plt.scatter(centroids[:, 0], centroids[:, 1], c='black', s=200, alpha=0.75, marker='X', label='Centroids')
        
                        # Customize the plot
                        plt.title(f"Clustered {ch1}_{subband} vs {ch2}_{subband} Transformed ({feature} - {vector})")
                        plt.xlabel(f"{ch1}_{subband} {feature} {vector}")
                        plt.ylabel(f"{ch2}_{subband} {feature} {vector}")
                        plt.legend()
                        plt.grid(True)
        
                        # Save the plot with clustering
                        cluster_plot_path = os.path.join(output_folder_scatter_feature, f"{ch1}_{subband}_{vector}_vs_{ch2}_{subband}_{vector}_transformed_cluster.png")
                        plt.savefig(cluster_plot_path)
                        plt.close()
    
    end_time = time.time()
    process_time = int(end_time - start_time)
    total_time = int(end_time - start_time_2)
    print(f'Process {feature} = {printTimer(process_time)} (Total: {printTimer(total_time)})')

classification_output_path = os.path.join(output_folder, 'classification_data.xlsx')
classification_df.to_excel(classification_output_path, index=False)

del scatter_df
del all_scatter_data
del classification_df
gc.collect()

print("\nFinished Scatter Cluster plots with power transformation")
end_time = time.time()
total_time = int(end_time - start_time_2)
print("Total Time = (" + printTimer(total_time) + ")")

small_string = '\n'.join(small_list)
with open('EEG Failed Analysis.txt', 'w') as file:
    file.write("Small:\n")
    file.write(small_string)

Process bengbeng dataresponden1 = 1 second
Process bengbeng dataresponden10 = 1 second
Process bengbeng dataresponden11 = 1 second
Process bengbeng dataresponden12 = 1 second
Process bengbeng dataresponden13 = 1 second
Process bengbeng dataresponden14 = 1 second
Process bengbeng dataresponden15 = 1 second
Process bengbeng dataresponden16 = 1 second
Process bengbeng dataresponden17 = 1 second
Process bengbeng dataresponden18 = 1 second
Process bengbeng dataresponden19 = 1 second
Process bengbeng dataresponden2 = 1 second
Process bengbeng dataresponden20 = 1 second
Process bengbeng dataresponden21 = 1 second
Process bengbeng dataresponden22 = 1 second
Process bengbeng dataresponden23 = 1 second
Process bengbeng dataresponden24 = 1 second
Process bengbeng dataresponden25 = 1 second
Process bengbeng dataresponden26 = 1 second
Process bengbeng dataresponden27 = 1 second
Process bengbeng dataresponden28 = 1 second
Process bengbeng dataresponden29 = 1 second
Process bengbeng dataresponden3 = 

In [8]:
# # Define the directory and load the Excel file
# file_path = os.path.join(directory_analysis, "classification_data.xlsx")
# df = pd.read_excel(file_path)

# # Filter columns to exclude non-data columns
# non_data_columns = ['label', 'emotion', 'participant']
# data_columns = [col for col in df.columns if col not in non_data_columns]

# # Split the data into groups based on the 'emotion' column
# group_interested = df[df['emotion'] == 'interested']
# group_meh = df[df['emotion'] == 'meh']

# # Initialize an empty list to store the results
# results = []

# # Loop through each data column and calculate t-test and ANOVA
# for column in data_columns:
#     try:
#         # Perform t-test
#         t_stat, p_value_ttest = ttest_ind(
#             group_interested[column],
#             group_meh[column],
#             equal_var=False  # Welch's t-test
#         )

#         # Perform ANOVA
#         f_stat, p_value_anova = f_oneway(
#             group_interested[column],
#             group_meh[column]
#         )

#         # Append results to the list
#         results.append({
#             'Feature': column,
#             't_stat': t_stat,
#             'p_value_ttest': p_value_ttest,
#             'f_stat': f_stat,
#             'p_value_anova': p_value_anova
#         })
#     except Exception as e:
#         # Handle any errors gracefully
#         results.append({
#             'Feature': column,
#             't_stat': None,
#             'p_value_ttest': None,
#             'f_stat': None,
#             'p_value_anova': None,
#             'Error': str(e)
#         })

# # Convert the results into a DataFrame
# results_df = pd.DataFrame(results)

# # Save the results to an Excel file in the same directory
# output_path = os.path.join(directory_analysis, "t_test_anova_classification_results.xlsx")
# results_df.to_excel(output_path, index=False)

# file_path = os.path.join(directory_analysis, "connectivity_results.xlsx")
# df = pd.read_excel(file_path)

# # Filter columns to exclude non-data columns
# non_data_columns = ['label', 'emotion', 'participant']
# data_columns = [col for col in df.columns if col not in non_data_columns]

# # Split the data into groups based on the 'emotion' column
# group_interested = df[df['emotion'] == 'interested']
# group_meh = df[df['emotion'] == 'meh']

# # Initialize an empty list to store the results
# results = []

# # Loop through each data column and calculate t-test and ANOVA
# for column in data_columns:
#     try:
#         # Perform t-test
#         t_stat, p_value_ttest = ttest_ind(
#             group_interested[column],
#             group_meh[column],
#             equal_var=False  # Welch's t-test
#         )

#         # Perform ANOVA
#         f_stat, p_value_anova = f_oneway(
#             group_interested[column],
#             group_meh[column]
#         )

#         # Append results to the list
#         results.append({
#             'Feature': column,
#             't_stat': t_stat,
#             'p_value_ttest': p_value_ttest,
#             'f_stat': f_stat,
#             'p_value_anova': p_value_anova
#         })
#     except Exception as e:
#         # Handle any errors gracefully
#         results.append({
#             'Feature': column,
#             't_stat': None,
#             'p_value_ttest': None,
#             'f_stat': None,
#             'p_value_anova': None,
#             'Error': str(e)
#         })

# # Convert the results into a DataFrame
# results_df = pd.DataFrame(results)

# # Save the results to an Excel file in the same directory
# output_path = os.path.join(directory_analysis, "t_test_anova_connectivity_results.xlsx")
# results_df.to_excel(output_path, index=False)

In [9]:
# confusion_matrix_folder = 'imotion_analysis/classification'
# os.makedirs(confusion_matrix_folder, exist_ok=True)

# start_time_2 = time.time()

# counter = 1

# classification_df = pd.read_excel("imotion_analysis/classification_data.xlsx")

# removes_data = {key: labels_data[key] for key in labels_data if key not in classifiers_data}

# # Remove data to be removed (e.g. baseline)
# for key in removes_data:
#     classification_df = classification_df[classification_df['label'] != key]
    
# # Remove corrupt data
# for corrupt in corrupts_data:
#     classification_df = classification_df[~((classification_df['label'] == corrupt['label']) & (classification_df['participant'] == corrupt['participant']))]

# # Define label column
# # label_column = ['label','emotion']
# label_column = ['emotion']

# emotions = set(label_info['emotion'] for label_info in classifiers_data.values())
# emotions_total = len(emotions)

# # Initialize a list to store the results
# results = []

# # Dynamically generate all feature columns based on channels, subbands, and features
# # feature_columns = []
# # feature_transformed_columns = []
# # for feature in features_interest:
#     # for subband in subbands_interest:
#         # for i in range(0, len(channels_interest), 2):  # Process channels in pairs (F7-F8, T7-T8, etc.)
#         #     ch1 = channels_interest[i]
#         #     ch2 = channels_interest[i + 1]
#         #     feature_columns.append(f'{ch1}_{subband}_{feature}')
#         #     feature_columns.append(f'{ch2}_{subband}_{feature}')
#         #     feature_transformed_columns.append(f'{ch1}_{subband}_{feature}_Transformed')
#         #     feature_transformed_columns.append(f'{ch2}_{subband}_{feature}_Transformed')

# label_encoder = LabelEncoder()

# # Iterate through each classifier and calculate accuracy
# for classifier_name, classifier in classifiers.items():
#     start_time = time.time()
#     classifier_results = []
#     top_models = []  # To store the top 5 models for each classifier
    
#     for classy in classifier_interest:
#         feature_columns = []
#         feature_transformed_columns = []
#         for feature in features_interest:
#             temp = []
#             temp_transformed = []
#             for c in classy:
#                 temp.append(f'{c}_{feature}')
#                 temp_transformed.append(f'{c}_{feature}_Transformed')
#             feature_columns.append(temp)
#             feature_transformed_columns.append(temp_transformed)
    
#         all_combinations = []
        
#         # Loop through different lengths of combinations
#         for r in range(1, len(feature_columns) + 1):  # Generate combinations of size 1 to size of features
#             for combination in itertools.combinations(feature_columns, r):
#                 # Flatten each combination of lists
#                 flattened_combination = [item for sublist in combination for item in sublist]
#                 all_combinations.append(flattened_combination)
                
#         all_combinations_transformed = []
        
#         # Loop through different lengths of combinations
#         for r in range(1, len(feature_transformed_columns) + 1):  # Generate combinations of size 1 to size of features
#             for combination in itertools.combinations(feature_transformed_columns, r):
#                 # Flatten each combination of lists
#                 flattened_combination = [item for sublist in combination for item in sublist]
#                 all_combinations_transformed.append(flattened_combination)

#         for r in range(1, len(vectors) + 1):
#             for vector_combination in itertools.combinations(vectors, r):
                
#                 for label in label_column:
#                     for combo in all_combinations:
#                         classifier_combo = []
#                         feature_combination = list(combo)

#                         for f in feature_combination:
#                             for v in vector_combination:
#                                 classifier_combo.append(f'{f}_{v}')
                                
#                         missing_features = [feature for feature in classifier_combo if feature not in classification_df.columns]
#                         if missing_features:
#                             continue
                        
#                         # Prepare data (handling missing columns)
#                         X = classification_df[classifier_combo].dropna()  # Use only available columns, drop rows with missing data
#                         y = classification_df.loc[X.index, label]  # Match the label data with selected rows
                        
#                         # Train-test split
#                         # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

#                         y_encoded = label_encoder.fit_transform(y)
                
#                         # Perform cross-validation (using 5-fold cross-validation as an example)# Check the minimum class size
#                         # min_class_size = y.value_counts().min()
#                         min_class_size = y_encoded.min()
#                         cv_value = max(2, min(10, min_class_size - 1))
#                         # cv_value = max(2, min(10, int(len(X) / emotions_total) - 1))
                
#                         scores = cross_val_score(classifier, X, y_encoded, cv=cv_value)
                
#                         mean_accuracy = scores.mean()
                        
#                         # Store the result
#                         result = {
#                             'Counter': counter,
#                             'Features': ', '.join(feature_combination),
#                             'Vectors': ', '.join(vector_combination),
#                             'Classifier': classifier_name,
#                             'Label': label,
#                             'Transformed': 0,
#                             'Accuracy': mean_accuracy
#                         }
#                         results.append(result)
#                         classifier_results.append(result)
        
#                         # Track top models
#                         if len(top_models) < 5:
#                             top_models.append((classifier, mean_accuracy, result, counter))
#                         else:
#                             # Replace the lowest accuracy model if current model is better
#                             min_accuracy = min(top_models, key=lambda x: x[1])[1]
#                             if mean_accuracy > min_accuracy:
#                                 top_models.remove(min(top_models, key=lambda x: x[1]))
#                                 top_models.append((classifier, mean_accuracy, result, counter))
                                    
#                         # Create confusion matrix if mean accuracy is at least 80%
#                         if mean_accuracy >= 0.80:
#                             y_pred = cross_val_predict(classifier, X, y, cv=cv_value)
#                             cm = confusion_matrix(y, y_pred, labels=y.unique())
                            
#                             plt.close()
#                             plt.figure(figsize=(8, 6))
#                             sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=y.unique(), yticklabels=y.unique())
#                             plt.title(f'{counter} {classifier_name} Confusion Matrix ({label})')
#                             plt.xlabel('Predicted')
#                             plt.ylabel('Actual')
#                             confusion_matrix_path = os.path.join(confusion_matrix_folder, f'{counter}_{classifier_name}_{label}_Confusion Matrix.png')
#                             plt.savefig(confusion_matrix_path)
#                             plt.close()
        
#                         del X
#                         del y
#                         gc.collect()
#                         counter += 1
                    
#                     for combo in all_combinations_transformed:
#                         classifier_combo = []
#                         feature_combination = list(combo)

#                         for f in feature_combination:
#                             for v in vector_combination:
#                                 classifier_combo.append(f'{f}_{v}')
                                
#                         missing_features = [feature for feature in classifier_combo if feature not in classification_df.columns]
#                         if missing_features:
#                             continue
                            
#                         # Prepare data (handling missing columns)
#                         X = classification_df[classifier_combo].dropna().loc[:, (classification_df[classifier_combo] != 0).any(axis=0)]  # Use only available columns, drop rows with missing data
#                         y = classification_df.loc[X.index, label]  # Match the label data with selected rows
                        
#                         # Train-test split
#                         # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

#                         y_encoded = label_encoder.fit_transform(y)
                        
#                         # Perform cross-validation (using 5-fold cross-validation as an example)
#                         # min_class_size = y.value_counts().min()
#                         min_class_size = y_encode.min()
#                         cv_value = max(2, min(10, min_class_size - 1))
#                         # cv_value = max(2, min(10, int(len(X) / emotions_total) - 1))
                        
#                         scores = cross_val_score(classifier, X, y_encode, cv=cv_value)
#                         mean_accuracy = scores.mean()
                        
#                         # Store the result
#                         result = {
#                             'Counter': counter,
#                             'Features': ', '.join(feature_combination),
#                             'Vectors': ', '.join(vector_combination),
#                             'Classifier': classifier_name,
#                             'Label': label,
#                             'Transformed': 1,
#                             'Accuracy': mean_accuracy
#                         }
#                         results.append(result)
#                         classifier_results.append(result)
        
#                         # Track top models
#                         if len(top_models) < 5:
#                             top_models.append((classifier, mean_accuracy, result, counter))
#                         else:
#                             # Replace the lowest accuracy model if current model is better
#                             min_accuracy = min(top_models, key=lambda x: x[1])[1]
#                             if mean_accuracy > min_accuracy:
#                                 top_models.remove(min(top_models, key=lambda x: x[1]))
#                                 top_models.append((classifier, mean_accuracy, result, counter))
                                    
#                         # Create confusion matrix if mean accuracy is at least 80%
#                         if mean_accuracy >= 0.80:
#                             y_pred = cross_val_predict(classifier, X, y, cv=cv_value)
#                             cm = confusion_matrix(y, y_pred, labels=y.unique())
                            
#                             plt.close()
#                             plt.figure(figsize=(8, 6))
#                             sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=y.unique(), yticklabels=y.unique())
#                             plt.title(f'{counter} {classifier_name} Confusion Matrix Transformed ({label})')
#                             plt.xlabel('Predicted')
#                             plt.ylabel('Actual')
#                             confusion_matrix_path = os.path.join(confusion_matrix_folder, f'{counter}_{classifier_name}_{label}_Transformed_Confusion Matrix.png')
#                             plt.savefig(confusion_matrix_path)
#                             plt.close()
        
#                         del X
#                         del y
#                         gc.collect()
#                         counter += 1
                
#             # Save the results for this classifier to an Excel file
#             classifier_df = pd.DataFrame(classifier_results)
#             classifier_output_path = os.path.join(output_folder, f"{classifier_name.replace(' ', '_')}_Result_Temporary.xlsx")
#             classifier_df.to_excel(classifier_output_path, index=False)
                
#     # Save the results for this classifier to an Excel file
#     classifier_df = pd.DataFrame(classifier_results)
#     classifier_output_path = os.path.join(output_folder, f"{classifier_name.replace(' ', '_')}_Results.xlsx")
#     classifier_df.to_excel(classifier_output_path, index=False)

#     # Convert the results to a DataFrame and save it to an Excel file
#     results_df = pd.DataFrame(results)

#     # Sort top 5 models by accuracy from highest to lowest
#     top_models_sorted = sorted(top_models, key=lambda x: x[1], reverse=True)

#     # Save top 5 models for this classifier
#     for idx, (model, accuracy, result, counter) in enumerate(top_models_sorted):
#         model_filename = f"{classifier_name}_Top 5 Model_{idx + 1}_{counter}_{round(accuracy, 2)}.pkl"
#         model_filepath = os.path.join(confusion_matrix_folder, model_filename)
#         joblib.dump(model, model_filepath)

#     del classifier_df
#     del classifier_results
#     del top_models
#     del top_models_sorted
#     gc.collect()
    
#     # Save the results to the 'imotion_analysis' folder
#     output_path = os.path.join(output_folder, 'classification_results.xlsx')
#     results_df.to_excel(output_path, index=False)

#     print(f"{classifier_name} Done")
#     end_time = time.time()
#     total_time = int(end_time - start_time)
#     print("Total Time = (" + printTimer(total_time) + ")\n")

# # Convert the results to a DataFrame and save it to an Excel file
# results_df = pd.DataFrame(results)

# # Save the results to the 'imotion_analysis' folder
# output_path = os.path.join(output_folder, 'classification_results.xlsx')
# results_df.to_excel(output_path, index=False)

# print("Finished Classification")
# end_time = time.time()
# total_time = int(end_time - start_time_2)
# print("Total Time = (" + printTimer(total_time) + ")\n")

In [10]:
confusion_matrix_folder = 'imotion_analysis/classification'
os.makedirs(confusion_matrix_folder, exist_ok=True)

start_time_2 = time.time()

counter = 1

classification_df = pd.read_excel("imotion_analysis/classification_data.xlsx")

removes_data = {key: labels_data[key] for key in labels_data if key not in classifiers_data}

# Remove data to be removed (e.g. baseline)
for key in removes_data:
    classification_df = classification_df[classification_df['label'] != key]
    
# Remove corrupt data
for corrupt in corrupts_data:
    classification_df = classification_df[~((classification_df['label'] == corrupt['label']) & (classification_df['participant'] == corrupt['participant']))]

# Define label column
# label_column = ['label','emotion']
label_column = ['emotion']

emotions = set(label_info['emotion'] for label_info in classifiers_data.values())
emotions_total = len(emotions)

# Initialize a list to store the results
results = []

# Dynamically generate all feature columns based on channels, subbands, and features
# feature_columns = []
# feature_transformed_columns = []
# for feature in features_interest:
    # for subband in subbands_interest:
        # for i in range(0, len(channels_interest), 2):  # Process channels in pairs (F7-F8, T7-T8, etc.)
        #     ch1 = channels_interest[i]
        #     ch2 = channels_interest[i + 1]
        #     feature_columns.append(f'{ch1}_{subband}_{feature}')
        #     feature_columns.append(f'{ch2}_{subband}_{feature}')
        #     feature_transformed_columns.append(f'{ch1}_{subband}_{feature}_Transformed')
        #     feature_transformed_columns.append(f'{ch2}_{subband}_{feature}_Transformed')

label_encoder = LabelEncoder()

# # USE t-test! (Remove any data with p-value < 0.05)
# filtered_features = []

# error_log_path = "error_t-test.txt"

# with open(error_log_path, "w") as error_file:
#     error_file.write("Error Report\n")
#     error_file.write("="*40 + "\n")

# for column in classification_df.columns:
#     if column not in ['label', 'emotion', 'participant']:  # Skip non-data columns
#         # Calculate the t-test for each feature
#         try:
#             t_stat, p_value_ttest = ttest_ind(
#                 classification_df[classification_df['emotion'] == 'interested'][column],
#                 classification_df[classification_df['emotion'] == 'meh'][column],
#                 equal_var=False  # Welch's t-test
#             )

#             # Check if the feature is statistically significant
#             if p_value_ttest < 0.05:
#                 filtered_features.append(column)

#         except Exception as e:
#             with open(error_log_path, "a") as error_file:
#                 error_file.write(f"Error processing column {column}: {e}\n")

# classification_df = classification_df[['label', 'emotion', 'participant'] + filtered_features]

# Iterate through each classifier and calculate accuracy
for classifier_name, classifier in classifiers.items():
    start_time = time.time()
    classifier_results = []
    top_models = []  # To store the top 5 models for each classifier
    
    for classy in classifier_interest:
        feature_columns = []
        # feature_transformed_columns = []
        for feature in features_interest:
            temp = []
            # temp_transformed = []
            for c in classy:
                temp.append(f'{c}_{feature}')
                # temp_transformed.append(f'{c}_{feature}_Transformed')
            feature_columns.append(temp)
            # feature_transformed_columns.append(temp_transformed)
    
        all_combinations = []
        
        # Loop through different lengths of combinations
        for r in range(1, len(feature_columns) + 1):  # Generate combinations of size 1 to size of features
            for combination in itertools.combinations(feature_columns, r):
                # Flatten each combination of lists
                flattened_combination = [item for sublist in combination for item in sublist]
                all_combinations.append(flattened_combination)
                
        # all_combinations_transformed = []
        
        # # Loop through different lengths of combinations
        # for r in range(1, len(feature_transformed_columns) + 1):  # Generate combinations of size 1 to size of features
        #     for combination in itertools.combinations(feature_transformed_columns, r):
        #         # Flatten each combination of lists
        #         flattened_combination = [item for sublist in combination for item in sublist]
        #         all_combinations_transformed.append(flattened_combination)

        for r in range(1, len(vectors) + 1):
            for vector_combination in itertools.combinations(vectors, r):
                
                for label in label_column:
                    for combo in all_combinations:
                        classifier_combo = []
                        feature_combination = list(combo)

                        for f in feature_combination:
                            for v in vector_combination:
                                classifier_combo.append(f'{f}_{v}')
                                
                        missing_features = [feature for feature in classifier_combo if feature not in classification_df.columns]
                        if missing_features:
                            continue
                        
                        # Prepare data (handling missing columns)
                        X = classification_df[classifier_combo].dropna()  # Use only available columns, drop rows with missing data
                        y = classification_df.loc[X.index, label]  # Match the label data with selected rows
                        
                        # Train-test split
                        # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

                        y_encoded = label_encoder.fit_transform(y)
                
                        # Perform cross-validation (using 5-fold cross-validation as an example)# Check the minimum class size
                        # min_class_size = y.value_counts().min()
                        min_class_size = y_encoded.min()
                        cv_value = max(2, min(10, min_class_size - 1))
                        # cv_value = max(2, min(10, int(len(X) / emotions_total) - 1))
                
                        scores = cross_val_score(classifier, X, y_encoded, cv=cv_value)
                
                        mean_accuracy = scores.mean()

                        # Calculate mean of X for 'interested' and 'meh' for each feature
                        X_interested_means = X[y == 'memorable'].mean()
                        X_meh_means = X[y == 'ordinary'].mean()

                        # Compare means for each feature
                        comparison = X_interested_means > X_meh_means  # Boolean series

                        hypothesis = 1
                        # Determine hypothesis value
                        if comparison.all():
                            hypothesis = 2  # All features: 'interested' > 'meh'
                        elif not comparison.any():
                            hypothesis = 0  # All features: 'interested' <= 'meh'
                        
                        # Store the result
                        result = {
                            'Counter': counter,
                            'Features': ', '.join(feature_combination),
                            'Vectors': ', '.join(vector_combination),
                            'Classifier': classifier_name,
                            'Label': label,
                            'Hypothesis': hypothesis,
                            'Transformed': 0,
                            'Accuracy': mean_accuracy
                        }
                        results.append(result)
                        classifier_results.append(result)
        
                        # Track top models
                        if len(top_models) < 5:
                            top_models.append((classifier, mean_accuracy, result, counter))
                        else:
                            # Replace the lowest accuracy model if current model is better
                            min_accuracy = min(top_models, key=lambda x: x[1])[1]
                            if mean_accuracy > min_accuracy:
                                top_models.remove(min(top_models, key=lambda x: x[1]))
                                top_models.append((classifier, mean_accuracy, result, counter))
                                    
                        # Create confusion matrix if mean accuracy is at least 80%
                        if mean_accuracy >= 0.80:
                            y_pred = cross_val_predict(classifier, X, y, cv=cv_value)
                            cm = confusion_matrix(y, y_pred, labels=y.unique())
                            
                            plt.close()
                            plt.figure(figsize=(8, 6))
                            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=y.unique(), yticklabels=y.unique())
                            plt.title(f'{counter} {classifier_name} Confusion Matrix ({label})')
                            plt.xlabel('Predicted')
                            plt.ylabel('Actual')
                            confusion_matrix_path = os.path.join(confusion_matrix_folder, f'{counter}_{classifier_name}_{label}_Confusion Matrix.png')
                            plt.savefig(confusion_matrix_path)
                            plt.close()
        
                        del X
                        del y
                        gc.collect()
                        counter += 1
                    
                    # for combo in all_combinations_transformed:
                    #     classifier_combo = []
                    #     feature_combination = list(combo)

                    #     for f in feature_combination:
                    #         for v in vector_combination:
                    #             classifier_combo.append(f'{f}_{v}')
                                
                    #     missing_features = [feature for feature in classifier_combo if feature not in classification_df.columns]
                    #     if missing_features:
                    #         continue
                            
                    #     # Prepare data (handling missing columns)
                    #     X = classification_df[classifier_combo].dropna().loc[:, (classification_df[classifier_combo] != 0).any(axis=0)]  # Use only available columns, drop rows with missing data
                    #     y = classification_df.loc[X.index, label]  # Match the label data with selected rows
                        
                    #     # Train-test split
                    #     # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

                    #     y_encoded = label_encoder.fit_transform(y)
                        
                    #     # Perform cross-validation (using 5-fold cross-validation as an example)
                    #     # min_class_size = y.value_counts().min()
                    #     min_class_size = y_encode.min()
                    #     cv_value = max(2, min(10, min_class_size - 1))
                    #     # cv_value = max(2, min(10, int(len(X) / emotions_total) - 1))
                        
                    #     scores = cross_val_score(classifier, X, y_encode, cv=cv_value)
                    #     mean_accuracy = scores.mean()
                        
                    #     # Store the result
                    #     result = {
                    #         'Counter': counter,
                    #         'Features': ', '.join(feature_combination),
                    #         'Vectors': ', '.join(vector_combination),
                    #         'Classifier': classifier_name,
                    #         'Label': label,
                    #         'Transformed': 1,
                    #         'Accuracy': mean_accuracy
                    #     }
                    #     results.append(result)
                    #     classifier_results.append(result)
        
                    #     # Track top models
                    #     if len(top_models) < 5:
                    #         top_models.append((classifier, mean_accuracy, result, counter))
                    #     else:
                    #         # Replace the lowest accuracy model if current model is better
                    #         min_accuracy = min(top_models, key=lambda x: x[1])[1]
                    #         if mean_accuracy > min_accuracy:
                    #             top_models.remove(min(top_models, key=lambda x: x[1]))
                    #             top_models.append((classifier, mean_accuracy, result, counter))
                                    
                    #     # Create confusion matrix if mean accuracy is at least 80%
                    #     if mean_accuracy >= 0.80:
                    #         y_pred = cross_val_predict(classifier, X, y, cv=cv_value)
                    #         cm = confusion_matrix(y, y_pred, labels=y.unique())
                            
                    #         plt.close()
                    #         plt.figure(figsize=(8, 6))
                    #         sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=y.unique(), yticklabels=y.unique())
                    #         plt.title(f'{counter} {classifier_name} Confusion Matrix Transformed ({label})')
                    #         plt.xlabel('Predicted')
                    #         plt.ylabel('Actual')
                    #         confusion_matrix_path = os.path.join(confusion_matrix_folder, f'{counter}_{classifier_name}_{label}_Transformed_Confusion Matrix.png')
                    #         plt.savefig(confusion_matrix_path)
                    #         plt.close()
        
                    #     del X
                    #     del y
                    #     gc.collect()
                    #     counter += 1
                
            # Save the results for this classifier to an Excel file
            classifier_df = pd.DataFrame(classifier_results)
            classifier_output_path = os.path.join(output_folder, f"{classifier_name.replace(' ', '_')}_Result_Temporary_Newest.xlsx")
            classifier_df.to_excel(classifier_output_path, index=False)
                
    # Save the results for this classifier to an Excel file
    classifier_df = pd.DataFrame(classifier_results)
    classifier_output_path = os.path.join(output_folder, f"{classifier_name.replace(' ', '_')}_Results_Newest.xlsx")
    classifier_df.to_excel(classifier_output_path, index=False)

    # Convert the results to a DataFrame and save it to an Excel file
    results_df = pd.DataFrame(results)

    # Sort top 5 models by accuracy from highest to lowest
    top_models_sorted = sorted(top_models, key=lambda x: x[1], reverse=True)

    # Save top 5 models for this classifier
    for idx, (model, accuracy, result, counter) in enumerate(top_models_sorted):
        model_filename = f"{classifier_name}_Top 5 Model_{idx + 1}_{counter}_{round(accuracy, 2)}.pkl"
        model_filepath = os.path.join(confusion_matrix_folder, model_filename)
        joblib.dump(model, model_filepath)

    del classifier_df
    del classifier_results
    del top_models
    del top_models_sorted
    gc.collect()
    
    # Save the results to the 'imotion_analysis' folder
    output_path = os.path.join(output_folder, 'classification_results_newest.xlsx')
    results_df.to_excel(output_path, index=False)

    print(f"{classifier_name} Done")
    end_time = time.time()
    total_time = int(end_time - start_time)
    print("Total Time = (" + printTimer(total_time) + ")\n")

# Convert the results to a DataFrame and save it to an Excel file
results_df = pd.DataFrame(results)

# Save the results to the 'imotion_analysis' folder
output_path = os.path.join(output_folder, 'classification_results_newest.xlsx')
results_df.to_excel(output_path, index=False)

print("Finished Classification")
end_time = time.time()
total_time = int(end_time - start_time_2)
print("Total Time = (" + printTimer(total_time) + ")\n")

Random Forest Done
Total Time = (1 second)

SVC Done
Total Time = (1 second)

MLP Classifier Done
Total Time = (1 second)

Naive Bayes Done
Total Time = (1 second)

Finished Classification
Total Time = (6 seconds)



In [11]:
# start_time_2 = time.time()

# # Load the Excel file and specify the "Analysis" sheet
# file_path = 'imotion_analysis/classification_data_analysis.xlsx'  # Your actual file path
# df = pd.read_excel(file_path, sheet_name='Analysis')  # Load the "Analysis" sheet

# # Clean 'diff 3' and convert it to numeric after removing percentage signs
# df['diff 3'] = df['diff 3'] * 100

# # Filter rows where 'diff 3' is between 30% and 60% and add an extra filter condition
# # filtered_df = df[((df['diff 3'] >= 1) & (df['diff 3'] <= 95) & (df['up 3'] == 0)) |
# #                  ((df['diff 3'] >= 0) & (df['diff 3'] <= 95) & (df['up 3'] == 1))]
# # filtered_df = df[((df['diff 3'] > 0) & (df['diff 3'] <= 99))]
# filtered_df = df[df['feature'].str.strip().str.endswith('_mean') & ~df['feature'].str.strip().str.endswith('_Transformed')]

# # Ensure the folder for saving images exists
# save_folder = 'imotion_analysis/plots_analysis'
# os.makedirs(save_folder, exist_ok=True)

# # Custom colors for the bar chart
# colors_up = ['#FF6347', '#87CEEB', '#90EE90']  # Coral (meh), light blue (interesting), light green (baseline)
# colors_down = ['#FF6347', '#90EE90']  # Coral (meh), light green (baseline)

# # Iterate over each row in the filtered data to create a separate plot for each 'ad'
# for index, row in filtered_df.iterrows():
#     feature = row['feature']
#     up = row['up 3']
#     hypothesis = row['hypothesis']
#     ttest = row['ttest']
    
#     # Create a new figure for each feature
#     plt.figure(figsize=(6, 4))
    
#     # Bar chart for 'meh' and 'interesting' with absolute values
#     bars = plt.bar(['ordinary', 'interesting'], 
#                    [row['meh'], row['interested']], 
#                    color=colors_down)
    
#     # Legend labels, using scientific notation when necessary
#     meh_label = f'ordinary: {formatLongNumber(row["meh"])}'
#     interesting_label = f'interesting: {formatLongNumber(row["interested"])}'
#     plt.legend([bars[0], bars[1]], 
#                [meh_label, interesting_label], 
#                loc='center left', bbox_to_anchor=(1, 0.5), fontsize=11)
    
#     # Add labels and title with 'diff 3' formatted to 2 decimal places
#     plt.ylabel('Values', fontsize=12)
#     # plt.title(f'{feature}: interesting vs ordinary\n(Difference = {row["diff 3"]:.2f}% ({formatLongNumber(row["delta 3"])}) | Baseline = {up})', fontsize=14)
#     plt.title(f'{feature.replace('_mean', '').replace("_", " ")}: interesting vs ordinary', fontsize=14)
    
#     # Add gridlines for better readability
#     plt.grid(axis='y', linestyle='--', alpha=0.7)

#     # Adjust font sizes and layout for a cleaner look
#     plt.xticks(fontsize=11)
#     plt.yticks(fontsize=11)

#     # Adjust layout to fit the legend outside the plot
#     plt.tight_layout()

#     ttest_folder = save_folder
#     tt = 0
#     if ttest < 0.05:
#         ttest_folder = save_folder + '/significant'
#         tt = 1
#     else:
#         ttest_folder = save_folder + '/insignificant'
#     # Save the plot to a file (in the "imotion_analysis/plots" folder)
#     plot_filename = f'{hypothesis}_{up}_{tt}_{feature}_{row["diff 3"]:05.2f}_ordinary_vs_interesting.png'
#     plt.savefig(os.path.join(ttest_folder, plot_filename), bbox_inches='tight', dpi=300)  # Save with tight layout and high resolution
    
#     # Close the figure to prevent display in notebook and avoid memory issues
#     plt.close()

# # Iterate over each row in the filtered data to create a separate plot for each 'ad'
# for index, row in filtered_df.iterrows():
#     up = row['up 3']

#     # If up == 1, add the baseline bar; otherwise, show only ordinary and interesting
#     if up == 1:
#         feature = row['feature']
#         hypothesis = row['hypothesis']
#         ttest = row['ttest']
        
#         # Create a new figure for each feature
#         plt.figure(figsize=(6, 4))
        
#         # Bar chart for 'ordinary', 'baseline', and 'interesting' with absolute values
#         bars = plt.bar(['ordinary', 'baseline', 'interesting'], 
#                        [row['meh'], row['baseline'], row['interested']], 
#                        color=colors_up)
        
#         # Legend labels, using scientific notation when necessary
#         ordinary_label = f'ordinary: {formatLongNumber(row["meh"])}'
#         baseline_label = f'baseline: {formatLongNumber(row["baseline"])}'
#         interesting_label = f'interesting: {formatLongNumber(row["interested"])}'
#         plt.legend([bars[0], bars[1], bars[2]], 
#                    [ordinary_label, baseline_label, interesting_label], 
#                    loc='center left', bbox_to_anchor=(1, 0.5), fontsize=11)
    
#         # Add labels and title with 'diff 3' formatted to 2 decimal places
#         plt.ylabel('Values', fontsize=12)
#         # plt.title(f'{feature}: interesting vs ordinary\n(Difference = {row["diff 3"]:.2f}% ({formatLongNumber(row["delta 3"])}) | Baseline = {up})', fontsize=14)
#         plt.title(f'{feature.replace('_mean', '').replace("_", " ")}: interesting vs ordinary', fontsize=14)
        
#         # Add gridlines for better readability
#         plt.grid(axis='y', linestyle='--', alpha=0.7)
    
#         # Adjust font sizes and layout for a cleaner look
#         plt.xticks(fontsize=11)
#         plt.yticks(fontsize=11)
    
#         # Adjust layout to fit the legend outside the plot
#         plt.tight_layout()

#         ttest_folder = save_folder
#         tt = 0
#         if ttest < 0.05:
#             ttest_folder = save_folder + '/significant'
#             tt = 1
#         else:
#             ttest_folder = save_folder + '/insignificant'
#         # Save the plot to a file (in the "imotion_analysis/plots" folder)
#         plot_filename = f'{hypothesis}_2_{tt}_{feature}_{row["diff 3"]:05.2f}_ordinary_vs_interesting.png'
#         plt.savefig(os.path.join(ttest_folder, plot_filename), bbox_inches='tight', dpi=300)  # Save with tight layout and high resolution
        
#         # Close the figure to prevent display in notebook and avoid memory issues
#         plt.close()

# print("Finished bar char plots")
# end_time = time.time()
# total_time = int(end_time - start_time_2)
# print("Total Time = (" + printTimer(total_time) + ")\n")

In [12]:
# start_time_2 = time.time()

# # Load the Excel file and specify the "Analysis" sheet
# file_path = 'imotion_analysis/connectivity_results_analysis.xlsx'  # Your actual file path
# df = pd.read_excel(file_path, sheet_name='Analysis')  # Load the "Analysis" sheet

# # Clean 'diff 3' and convert it to numeric after removing percentage signs
# df['diff 3'] = df['diff 3'] * 100

# # Filter rows where 'diff 3' is between 30% and 60% and add an extra filter condition
# filtered_df = df[((df['diff 3'] >= 0) & (df['diff 3'] <= 99) & (df['up 3'] == 0)) |
#                  ((df['diff 3'] >= 0) & (df['diff 3'] <= 99) & (df['up 3'] == 1))]

# # Ensure the folder for saving images exists
# save_folder = 'imotion_analysis/plotsss'
# os.makedirs(save_folder, exist_ok=True)

# # Custom colors for the bar chart
# colors_up = ['#FF6347', '#87CEEB', '#90EE90']  # Coral (meh), light blue (interesting), light green (baseline)
# colors_down = ['#FF6347', '#90EE90']  # Coral (meh), light green (baseline)

# # Iterate over each row in the filtered data to create a separate plot for each 'ad'
# for index, row in filtered_df.iterrows():
#     feature = row['feature']
#     up = row['up 3']
    
#     # Create a new figure for each feature
#     plt.figure(figsize=(6, 4))
    
#     # Bar chart for 'meh' and 'interesting' with absolute values
#     bars = plt.bar(['ordinary', 'interesting'], 
#                    [row['meh'], row['interested']], 
#                    color=colors_down)
    
#     # Legend labels, using scientific notation when necessary
#     meh_label = f'ordinary: {formatLongNumber(row["meh"])}'
#     interesting_label = f'interesting: {formatLongNumber(row["interested"])}'
#     plt.legend([bars[0], bars[1]], 
#                [meh_label, interesting_label], 
#                loc='center left', bbox_to_anchor=(1, 0.5), fontsize=11)
    
#     # Add labels and title with 'diff 3' formatted to 2 decimal places
#     plt.ylabel('Values', fontsize=12)
#     plt.title(f'{feature}: interesting vs ordinary\n(Difference = {row["diff 3"]:.2f}% ({formatLongNumber(row["delta 3"])}) | Baseline = {up})', fontsize=14)
    
#     # Add gridlines for better readability
#     plt.grid(axis='y', linestyle='--', alpha=0.7)

#     # Adjust font sizes and layout for a cleaner look
#     plt.xticks(fontsize=11)
#     plt.yticks(fontsize=11)

#     # Adjust layout to fit the legend outside the plot
#     plt.tight_layout()
    
#     # Save the plot to a file (in the "imotion_analysis/plots" folder)
#     plot_filename = f'{up}_{row["diff 3"]:05.2f}_{feature}_ordinary_vs_interesting.png'
#     plt.savefig(os.path.join(save_folder, plot_filename), bbox_inches='tight', dpi=300)  # Save with tight layout and high resolution
    
#     # Close the figure to prevent display in notebook and avoid memory issues
#     plt.close()

# # Iterate over each row in the filtered data to create a separate plot for each 'ad'
# for index, row in filtered_df.iterrows():
#     up = row['up 3']

#     # If up == 1, add the baseline bar; otherwise, show only ordinary and interesting
#     if up == 1:
#         feature = row['feature']
        
#         # Create a new figure for each feature
#         plt.figure(figsize=(6, 4))
        
#         # Bar chart for 'ordinary', 'baseline', and 'interesting' with absolute values
#         bars = plt.bar(['ordinary', 'baseline', 'interesting'], 
#                        [row['meh'], row['baseline'], row['interested']], 
#                        color=colors_up)
        
#         # Legend labels, using scientific notation when necessary
#         ordinary_label = f'ordinary: {formatLongNumber(row["meh"])}'
#         baseline_label = f'baseline: {formatLongNumber(row["baseline"])}'
#         interesting_label = f'interesting: {formatLongNumber(row["interested"])}'
#         plt.legend([bars[0], bars[1], bars[2]], 
#                    [ordinary_label, baseline_label, interesting_label], 
#                    loc='center left', bbox_to_anchor=(1, 0.5), fontsize=11)
    
#         # Add labels and title with 'diff 3' formatted to 2 decimal places
#         plt.ylabel('Values', fontsize=12)
#         plt.title(f'{feature}: interesting vs ordinary\n(Difference = {row["diff 3"]:.2f}% ({formatLongNumber(row["delta 3"])}) | Baseline = {up})', fontsize=14)
        
#         # Add gridlines for better readability
#         plt.grid(axis='y', linestyle='--', alpha=0.7)
    
#         # Adjust font sizes and layout for a cleaner look
#         plt.xticks(fontsize=11)
#         plt.yticks(fontsize=11)
    
#         # Adjust layout to fit the legend outside the plot
#         plt.tight_layout()
        
#         # Save the plot to a file (in the "imotion_analysis/plots" folder)
#         plot_filename = f'2_{row["diff 3"]:05.2f}_{feature}_ordinary_vs_interesting.png'
#         plt.savefig(os.path.join(save_folder, plot_filename), bbox_inches='tight', dpi=300)  # Save with tight layout and high resolution
        
#         # Close the figure to prevent display in notebook and avoid memory issues
#         plt.close()

# print("Finished bar char plots")
# end_time = time.time()
# total_time = int(end_time - start_time_2)
# print("Total Time = (" + printTimer(total_time) + ")\n")